# Mashtots System 2, proof of concept: what Armenian costs a language model, and what an Armenian-aware model gets back  (build v1, `3297217c`)

**What this notebook does**, top to bottom, each step graded (**Measured / Established / Estimate / Conjecture**):
1. **The token bill.** How many more tokens the big tokenizers spend on the UDHR in Armenian than in English, and per word in Eastern, Western and Grabar.
2. **A script-aware tokenizer.** Byte-level BPE (what the big labs do) against a *Mashtots* tokenizer (the 38 letters are atoms, words are cut at their endings),
   trained on Eastern and Western text in equal parts.
3. **Small Armenian-only models** in plain PyTorch, every tokenizer at every size, bits per byte on held-out Eastern, Western and Grabar text,
   and the scaling curve against the deck's `bpb(N) = 4.9e5 N^-0.795 + 0.439`.
4. **The real model for the X3D demo**: letter-to-letter attention, letter embeddings and the landing's two texts, as one JSON.

The rules are **pre-registered** (`PREREG.md`, hashed in the Freeze cell before any model trains); my forecasts are in `PREDICTIONS.md`.

**How to run it.**
1. Runtime ▸ Change runtime type ▸ **G4 GPU**. Sign in as **hagopjay** (the account with the G4).
2. Shift+Enter on each cell, top to bottom, with `PROFILE = "smoke"` first (10 to 15 minutes, about 0.1 GB of downloads).
3. Cell 7 prints every file that will be downloaded, with its licence and size. **Nothing is downloaded until you tick `I_APPROVE_DOWNLOADS` in cell 2** and run cell 8.
4. Cell 12 measures the speed of this GPU and prints what the chosen profile will cost in hours (and in compute units if you fill in `UNITS_PER_HOUR`).
   Read it before cell 13 trains anything. `MAX_HOURS` stops training cleanly at a checkpoint; running cell 13 again continues.
5. If a cell prints ❌ or a traceback, stop and send Claude what it said.

**Where results go.** `My Drive/x3d.io/mashtots-poc-runs/<RUN_NAME>-<PROFILE>/` (small files, finished models, figures, the export JSON), inside the folder
already shared with hagsmac37. **When you are done: Runtime ▸ Disconnect and delete runtime** (an idle G4 still spends units).

In [ ]:
# 1. What machine did we get?
import shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else
      "No GPU found. In Colab: Runtime > Change runtime type > G4 GPU, then run this cell again.")

In [ ]:
#@title 2. Configuration { display-mode: "form" }
PROFILE = "smoke"  #@param ["smoke", "core", "full"]
RUN_NAME = "run1"  #@param {type:"string"}
I_APPROVE_DOWNLOADS = False  #@param {type:"boolean"}
UNITS_PER_HOUR = 0.0  #@param {type:"number"}
MAX_HOURS = 0.0  #@param {type:"number"}
ARMS_OVERRIDE = ""  #@param {type:"string"}
SIZES_OVERRIDE = ""  #@param {type:"string"}
EXTRA_SOURCES = ""  #@param {type:"string"}
ALLOW_MIRRORS = True  #@param {type:"boolean"}
COMPILE = True  #@param {type:"boolean"}
DRIVE_BASE = "x3d.io/mashtots-poc-runs"  #@param {type:"string"}
BUILD, BUILD_HASH = "v1", "3297217c"
import os, sys, json, time, pathlib
IN_COLAB = "google.colab" in sys.modules
LOCAL = os.environ.get("MP_LOCAL") == "1"
ROOT = pathlib.Path("/content/mp") if IN_COLAB else pathlib.Path(os.environ.get("MP_ROOT", "/tmp/mp_nb"))
SRC, DATA = ROOT / "src", ROOT / ("local" if LOCAL else PROFILE)
RUNS, RESULTS = DATA / "runs", DATA / "results"
for _d in (SRC, DATA, RUNS, RESULTS):
    _d.mkdir(parents=True, exist_ok=True)
EXTRA = tuple(x.strip() for x in EXTRA_SOURCES.split(",") if x.strip())     # optional sources: hplt, culturax, glotcc_hye
WORKERS = max(1, min(46, (os.cpu_count() or 2) - 2)) if not LOCAL else 2
if IN_COLAB:
    try:
        from google.colab import userdata                                         # a Hugging Face token in Colab's Secrets (key icon), if you added one
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
print(f"build {BUILD} {BUILD_HASH} | profile {PROFILE} | run {RUN_NAME} | downloads approved: {I_APPROVE_DOWNLOADS} | workers {WORKERS} | data {DATA}")

In [ ]:
# 3. Google Drive for results. A pop-up asks for access: choose the account that owns this runtime (hagopjay).
DRIVE_OUT = DRIVE_RUNS = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = f"/content/drive/MyDrive/{DRIVE_BASE}/{RUN_NAME}-{PROFILE}"
    DRIVE_RUNS = f"{DRIVE_OUT}/runs"
    os.makedirs(DRIVE_RUNS, exist_ok=True)
print("results folder:", DRIVE_OUT)

In [ ]:
#@title 4. Write the source files (code, PREREG.md and the UDHR texts are embedded in this notebook; hashes below) { display-mode: "form" }
import base64, gzip, hashlib
_FILES = {
    'mp_common.py': [
        'H4sIAAAAAAAC/61Z6XLbxh3/zqfYwuMJEIMQb1F01BlGpm1NdFWSc5SjYEBgScLCVQCUxHo8k8N2nLSuj7jtS+SL0zpp6qR5CSKv',
        'kVfo778AKFCyk87UnpG0u/jf964v/GZpEoVLA9tb4t4RC6bx2PfqJUmS9sZGyC1m+l4UG14csaEfsnjM2aYRjWMfBzv+WkecGE4w',
        'NgY8VsXO4ubhGxEbhYYF/KE9moQ8UpnrW9xhkf1H2gShP7QdWkWu4ThsfZuNuRPwMNJKpd4RD6fMB62QsCYOZ7Yb+CFYxmM7Yr7H',
        'LzM7ZmMjYp5PZ6FVDowwnoJ3wD2Le6bNQQlalIah7zJdH05iiKHrGSVmeJ4fG7EN7Uql7Az0xo49yLc3I9/L164Rj/O1H+WrAKcF',
        'hNh2ean0bm93b317i60yyc0sVQ58kx1VpdLG9lp3A1/8SIO17dD3tBGPZWlzRxefJIWtAg+QpQus/H/+W3AN6OV+e4PVW8zhcQxr',
        'M9ubgzA/tGBxeRLAD8w0Iq6ozBAgmrY018UxPMv2RkshP+KGo1tGbGjBFJCeJTjGx/6cumFRCICAiBvbsuDK7ohcs9ndu76/vb+n',
        'QxZoPHs2+3r2j9k/Z89n38y+nf1r9t3s37MXs+9nP8z+M/sx+Sj5OPkk+TS5k9xN7iWfJfeTz5Mvkj8lf04eJH9JHiaPksfJk+TL',
        '5ClM3N3v7eo1Ipr8NfkbDnr7OAGjNs6KbC+xDBZuea+3i6+3zA4zx6EMQ8imAoBGWxFhb5IOp4Rus/m/C+zGpUqzXtW0SrPZYuXf',
        'in1L7NutAnM9ZyJJ2k3f9mSx75sHL2OglHpr1/UP1t8TYfTTZxL7n/9dYD99Bmcwxx4ZFPJE2HSMKLJNA+kXcMeB9wSEcFLunZBD',
        'DhfeQnaVurubva317paeiQQpIsRpwZSX2DnVLrFcaAXRNg8B5OjEHWDZYbDKisqqFfyp0F+xSFfpsnI+HktbNzZ7uyJryD2yzS6y',
        'FfCqKuxNILI336SzpSW2krrKVlNjcnDloRFzueBz5TYl1j7UzaIYdYriNeYnKC78xDBjZypiHtuRMw3GzIhhPHYc2jGn8sPdYoKk',
        'Bc/0XdcgLNr4HigEE8+MJ6K8XEbtOOR6xg+Z0mGRQ9WT8mVnt/tBb1fRMqFEiYzgCiNNJcc/5mGZGJFqLju2YwgEzoEdw5lDO4zi',
        'zNCXEbhGCPnJo8wfCnxU2QECIFdlfp4Jw+QoMExOGW46E2QqBNnobl1Z37qm7/fe3ye33ypRUEmghKI8iPSqXpM6SK1PZ8+Se/j5',
        'HBn7PUPSvkjuYPsAyXg/eYL0fYGEvMdmz5Mn9PWH5BG+PlRZBgfUFO1r2iQPWPKRWBDZR6BwV1JTxg6CINKD0JjyEIxlSQDeTR4x',
        '0HmePFIZoLG5S5vZjyBP1LB5gC8PSBYcf00Hn+Db49kLJvjcF9IlT5P7EOk58ICAr98ywf+pYFHUI4fMqZBqPwLqTvI0+0ZiJF/M',
        'viVdzgnzZH6Gnxcqk0ovzV0pNdd3oPWJ0G/2DWwEqiTUQ5B7sKB1xi6lT+dPM1yhnhBNQGP7mM2pPBMr8t1jdY77gokv98DjEbHM',
        'eOP4S8Hp46I1VLLSHfC+p0qKWkJOlSw+TKNNp1RKM0TpCC0RxGernmh6JhK1UF1TFIERchQujxD7neqBJr7IVI7ppNo5YPZQUEWu',
        'iBwB/53d7bd7Il4PO0VBjlIOhyo7EhW2GN0aUtqN5LQmvJZmm00+85FH5OvZkSgqXemtvXOaWrF/yD19YDuOPrHGoT6e6pRrOvcQ',
        '77ckgUMpt8mNaEITmUzUFAlRdGQ4Ex4R2DyeJL9WqRxKVGubVYDE/PBQUKpqyw3sRxy1Ctua1lrG1uI8iDg/1I/q4rBNh3845l5N',
        'a+KgqTWIiOMYrkEAbW25jr3pVDMmFa3avH07y9Vj5KoejY0glekXRRewVNQiUlhI2K+urFRFN6hXqwdqZhkkP4whwGGm8VQAau2a',
        'Sio0DnLeNIagyB4vMu5hdB04djQm3sEkXw7tWLksanyAUhzjJEBsUk1Gvwwc1MSx76D3RCTphnxFIVpVrUV9pwGdl9mVD8sVrdaW',
        'cvYGWqdnG54+dHw/fLUQFBGIw+XKZrn6NubH0HA5tUmVAcrGlMnZgE99zyLOXWA3tBXexFo0RezBfIX2PbFu1FfU0isGASk0vBEJ',
        '0F+u8BbMylfm1jIdP0IDWJRzzfducpNmBkkEzklMpwaLDJN8h6YxmhgjtDvDg5BoepgumeW7BhRCyoGO61JXyQHJOHltIL+ntpE9',
        'XagdZeVhiEyghOifteFBsRYM+7DGATp+jk19v4zT1C4HcAw2PSCl/AQJXah2lqHjq2xsv5JrP7PbAvuiD6W0/LC3Vk+FwRokRT2S',
        '5o6U/Ukc2RbPvK1gqEehgVl56KF7Y+DiIW4qPA2+QnwqHba2vdMtd9dZN3Tf4wOYPLTYEsLzffuI1VqVFa1SrzeRK4jGzfI8jBhk',
        'TO80uGEZHnfKLidnBYMSEdR3e1d7u72ttR5VHymVnWKsKgJEApwu0HRC0yceKIkwa9Zr2ecBj2KdQsfh6ZcGBeOv5JvKLHsolI0Z',
        'R81KZVOk11R4qUrgVgntUU3EWFW4atIoS32hLw2mmOAkoYcI8PxCA0+/i7vX2wCqo3ZWIJOlCwoqQnkqknOMiSyi4d7zvTLHNGuJ',
        '+SnzvjGAp1m1hvDcwI/1YY3Je8f2tY0biArcgDyQbi/VmaWU9tZ/3yuMVa1NWM2yzVi2VmtN+GBjta2y66sNJUvUarMAUW835hCt',
        'HKJRKUA0qzWCoN/XV9s5SLtIZLnVPgWp1k4ZFem0V4Qs1ZYAmksT2970FKghCAk6ILN4B3F8um1YuMaHE0/MxCgFe73f4UJBV5pq',
        'pdYo7W+/09va03dwBdvp7nY3VSZO9LXuDkBq1Am0Jl8Rzf3q+kbRbpELp+eSCD+v9smY6BoUBKvkdJVl/YO6aYg8XK2JMEewDoxQ',
        'z7CAEfmTEFOwHts8XK3kupp++DIOauoTNTX8L/PbwlXqLL8z2Au8qznv4cRxfpW3mjpWzXz3ekWpiblOvEnomflh/QWBRDD8MtdW',
        'Ra/Q/e4M3xzzjOFLpWu73SvCzfJ8ZCABi8VXPS2wtC70LCUr/UhRfZ6iWYEWQmf1PwJ9kYZ9OkzLPEoUXTCjvmQJwfrSRtYA5gkc',
        'HdsjZ6Kne9lamFQp6+UGpb74ucTq2TrDvsSoNlg0wlqZlGfJdYr00A5kenHSTG47clsQWwLRJdZq0LUXvzMycYjuq6d2F0rOn9Z0',
        'czjKqNrD4mH64nTGVVIGekaIAlr/HMqBck5mDPhns5p69is9Ukh5JVcJBUO3LRlhpbJUozR8Vq8a6K+LhhpKtwB3u3yLAG9LsK8s',
        'la+JDp0iZT1ZBMdr6DXpY2X2Upn6cWygcOtkJZmeA1XmiWTL5BwjcrJnRS0FlVOriauBH8D3KZYUDiSF3hCG41NPiMvReOId0t0F',
        't5VQxiA+sIwOoDR6JZCr7K23UCphyIFU9KHgjZuTRQ8ggsSCs8bamJ9Y9gjdXFb6HU9cqbzUWAvf8vFt4gY6vYfK/uCmKt49M2a0',
        'hI7ZS6i2g79CI2X+VYPD0fk199CyQzndRKv74QSe5Sc25gn/UGxTlNgNMnoa2UiPJsOhfSJoaukaXpY0gElzBE08z6Q3PpJSI3mj',
        'VFaELD38GpFp22kIqbAmki5GvT2lEHIx9xdkz0xFB5kVHN+wUiukPsOZMXHior/jcHoukYREhBvJ580k3JhKnknDT0wexOwqImrL',
        'j6/6E8/qhaEfnqObsc8LSmy4gbyYHvQgrUVxOKSFLF38oHzRLV+02MXrnYubnYt77Mb+GqqoABu5Amiehnakz0djelpCFC0Sl2bP',
        'JBp8zTH9ln56KDGEq/TzV18Wj3/+6u9SRnFOLqL/WZBJ54xi/r8N9D6VvxvKhjdlkRnaQazgDm3E4nLWzWho9LQvpvkMHBOeefqY',
        'QLQppE3NjsQNQVYO8lLo+fmbWXTOpBWtUlQxmrjIsDnV+Zv5y6yjKCjPDvI5g1JK/wVkZjwQWxkAAA==',
    ],
    'mp_data.py': [
        'H4sIAAAAAAAC/8192XLjSJLgu74iGmVtCVSREC9RFLNQvUqlMiun8+o8KreXycWAQJBEkQRQAKijNJqHeRnbn1izfRgbW9vnWZsf',
        'qf6edfeIAAIgKCm7q7u3rDsFAoE4/HYPD8dXvzncZunhLIwOeXTBkut8GUf9A8Mwnnq5x+ZxyvIlZ6+8bJnHecbexmdjupPF29Tn',
        'LOWLMMvTa2auQ59HPm+xJI0veOTRdRb+zK0Wm/PcX4bRogXtvYAusI8LLw15fs38tZdl4TzkqbgfpzCLReoly2uW+XHKWwf+mnsR',
        'vRjwYJvAYF4exhH+9OMo9zZhRDeYFwXUx7mX5TyN2CH7xNXV89SbeSlL4nid2QcH28xb8PEBU6tm7Rdsk7gBLNxOrlmy9iLG2m1Y',
        'zzxcw4o38YqzSbvNr/LUY8tknbf87Trfpt5Va7GOc993l9e8Nb/s4d8pE/99RdMJ4stoHXsBWwO4WiyKc4SHAAwP9s6Bnu/Mod1O',
        '4zhnT1+8K6dj27YaseG/r4oJZMzE+cAE+CyOV8zLVhmh2UsQcd6azcM0y629U5ptw3Vwz5Qu43TF04y9vnNKHEfLeJ61BEoAmTCG',
        'm+Ventk/ZnF08DL2oUkA5JVuozF79dZ9+ebs9KXTJTTDz49PXRjS+XYerwOessswXxK450Cd7ONTlqecz7xolX33uIB5mBXA4IGN',
        'pH4wT+MNc935FpDJXZeFmwRoEAaBd4issoMDdS9dJF6acfXbj9dr7os28hbQwqy4/jlM1PUSmGgdFo9ohfJ64+XL4hpoKgTo+hyY',
        'AubrZQB+9TAuRkngFa23tJhRdl20ycNNcXsbhX4MKwYQF7fSNfRgp/ynLfCIuvsz9noAvdg4hB1GGU9zs9OCscWdIEwjb8NN9dub',
        'ZfjXBAACQbiuZVnFWhLXjzcb5MuMnSHSo/gnb8zOB53ewQFg78Ppc+YwI+3Z3ZEBN+DHDTYLgzEzU57EKE/4PLxqKWmBlMmYsQ1c',
        'L93kQWJAQwN6Ok03PAq9qA0XH56+NVrMWF6rNvDj3LBaxauzbJvvvPjk/ccP8jV6Xn9peX2p3pEypXnQS23UT3oHvgcUr7o4I6EH',
        '9F12cnb6g/cS37ry17Ix/HiOXdwevHoCsOny4cHBV+z8ggNPCBE8ZpdLL2ch/A84KAQZXUjiyyVPOT4BHHDgcqDyFlvGl2wWLqpS',
        'CV8tu2HbjAcoFFrEZ8Qdl8sQJJFi+hxkNYsjG+ZClx2YGwmCFuvCJcls1oOr+Xa9brE+XMUJMgmwsxlH62vBqlJ4WTZ7D4oiA+bi',
        'gn1hjOwRaY8MhmHfbxcLZIVnHqgckONh/v12hg96nd6w3e20eyP74P2bj+/Ozt/DUBOCeBD6uRkGjkYqIHiBcB0AP1NAZwJzzJQK',
        'w4JGqzCi1+BSwtIxzs7Ykz+235+ygd0xCmJ0gEJAqcVraAELCSOWgW7K2TdCuJk50IkFv0oNBz82cZos2QJEliFog/4DZnSMZZ4n',
        '2fjwcAHQ2c5swNvhxygEbGfe+ilPeBTAbEKeHTYSPMLL6cIE2dfs1ZMWocbpSALUwSHJexcYyAJ/M1D8OquXXEtrH9rd+5aOTKyt',
        'XBkHNXL4wpV/+rsRwX5BJIjhARBRgqaASSGZSqiQZGKmNKCex1nC19nd9PH6TAKKmVEctVEN8NQPvbWlQ+65vnoBQjmIDknzm+cg',
        'GzYMJYd1J0V9OQTvkMMChD376B4QLlMNfN+/a5VwQ/F5Hi3A6Fsyk29mPAANzGBdug1WwlG10KH58TXLQUqCEQ22DMtj0PQghYOt',
        'z2u8V4djDuI4AlG/XhPcYHxQK1scnwQ6wReF6uy6bkRbd8CRp6HfBhNlzdNDuXICU8fuDAZ3wgkQeMlnrk5sAKdPfAarWIeBMN8J',
        '3wU8lnNkLLACgKjevD1tn744FL3Q7QvHGHUGxycdf+gHx6P+Ub83HHRnx0PeO+6PuvOT3vDk6Khz5POBDh1SLs7ENNAUOizHPgS7',
        'Lm934D8wa1IwiXAe3cEQVEx3aE01lLx5etZ+cs26+8QfacLCQFaeSAFwROdepl8KVTcHTQfwPsQ5ool8uLN+AeSWULsuYtj5kG75',
        'Hqgjr9TBjvfcMAz+9kBXI+8B+XFvdDLofxHIhUSQoIZ18ksUT78OjOsgvQxXYUWP7CiRT9Ai4UHoMRMslH632+na8ILVBGPsbYNt',
        'Dy/VWwrQs87AHwVdn8968yM/GByNRqNhr9vpBJ1RwPu9fuD785nnNQJaH/iQaI8A3WnHc7roaiDvHQ8AZYNuBeSFsuuDDAej69nT',
        'lzWdV5c4Cg4VSn/Mer9l8Rzkdx6CF5+xJQf/Md7m5HVyIQ4ehJ5mSDXjSDjhl1W6fxZGHAi/3WP4CLAVEQ/sp39pd6LZ+ezT4Rxe',
        'v8TXFX68+Ynf7cN/fHYy6vYBpoOT4Mj3hh1vxE+C4dHJrHc8nJ3sZwQ1D+KIQ8CK26lxQ++o1zsZ7JM/oBnZmXCwzlLvcg3rSTeZ',
        '1WybCB6RSAJ7n8++nEX2Q6QZD+TMztbgDhZYeIW3nrwMX70FNHDQ3wAEZoLuCTcwv8QL06yRUX7cbvia54eVLgkPAT/p9fncGwW9',
        'wagfDGeAlOGA8+GID4/6HcDFScc7Oek24gHmQMuz8+yC4H0y6A2HVovho8vKo1736GTY2WWSHUP4mxrQhf5dpN4GHH20NcL8uljp',
        'F2KgEQx3adwdsU+GlSmFXTWCBsYBik7wq7rdNjpX1t9GLcylXsCpCaVwE447g+BW44TMIokRwhVaUDwCQKQwZfOgElea9AedE8Bh',
        'rwt+Z//oeNDpdoZwORgO+kcnnRO4PBr0usPj7hFedoYnx0AjeDk6Ojnp093jwRAk5wDvDjvHg8HweAToh9v97uhoOAUKKMdE0Lti',
        'HTfdMYPXe2N2cnu/7kLezcAxZ5+sglo2iJq/psHQ3aPMdCK5S4f9vVRYkwbra8Rxcgz46nZ6xLfN73Wb3uv2hqP+cNQ9sVoH9Qjl',
        'vo56TR2BzOjBHE5OvkSHlvbLr4PzOzTkDuKTwBVaT+D97RZkid8OYiLBggqgCTLZNuVgXZ2+PoMbs9RLrx+zTZitvRkHRyBgj3zl',
        'RD2icM2zsQwkdU/yJciRfMl8u6BrMIwzXhF8OxTle+AgpaBdvDTieX5Y9I+RHJqZmwSKxvr+qBeAeJkdg/DpD4fcPw76fHDsg69z',
        'dNKbAaYCvzPoN9JYY886XgfD/jEgtobWTgM7F10xdI/XiMV1uOLra+tuNOPeCUI5vf5ybfBQQO1nfhFF3CsC5D6PkAG9h8oA8ZZC',
        '0Lzf9Tr9/vxo3g+A949nfT4C/Xx0PDrudoJ5tzPwOp07hUBvrxDQzVgQzseD7qh3Hwti6OAqz9gmznJQzAmRPhOkX7GdNuEVDxo8',
        '6wr2UrBlgQfAjUanXgvy0EaLjGYEsQ/qKoJB55yLHSoR07D+POu3AHAzYqX5e6fli4v4Oxu9pO4brd7h4OgYDIiTv8DuZebzdZy/',
        'fPGUkZyydvFIoq7RacG1tgvxBqjF/aT9caY/01reQVuxi1hiDtdwdtb+oYuYw+heI878MGuvN9vDorXC1okXgB0zG8543/OPZ0d9',
        'MGCHXm82GHVOZoC4bq83AiXMG7F1AZA+VMMWF66OqEH/uIds1yQefx0k/TXx0wS2uxiK7/cn+f8n/iS/z5/sjQaj3knvz2esvUGX',
        'v4ZDuQ8Rd+Lg7yjZ+L2SDTyT3vHR8Ogvw0CdS0QSgDIvLim6d5UX+Rujo1cUi+4edV7hXnrGzJrbBzy0zR4z9BrYhnsZWHzQ/Pi3',
        'bAYTXG68dMXiC56uvQTuQLf8V0d1b6805I3SkP99pCFX0pDvSsMuaC2wbADjf6Y8LHmr2C6t4/TLTcR9Qq5fBzimtBSg/v7tyw+s',
        'B7Om1BsghoK/TMoQYSOM6g0GLFt6adAcssE+DvEf6MeV/RRBm26/N/COeqP+/HjY43zY83vekT/qBrwzmvtgAHS4dzToNIcLqpxG',
        '0YKj4FYahYOBznBdsR0qowcYOki9aMHN0T4c0coJN7jCLfkqdzLfXlyJKIvGTx37pIGjHmwD7oFnMzZVblKB0TNx478CJpm5QK4f',
        'M8/3eZJT2oBccW23nXD9mEVgtWbg2Lkf3vz+/HUjsrdxtE4O1SAVKCG+h97ouD+Y+cOTOZ/7s9nxcf9oPpif9I468wG4a35w7A0G',
        'PeR1LTYHfOZiPGhHjnZHx8cnw0G/tjWzORswUwhT3Bp88/7s9J1YWYvRkvV5fRnbPQxFdShI5MjR1RbN9OD9uzPMdckmRhgY0zET',
        'SVgU2ZKJDLcHr998cE9/OH3x8vTJy3PMa8DMmHUcr0R2RjUBAtHgxynu72Uxi+JZHFxT44x5CzR0MeMCw+4pUGMcHYgIx+sfXjx9',
        'ccpe802cp9CfWgcz1a02OFAXvcOLnt1tFe3a7599aFMgF93cLcDrogtNWuxtypWh1n6fcNzvDX/mQaEsjCguggsikRC6pwEx90Nu',
        'lbYBmXkKLplK5tvoI8HQ4g1MpWNHuCU6ZDG6wWBDYpMFz+xyRdAVpqJRCDZfgrO2WApyx+1RmBv4aQ57hUmM0PcHvoK7lq2ydszC',
        'Yy+c4teepJCzOE22mQiLtNg/d7ugWqlTkoVGBi65D0Ph8EglNAPMQmOz7XqlKNZmL2AuSSX4IkMsGD/BxYtAjcz20Wb2Cggq8gIP',
        'pBkw/SLMw0zkquHo4QbAQNydZQTGNFwsJZShfRhleZhv5Uoo/Qt3X82z5RYnfag3yNCg8lc6UKRT+yScrTmIs+tY4unjU5Ub0GIf',
        'Xnz4+J6t4jREZ8aQ683YNkLBlYo4EU2NWlJ+nkg82kaAbOapKS9Fbp+3SDkgtpzE09DLkhisHoxYJ16CWDZPf869RYudZjA//jMQ',
        'LLDJ9wCi3F/C3UWc2bZN6PHj5JoG4MFjAWbaoAoTsQ9NSxbJkYScbIndI9OB1ZQBTdFEpgcH5z+cvnTfn394rzLXUNaOmSlli8i3',
        'EsEwzJzMZfrauatvzGJKGMqg+s5sS4hfNgD9hbqoiCIUWWXnbpHdVOtKpPVgapPmk1Rewyyg3ZcoBajxFTS9q80rxrf2kj7vooNP',
        '1Q4+1Tu41DpQwTR05toAcehuA5ShfMECDpbWu0juqQ1QTfHZA41ProiC83UQb7WX1ZtFKLxV7mJeLuOMszCgrE4299brTOV19H5L',
        'NNvGvtQQz12VaVNO8LmcoEiu0RdfJNhQ5h+R2NnpW6AwkWFx8PbjKWZNft5yUIpGkWRMOUV8wxkSKEB2LfgJTPsNUEe8YqbHkjS8',
        'AMHRhifMB8vN82GJ1mPQChuwRmRuCKxG2C+YQHiAGYfOr/QfZVWTG0+ZhTIv/CDgc5g0JtLywJSJEy1GTo1jWtb4QO20wLLPbEoC',
        'dt++e/PsxcvziSEkY+bic2PKwnmRpgjjGWvMITYYgBN0jS1fej+RTab112kkVJOgbsWPul4eFwYEjJTZC56bRpmEYVhCa+Fcv2Od',
        'cWVXAT2uMNrySg8TQ078W0fmVaZMmgW0z4VAqHZD9hFmXIrB6adhVZpAx6IVTka2K3eqDGu8s9uhOqW/k7Gal3xhSl1svCszpzR9',
        '8MvZmkcmPbWsaaU7AJ/tJZhmZZI1mimTTrY+EEYhWEgRtgUCQ/wjZezFfRpfQi95nIO6Qsy0WKeKnD3Uoy0U05QQaNuNOZ90p/Tq',
        'nF6dSBBOLQ2lElCCcBSaoQtg2o5VQyDawoAtJLcihasKYTl4p7gplvKNQ08OSn/iMquADgx5SQvSiodf+NfQjF64JS/xriBnvAn2',
        'Kt4QoEdk1VYG6wc4ACI3MwfsIhiSZnlIOVzdhg2xqtsDI+BfHELYtrM4XqsxhK1tWVVka1hUSE/BQHK/APUNTcUQ1BN4hk+Lgw6I',
        'YCUJHt3Iq9tHmI8oTgKA6AbrhUfqGAa75rk1Nqr93TwKg0fjb7uD25tHr548Gn93csvYzSMJcXjSH94SauR7OGpKXibMuySCort0',
        'gh1ORY/wYzODH9+d2N059gs3VM/Tybh/NBX9430c49H01gC3xgSZP3l++uH86dRAEkwnEuBTQa6GYdUXQfCT6xCwRDyrgcG8YGZx',
        'v8tPxnYPbj9/Yhn3INBFdAOlYR5BRnFZsNyB3voSe3gX0SZOK9hv8aAA3hPdwtzxh82vwADPTCE76Q6eAjEt+OMSUVZEqZwMtivG',
        'QNcQzbLNKghTU/zIyN1CKoHe3XglvK8CS6syFiBnrUmLPL2ucnC+SWAdNBR6Um62nc/DK1NMlq4BMQZOI6/JYkp1r561sOFnDFxu',
        '1m6/E38FQJfcA8M3c26MjxmYQacLWBIYDsZGHsZqJ7Fv3FotOuYBctTp9joWHrQA+Uydw5TBwricGXR3vtwV+pdLZA4Ey7iR22ew',
        '5NTGQ1tml337LbibVmM7wCOa77PxXpkxg05WjU/nS/syDXNuzqw6wGFk4HefaxSzjwbwP35FwYxz+hOKQydwrzopxRLUBXhZNyvA',
        'W/d2zG4A6PgH3ritYRABbGdrzhOzDxaYSa8o0eaFwHHv0GnZ8PM0jVPo3I+364BAUhyxoO6hX8E2JHFKOYaHpxoEX7zDOnhPjXuJ',
        'yME2h8yAXwbdXseLBtPlfu2Y4xEOWif+Y5brD3AUGOuwMEru/w/PdpUnVHAC6HEDJMBRTWEicXr9xRp0j+2kvbitv1I9P+TCQj4+',
        'nchVVC0WAhTZ3pgaZagdBCPgmPhlkMneZDNFaDAZN2KI2/Y2aN9QN7c2THe93ho7r5C8nBdhJICsLbKrwSBPcY0oxPYnrN/gmm4P',
        'b8TRKbiYR7cwwwDQM49KpKEWqNmMaNdrQSsXhlTn3JZz/OUqUt2BzDwC6JEYrlhLu/CodWSiiQDzJSskEnFCN79OuGPIOJMIBocZ',
        'MCvZEwBuNGzQZHeBVJwsT83gLkuksKZWPHLwRBqPLsI0joQVouKVwth5HUdct/CkjSJUJj0sxlkQ56GpmNQ1EVkVCIjATvGkn2l8',
        'bVCviR1mZC1LNWbYvgdWhUFiANonpBsy62BXEt1kmlHAbhbE06CdR4VyvtFYk7WBW8dHdgee6XY/sP5e23F2DTTsLFDKZLjxE2TS',
        '5Kv3K0z7EikCVCnxgTyv18R4pd0h5nNmB1twOvGMownzahWCigSfi1PFZ1XrAu7+us5mKjQoSVzwv9EyEJI2CwPJzrAY6VDq9kUS',
        'S2ERBtNJZ1q3PCoiuU50xbFUlBwAEFg1dniw730S6UqGo5gNA6kkcPquECQmviKnbBjGH0O+BgsX/XyKqo1V2L8lt8eD2G/JaCTu',
        'l8TpBv21zcZrsW0SZ9bUsvHgK7km0Am9AEtGLsBZHxR2CxkSODiojMiP8fC2Y2zzeXu0a1Uga6zDiMRE3dqg+w79sVNg6zAxjc/R',
        'rtdKDYDlgFNwfNP4CuN7MLsm+SsmLV5BsWsaDm4NWJN2d2qLQYg14TajwCrMgQgVryq98XXz2AgcF1ioYXB8hBahZLWGWZSTEGhx',
        'DIMA7VSwM7UeNBOKxzgyvEBjhxmJFsRZ8+zAg8edi2kzjMrZNY9PA+EFMACINgoxmw8dvo6Xz3WjWKLbaBNmfBgDRbRhFz+bTckd',
        'K6C2XoKpMVVy0PRhmS3or0f/9qdWw2JxFcWC71/aNfIetdtHEvieEi77+9P6OVASqmDoUk61hGEiCVBYGppw6grhhL7JjoRDo+Be',
        '40TJXnCRzIrAsaxyXkXA1cQZ4oF9nuJNp3dUCqXnoFESTSjh3kdcxmrHGIgUvCxPJsPMhPCBB3gQzvaXXpLzlASJiIqGWLDg1emH',
        'D+7QRsVE3CuMy3+Uc/jHcsRCpi3EVBz9eL79JgVlwIOnyLClF7ibT05L1Bh+gxYwtze4iWCmhjk5bf+36Tfu5+Ab67ONielCyZaU',
        'teLXaD1MDBRbRNfmxqYpmV2SRxuxDvATlttodROyw0MFUd31EKuwwU4CNHjbdW5Czy0w7i1F35li8YoenRjASD/GYWReiF3vC1yg',
        '7O3CW4OLaVpTFYERW6oupURKSb/xrlxhMQiFMEdSpesdFWTCnF2ldywZPladMuFrzBByhGS8eMyyPAbkeHNEdTEU7m9gHwUOVZ2D',
        'aw+cqku19Ys6J/lJ0PwcvaOfQIvSk2doepGapKcYLkEKSOZ2BmbYxnNFP3T/oIiRIg8Ce1ZZU9x2gFqvctOcl6FC05B7wYY01fES',
        '9Lar/VQEL6xCeo3GtFqakXnX2KmImMzJ73Zhzq6gnQ4YoRgcMslIs/PYTa6JbS3dRFFzx1gtOKoU6GgJCqCubXC1N5nQi2DjRaBl',
        'gHXMC6DjPLVgkkBjztrbzAKPrcYyyAvvTVZKeIbBvBk2YSDX74orDEDeBQRgTsTQhKY8xbjWBPqmCD6OgeIY//7GkYsiplH6MuMc',
        'D/phUFUPBgtaQ3N7jitNXboBBE9/yYR3ep3BqIWjbzeggHEWuiNMqcGO6MgWjUwavwLy0g0NdlrDpCtt71kQAmTKviZQ0+hWxaIC',
        '+UhZKz+DqqbHLRyzZpBEGEruHjTEZPIG5dykRIU6MtHjCsWMdbWlBNZNxXQlbgMrgm9uxzfgiKKdUw17Y3QVZkZLs8mA5KYyH3ds',
        'v1IWCE0M737nlHd3FyLknRRkMpGdnNMdM19KxgwkCg9okYlVunLmfQa5Vfh6RQqKMCz3+XiVKQFHVBR6VbzKGc62AbgPTFtvQdNi',
        'l+Ku5amWujCm+VSE+1wfWoxXIyNBBHovdRzJad5pIck2bacg6btRX+36W6e+bdaIbhKO5SG4moP0Lr4kjWIu4jhADQT/JEsexSCZ',
        '46jQU9pZxDy7EDVQ0Lqg7VR+5fm5FBNSnXhgB9T0k59dFNsUZeTtPt+Jjiij0Qk+wY4flaL8hn5tNFXekQ9rzpcYYYeVolBzqva0',
        'lKNpYJNYEVPF3Z2pFtCIS4Xmi0QGsrTxVYC9b6/jSxjGEpIcTCCRwyIMK7zGHuhveYpRNnHpkVUR7UQG3pcMeQkO9MKVA8OrpECi',
        'htHks6K9mOHu8IjuLxi/pA7sfc7p0BMF/64TXlu4qx7vjiqlLsEbRqSouBfU4pIUMv4BDTEVMNYIUeolehVTuhYRtA7G7AYnrpuH',
        '5U5TrX/aEcIZTEXqzwRm0OBP6ZuNpnyhJVvDX4pmIFQojEERLoRodYNJ2xf6dUM3u7XhaC07peHkJhTgJjNF6CMCGecMcHsUeMXp',
        'HnU6Kn8AcyYY7p8BF2fbGdjyn7NvELXMEMJOUcRkTC9PMYUQHhaCOCp3jLpiINoIqMpgLcWUxB/G1aJaO2JJDLhPwnEIz6JpHYGL',
        'In6By/aia/MM44vFqS7M3TD9pVBiwuJZWPucVQyrESCB6Aisr58UkvJMJYGwbnvQRkCKdDtghtBbg9wLLzh74l2jSEHJeF7WB2qx',
        'T8V5cvjxXBWIQTMAI+g8wCwvmRim1XPB4nsUJPQugI/KvEV4/vbd+bvz5/YmGKv3yvMfopQbqh7k4t3DdvXSf8CX1DTFfHVMH1It',
        'cZO8dliL5rPxgKh4gulivOw2pSN20J7y7Dx0J232oalMIZvxNWZ7ZRTwSymtnoOSphpcVO0ANQrlIXoqD9NWeBAWMU6I3BaZYfVJ',
        'VgA7kBudQOtuGIW564Kbup7r6Q3wE+h3kbTEpXQZ5a8kDWNMnrm5bcn/lz3Ow5w6o9hUBuaBS9NoMW+dLD2nYx+1KFUoc9FHpWfO',
        'wJWHG7QZ+PE2ItP5xh9XHO8zfECSthDCNCm52tsK+/hiGhRh16ej/JYqjVOCk55XUVhCqoMm8xfnOfGn9jYJ0OGX4iMoZEaHff01',
        'O2mwUmg4ac8GjUYMtQCrtQawPdurlU3Ri9j3cMMV/H3TsrcRwM78eoK/dDdezL5048tZ/MDExKgfq5ppotCCWxrl+lUnD0NM/XEt',
        'w0UjOuibDNl8iRRpEh2BIW2KqeDTbwRxgdfzg1XVaAschpbQ0D/S92SBnefbBNz9YohyUfhY9m7dMWbzksqpRHL/hwCqgb9KkwUA',
        'q2xYcBtAvJgi0Yz+NswJZxjdC33lvMDDkmtJ4mSScSn4oskCNEQ7NjrG8p9Sv6wq7BLM13kBzToK7wFRgS1N/dbYc50QPUvMkUWB',
        'JVzn69pufobxX+SrRA9liCdd9aRbf9JTT3rVJ6uqM6zA12KrEnxJykVuWiP8WgQmmriEM7XQrbwVbn11xk1JCcbvDAJ6JYKI8Rh9',
        '8w9RRLTBrxLzCmyEjaCDKwJ4uaCf1UakNrrsThgZ/WrM5kdM3ftREwsa9SgcTkIw8BL4F8jv5xImuGUnAaJ5U4U7g7VMbc2nMS4x',
        '3nXvrhD+h3t9tCModiaRGBxNYUmac6qKi7jIKRkKCwmTK4WHaVwv88PQeQa8zaV2/C+0ug0HrRwUa6J9aB8T0h+0pPTBS0K1Q8ui',
        'EeZLq4JtGFGLEW1onSgQVmPJa1KmqNBcMDGwiTFVau5We7mAyKaQLNBe3sXdXfhFD4ydncoNWH7u2cvT9+9fnJ2+RJpLjV/+80//',
        '8sv//uX/wL//FzfgDfN33/5m8vlyav3yb3/6F/N34pqeYCNx8W+//C/V1vl8af3y78Wb2J31y3/+8h8wuvvu/Nmbd6/On1ZG+g/6',
        'K8b607+K/uieuMSH9PKZmCAeqNkkFM3VYjTlKnCYZ3e1VJNQIW7NUNOFFBq/hRV5kZV2otk96fUOuyeDjqXZjKpqXpSJ7DKGR7h4',
        'mtnsHcE6YyadSG5JYxBm0u7adteS1m5pscoXx6yto0LUvr4EU5ghHloMb8O/AHn499/xSAaakh41gTsAcUplZAKAlMMtF7AzgMRA',
        'i/3pX+FN+tWS7wkD9JQt+TYNM3BvqTo1DIO1KcWpbGGLl9WQWe5lK2XhiUMqy+0Gq4Vg66pFS9ql4lrh8UhBp76mYxN7HkaBt16D',
        'w1TFunQ0H9D23TMZGib4gw3tg1hNyQrw0esjJ0x3W7XyH5RHKV4E661j96Wjq2CqPf/WYe2ygTzPgrFPmUQrqt/2rV/XIa5VQN+p',
        'fH7gfnovNqoUTwj31oIHb949rT+a/PfPnz4H7vSbIiWNBtD5Q+w464Wj7QhgQSe6TOP1szPpNVtFmt7bj6fkTZd3jM9XnY5wsS29',
        '08IDn7DP+bTihDe3+xzd9FuY6gRXn6NKU4lMojK1oy2WhNN18YCGvizZXFbjBlfzyATQ0TA7gQBLdbgTwLSDcMFp72Xc7SlBg5zp',
        'RpVghBwUc/UIDyXp6sMc7AkeXDYFD4RPj2W6bT/1+z2z2O67LMIJuxNWWCaaccFgyORmQyEMu33h/uMzjJ2uY/B+6Yy/2BNB/3pU',
        'bZItqSg5Otom+OrFRiyoKr2eG30UAFzhMulk1e2D1TJS3k5L/CnAQMFrGrVcdnQp/ZsaHK1q8O0S+bfbryprGE45exUctaBl9f0R',
        'sjd08i0b1Iy71WhPF6NqLEwuDRXvc/K4VaY0+e2iNL65AdysXWF0yBcIZhTXEeBz8WC12h1wn6vBo5lTBHGE7VH2ZVFnjuzQqfTp',
        'NPXulOOoWcrK82K/TI6+ex4mFW56ZbOB3tA20pijyxQdyDIu9i0sa2IU86pnFGrHSMzKgDJLiYivvpfQlCZ6ZhdhM1prdWy69aVj',
        'gyosYnEPmQIG8ChIh6MCfgzhXdGtBoqknAFqOjKmRQSw7GB0x/tAjtaXLUYvBHHvYkDHt8h7wYlEM7BYlSul4ZgsLvxiAKr4ivml',
        'tdk3K9GZA0u5cMRoMlcxQa0KvTpF9w6M0GJLRxP0lr3kVyYG5o14ZTQdNvoVlTKaPvRBDOIdeYCwSIlI/NzpVbWOJrUpt5VaFsIa',
        'EP9b1u10gDrhXaVUMDmeMinX0kvBaLvKB4Ib9RzxMgeCPs/wZQcj6htXxqXxwMy/hkMvlfT+wgfMzLTRi8NgO6UEaht85bKVIv01',
        'kxLLTDtS8vvC54Wbl5nYXE2QEF/P4ZefPpGJMyqjH65EUNr1vcRdzPSN33vS/OHqobhrzOH/252aXHoXXKtpcP/pA+Hj0vdeVCKl',
        'bOEULenQtHOGyZAbSqTEpHjMTD6zfzh/9/7Fm9fqyFvmYGAb7RW6oE/K0FUZ5Rc/0ziD3mbwQwYOvmJt+I91bX23Sbh7u3sX0r2r',
        'bG0UR3fFnj01p11/BIdxbozpxKLxSV08p4vbUp2itDDdFnOp0AjtSH58uhvxLjueQDOKflWS8/ZlEMoDDVaZt3ckk6Qw3lyYE2Dm',
        '4l5AOYpqJEI3KoVbbs25cJ84Y/GzofJ5onkt564I/X/xWqltRIbF/mU1nMuQ5kVWpr9aaBH2Oo15MPMJntRv4RqLiKD25qQztaZl',
        'NNHzUZ/dXIyVqwcOaUQzNT0gVasldgixXz3NykOWuiA1PqNLdEkxgNdtsWoX7t1doFYbaMFo4/zTc0Pjoomh1UCbKp4CBy53DPyW',
        '0Jpr2Zlmr/NNeQQ8sygBb4egQfrDqrfQ6NqBi3uPYuDEt8SgN3Pjxrttf3czuwVybwCTTMbRl2oBf4KlEGCZaFk0p3Ic02jYFsav',
        'Boi1FTMv0+LV3MUyqvzes0labOWXB6jqCaqqnEeq7A1lmorDomsP90jx41HVwllUXAUGBdBtxFT5hTD7xfVkt0bDlN3Pt5Wv1khC',
        't6pdlvUbHtqh+qxRvbtq8YSH9ibKFO50Vi918NDuig8eqQ6LGK6PZmJgYl0JuHRUPQSN8cHqErmA5clwJUOEiyLOoNVOdJYOitUk',
        'QshL6IE1htnIjS5Exx4+MLtOM3Zrg+3mx+0mxlEeHCy9IdWqsoOom7kqwV6vM4JQQC1d9iOISW+D6JIA301fq/RmFams1fKI+0Yp',
        'yn/cPUSlr50xLveNIdXtff3K9y1BSmKnt9uzdPcm26PDNB2Ro2YoIxWlrtREMX3aACwlfVyXIqRZKZtxOKsCp09VOE3y6jBEDvWJ',
        'oMX2CcYa94rYKkKs+FBFM0ZqZU9oNMUPKjU+11IPdWAWXZN2Uo5PKLJVJPOQ1u1UZiVE42oM6kvf/eAXuq67KA2jVZkMULbR9nA0',
        '10hZJwR2cfJhdUsGCgqVyY2BZ6fz2yo0p/uwtiowpIqErQDQYjaO2sm11Lk6FZjWWLhGHFQHyCmKBdFO6AprlctDYliIQeSgzFyZ',
        'SSg1CCJSq/WvUCmEUu3gGvk6XrQQGW2icKJBWKqdga6dHMa/+hiHohe4W3z+oC6SkuLM/q5QkmuYYCdT5uymau5mTtyNyfJd9wY7',
        '1fBKOMLUNWdBaZ7OTCR6OonMjcLbeAthUpnYXtzroJhKetXSABTRys6qG3bSTCk+bTmGORK902GQTHpFqovKuPsT83GN8kwpdUIZ',
        'EjXnpW/vfAoVg7RjeVimCOOKXaZadJbqX1GEtiVNwO/fKYseo8UqJ14KI40lVQJEja12XwYlNxElOYi8NfhVuhDYclUoofQhMW9g',
        'ZjV3TFn25KU+OX9fy8motwVDgwRx1QIRbfF7VnVPvPF05/fv5PlORar0JSw66kmXqL/sq83aUByMNxsYZndy8C7mn1G0DC2lZWrV',
        '4uV6AF9/v1rwo5kWbpCMoStr3LplMtwPKBe3R3RXBPgzo0pdA7usv6yVMkcXWT/p31LFQqnqTI6nrTT3O9A/3svFFj2GQ2irrhof',
        'Qel20xljEsVRi3XHWLydPqzR7dmdW8KErEMEvwHuX2PNEr3+QIKZijIPUMxNS3OIfTfLQShtdvzNB/qm2tmxoHp27IvccWtvxIma',
        'gQaDv7fjm5CO+NdnigNPql9VK7/4on3Xo/zSw069f/1TVlrN+Godcq3kcrUkr6oXW1Yane644jhPeTZir9a603AGN3kmzDqgE+jw',
        '/bszcfBRuYLGtHJQ9lw/0N80G1KND1+SNS6JCmuBUuQKWEl8xBTzOFV95UOtujKWS1WFEUVNv10lKRYGHGrSJUCmz/HrNPzE2s1m',
        'rG213HuwBLtsoC4JA7QXS+MQFUs1dG194eFbjWgrZ0eESIrF8QQMm2Dg1sUbctYXe4IZnFL3yz3BLPVdFSfUTXOZAyNNacx/2pN5',
        'KpUBhv7ujFCpL1c7xVWcqlANomowBAEBOsFPti6lHJq0V9PDfdqe7rOqY2eR0zsaavCclRtphRQB8SI1Z4NcopeU74hNG33VGZnb',
        '0T55skt+tXng8Zumo0AzsSIvXVBSMu5Z3BUG1DYzwejHJLhh4cApgH4H8rzUgDkeZt5QaqA4RHmFFcqgqfQsvMzLYdA4Q0EkbhN7',
        '117REuuAuhDF+ZX9FglNDttiuP0qytGmjr4XK57gCh38R0+cy4C26AwpdGSDwZSYanu0VSAYPUk8xStOFsowSKX8SmXj9+vqGNhz',
        'SxtJbBvsGUmGY4Hto7KYHJlUKtGdumnQacXWGh5xupZta/m1VPFE8RnJ2NpzYcLCaoxp42nDhsOs2sswbsNre8UK9jcxlqIGIsqD',
        'xk6NwpwwvqRz7M/2gsAUQ1Q5aibTDOAZOMV3n1zDKax4kjeNjs9IGLtli1m9xdygiIB7E0weXWB1t+aO5gZteVKzeE8zEm4TnPaF',
        'zAmMDRm9rg5LAnki2kk4Xwg4C1Et8prOYe07G3dya1aABpSOQ9QVO3Kwu7f1KlEvQcSVRWBkQ9xGjSQvf4uCpENboiBI95XyuhGt',
        '0YAtzVMEe6uhck6lcA66srDqcYUjbX8dZ7wmVESKjVU9lUkQK1wOjfMqXVR13PigyQ+nnmTg4kI5uHUPVTSaTi6K8IRQ9x0Zmehg',
        'RMKVJSDvUITVPVtto9WqC4XasJKkBf0hl6QP4pKdXqQz3EDrDQPKBcFvGA1uP3jwu3tTILy7laVvr8idxiIsICJE1bhAIUOrkQG1',
        'QSN3H10FR+iINkr+6d59Enyz3CipTEuUdKLe9pd10t+gTRZqX2y04Peq8OjSNnqUUR14y9gp5URvl24vtpJr1ao5VUonllmjbuZt',
        'MPO59Lf31et7SIE5yrwQZ/0z5gHVt8VXsXMAIZ3gXeMp5ZxqTVBjnKvM1ajKD0NLbBU9ZqsQg8gYu4HpVPafhIQ0Xz2xKLJT7gSW',
        'jKHqpwqul9VFqjRWUAY6ef/EbsTTRxKNIN8L8aSS97EBZpS9P3/9wX3/9uWLD/WETkzMnvzpf4zt3/zul/85tUTqp0hUaMYBUBe5',
        'p04PP3QZoYESZFhDEpPxeOAMOlptl1NZl2IHVmalbvrhh9P3v39vb8CD0HYasbZ+ThEAcYRQ27xT+3lX5Xk79c0aedYO1EfxCPOO',
        'W0VYB3OwadLFK0UBbGrIkvU2Y384IwNbVez2ouwSrdBFjB8O8Mrzfli0hdIf6HAenm8nMlqEF+CLYFOx8jjN6ifAU1BX8eYhuRzR',
        'goopYnP7Hf0xEdTSOeHr9Z3+TahqYP7l6kSeUt4XxN0xmqR8S6vBjori2N2GX4GpT8r7vm2r2na/RuSyfJSQ89ZkPGw4xlzkhcLr',
        'sn1zfYEitdNhgw7ZGDu7etCFhbVm7ZM9HjDiaIIxWXE8GqyeFh73sayy5BRNQ5X9EgpL7WIhHbZkeSDcqZQh/oQydKSXKZgSKwLJ',
        'G7g0GtfS8oRxCrHVKoKpSkdgu0JH7CRTtFA5kcqPFnYhCi5FRqiZoC9EahXzy3YQip1W6gsikzmGZFNXManMoHbEcLIYderLueOh',
        'AiFpagmlD86Aud4Nw2XuA7JE6OBBA/mQKVevj0cgdct6Ys3UMId38LAPvq5SuWU36mfjeytJsHFmU4aeTx+FEnW6zAn1OrX2FtpV',
        '8zON128+vkb6++H83RP6esTTfzDKHbg5hQSG9HtFm3GVR3BP/Vxv0EhYIWP0RbLpet143Fw7qTi39tf6LZFcMIWAMIgQgiol1whE',
        'W/XqvVjG6XWcP0O1StURajY/qA4lTO1suZ3PgYbLATXzXI3o51c0FB2RLxpOxlLjabjdR+KkVgzK7Qsc0a+MOzjUuaTyOX7M4QZG',
        'umX63vVP/heR+j6DR6nDP5yNy7PwpY6NQd5GWGFXaVSz0Joob2S4EquubSNvMwsX23irxSF2+IqKoOHEr4oqgsXpxHs4rZC1SFvF',
        '64VoJjLTTnk+DO+NKkkqDE2ayfJ0KM6GQpiJam51afaTX5DmF8oy4w9nGOeit2R7h1LI1Esit9OydqiUMGw1J9SHLSaKD5faVTQf',
        'H+z7qgKmw4RAj8YMv20tv5duaDriRyAzbAQmj1Xt+idf6zfEgDCT3fx08+O404d+NNeXasvAOtBwEUdTOgK0MCOrPFOl5hhGGU9z',
        'M8GQnT5RGMeq1MibwB30QGB+JRwICNDNA5wAnTuIRQUDF5UjSBCrss7imBqdadMOs4E1R59EkimdMpdLN5NCTAYE4pClboUoqB0M',
        'UVVmVitd8uwVPVQRDdpgh9g93qwHoQg8uSxVXTqqfOPOkEG89NpZ0UE9IMjqUh+jUZuAeUXF9a7FV5SA+X0guQ0eX8MPL+J3SnRU',
        '6DUG1bBonljK5SVXMcztJE5MdcvSXDFZ6fuS6p6jzVwx4b1L75o+Cb6GJ8V3cpRcEuWvyJtQQQ384RBFGpiwy+n7EloScA7WrWNU',
        'XQ+JUm4v8w1uhhExOJgZtquokKtd9UkqgqASqRkrPhg9wM8BijKU4t7rM3nbelx+E6ho/7j87J/4AN1jPB51KRx6zW+ir8Q0zEkg',
        'k7ZtQyr/g3ZhRh/oIt/Vm6HUKAlXfLaKCMkLAnSChWdRNmEYFiEqoQxGjOtJtAeap1BOTTB9INPNGxLM9VDATHxTCyuPG6K2HqJL',
        'jwPorRvQimfWqZ4lOVrCAHfp0wrlGuX38nZG+f35H/WRVCBwxzvFxmPwr7cbs1vqL4Qj8ODV5BFy9iPi7Ed12f/IutXPsz2oD5IO',
        '+CLxeYvpWTPgh6HMBNF7iz5p0fVj4gOMVTJP+3iEdPxpGeoY887m2bio/dpcwG6/E5h84bEQlVlIBy/2nQop68U2pVjsO0om9mQ/',
        'Pr3DyMeXKIGkYotkD7BFpn+2eVn/PFE9N0M7kyLjAFebtc1RjtjnII9hUh/gGmNa5x+KI8znHxDwmSxUij4kzln6CRFqWuMGvw0w',
        'Pjzkc77mdsTzQxz61miu8GrIvwlVuxR7YVZpa1FkxJMJYQnuLc7WXGQK0IfMDNFkRmYUdWHCLL7Bd7ST1TPtCUpdzyjKx+IXC5WD',
        '5iEleumCVmifpgsKxL/FX8oV8hLcdwG/QjwzDX+D1dz8ZRyiJJ4Y+H0fUYANCJ9KwWHITqnHndfbbXlSxCirSIivcRl730CI680P',
        '801yCIIKpez+t2jjX39tf1O55YjWI37sIKT66vK1zv4BRLpK2/eS9mKmXp6vY6/hdQR0IkgJO1HFJsRnoFU9l6uihlghtjybmqiy',
        '3C1hrRcNiz0RzwbEkMVDCKl9x0h8sMmzm77DRCW1y9cFHrVIBH34RHvVs7WPnzRtn4pjVg1veLa2R6+dt8Inu+lEokzfwUGIJbSw',
        'RKLr0gRdF0nYdeUcBT0f/D+6c72QGJ0AAA==',
    ],
    'mp_tok.py': [
        'H4sIAAAAAAAC/9192Y4kR5LYe36FTxBCR7Azs/K+hsnZPqrJ1vRR6mN6pGQqJg6PzNjKjEhGRFZ1bU0tuAvpQc/Sky6AwK5WAgXp',
        'RUsIELQCRt/Q/ID5EpmZH3FkZHWRoiBBBbIqwsMPc3Mzc7vc+5NfnOzT5MQNoxMeXbDdVbaOo37DMIw38TmPwj/jScqCOGHZmrPn',
        'TrrO4ixlZ/GjdqPxKRVmWI+54WYzg9+r1sZxRZloa2bhuagS9zqdc3bCvE0XHprMify8YvtP0zhiQbjhKduFUcR95l6xhF+EaQgf',
        'sG66dnrDkcXg9e3jL1+xB8kW2joRu0jZabTahOma6r19zLKEc9eJzlOAMd4nzEm2KQy42bDLMFuzfq/Z6XQYj7IkhPH4e4/vMhgv',
        '4+msweCHHlnx5+2bJ62JLDcBDHaffcbj9HNL1N9xVv7Bmq0Nv+Ab9vDsVAyL2Pri7E0rza42nO0S3kp3mzADFMEH10n5Jow4gzbJ',
        'FWKSISb3KU/FGFuFfP3jrZ3E8TKeFAaKoTUNpLGz4VmGC0EgIFgsAES4jnfOTEd+ZWHKIhyYefuMhVEWi6laMM1tnOzWfFsAWE3Z',
        'pk8aHMfdOBks1ozQUdewgB8z4EkWQtkVrOfmqsmimD17Xp6pHcVyhLxrjQWcTgzA1oxyrO9PWf7BJMoDWoM5X8aJbzUJa0RYCY7V',
        'ZGnGty033ke+kyApekg/zsoJozRDItvwLUAj2gVhAoUa51uerDiMxyMv9sNoRXVOgfbfwf9fsF0cb1KB5j387o4kEwnyV9yWJTAU',
        'TxqNfeqs+EzxJms9ZdudDS3auyviO8ZarSSOM/b46St4RKzg06LVAojjy9Y2TJI4SZcNifvajmi0YkfQPHW2uw1vbV3W73SWWHIR',
        'e0CUfeDkzu3d0cx5pT9A9DmS4oslCphGkMRbZtvBPtsn3LZZuN3FSQY8HMUZLXfaaKiyZLVzkpSrdy/ebLgn6siiYB95GSJWFaw2',
        'saueUbqo5+1+k4W7JPZ4muLaOCnArT7GuvnOydabUPeQgqyIVvrtStfLwi3XcEb7LUweeox2jQZUamMvbaAYIDyz04TuRYkfJpGz',
        '5aZ6d9wU/5qADaAB27YsS4O7s714u0UhmLJHjH0C5Py1M2Ong06vUMd3MgdrPK7UaGTJlRBqsm7CV/w91rQTTuVnT08fnbI5vrdh',
        'pB0AYCYG+9VXu+tnN/d/Tw8v6GHxj79KqZRKlvd//1V63/zVL756beGTYTWkJH1KI50i1c3Yj/v5BPjYWW0BemBaD6VZGfa7AA5g',
        'vvvKt5cEu68Bv6wB+PXpizfQw+P22dsHDZDN9uuzZ0+x5CchoPHg2TP7wavnr6GDhUES1GgyA8Qh/lGiSxYJ6Vn8oOSdsWw0PmHz',
        'n/GH5Ent5kzlm40FA5pIkU12HkZ+EzC9ixkIIiXBmnovbsqNmMVBdfvecicFVvZxi+51eqNWt9PqTRjxOY4UB0Hohc6GBF2TCblE',
        'QxVe1Cg+hx3qnF9ZjYdPAa1vXv769MXTf3T6ipBL628aQqUwcVMdQB+rXdaK09RCpCq1A5+pmo07LLy9iCNe91vUMqym7Pt5CCwP',
        'sL7h5yibzTl7wbdxlsDcXjhAnNOH7KLXzAv7VExjrwNaVtGBE57IrlpYt/UUto9k72Wt3qAzxnqdgT9xpp3hYOr3+oE3cv1xMBr0',
        'OqNBp9MfTnt+MOzyqds3BGTM4N2uN+6Oe6N+r+/3+74zdif+KBgFvdE06E4nk2m/63WcYc8dd3s91/P9idcZdga96XjAO75bQYKR',
        '0QzzmdfMM59VdBH6oXPy4jdPHz990FJVW1i1NX3YElVHw36fT3ze80YTPhj2R24w9gbdrj92+h4fjvvw3B2MR3pK/d547HU6AR8G',
        'bn866rt9z4WCsTsOupP+uNfze04w8MfjoTNwuDP2oNpoAgWe57leL6hZVz2dL3CfZv18Cqs4Xm34yQrLW/3WwG2FGS3EtB9Mg/5k',
        '4va7APt45PGez7ujget7vc6k2+8N3Ol4NNZQD0ajcQBfpsPelE/4eDREwEZ+1+tOp92Ow8fdIBgGvYnjjB23FwTQZ7/f7zhB4Pjj',
        '6RAH3UfpJs7Wh9C4wWDUHfa8wTgY+n6v404ALzCAO+w6rjfwOn13wvnEoxlh43wBH3O+e835OftNYdY+FKZQ2AKCVBVaogKfwEDT',
        'qdtxPdALJ4Np4AwHHW8Mixf0uhNcwKE35a6e+KjXdbwe7/d8ICdnNJwMOoNBt9fvTiZdmKwz8QAw3xlMe3w4nfQ73c7Un/jOCGAf',
        'B+6gW6VABZt90c9n8Q8uge167WE+BSw5wV9Q2uq2hw81M+HX6WQKeJ5OOnzgBZPOCP73xgHMwO06vjt2gWccp9/JiQ5QOOl1u2Pu',
        '9HtTzw8603EHCLPfC0b+GDlv4I6mY/hDzOX1p4PhuNvnU6CLyXjYn1Zn8bWALJ/Bs42DlNfuofRQL11GStR+A1plUVrwzGltsM4J',
        '1WxBs1a/PMOO504mziAYA0mOnW5vNO56w37QmfQB25N+1+lMBn0X6UrOcDzlfT7sgaQYBv0uAN/pTLt91x30uoPRwJmMfT7q9WCJ',
        'e05nOO64U7cH/OT0nCmM0HMHRQI9DhWgeuh7frc/7Xt9B2h1NPL5oDOdDlzoqNcbAsomUx/r0gQLayxsQSnBD+S2+PpRwS2qYadL',
        '2MYu1xzUBNxuNoDnDfNRed+jch35KZWDWsulrk0boAeGwB717o0P1gCZpVD65RPmOR7uWhG8g4H03PGsxrOXjx4UNiP7ydNnp7gj',
        'XdOE7ibzZ8y4FmPenMBcW7JRK4KqJ+UtVa7kHaRutdNI1Ymojgt1jvR9TDBUe9T1LvpHeqoVrNjNOrg52YI9sElbLVEJ/hZrnaSR',
        's0vXoASdfHqk81uYvzoE1hK/D+reYaCPcmJ1uLxBq1Xb4vZBbxqNhs8DIFfHt1EZs3UVM91xryno0AabgXwY8aUtLbr5E2eTckuo',
        '92BRveJgSYFRa3pgtWZ2AFpUGAWxhYqcKZhFvAOPRNJu3QD9k8XFXM7Wjs/AluNptrlqo4mGHZfVwoA0QVICldIGf6nIBj5AgIXG',
        'HlATUGMLPD1TNiPTxpP4ostB34ReVHF7xTNbqaAmDG7pignNlZmAb9d32Pt0xhYbHplQuy2MTxvM+jACKWu+tyyyqt/D9KHmEox9',
        'P/QyM433icfnNCkSynNsHdn0DEU8CYOQ+3M9AUBQumawGB6IGwJF6La5Ri3npZ1nVAlNuyYg9hKmJmWWoZD0qE0iJccMVIZqtXIG',
        '0WEStIZhtWFGWyczBYvOwZLkICWA46mW8fzMfnT64vXb1zYY3wa1ABCCubI4+fsdSLo9GKam8ecnbSKxk/V+tQJMB46Hz65h5fhe',
        'h1kKcKFR3cZfJvRiFYHWMzSx6qKzRHFPEtiLd1eGhZOlToBnuCJHAwmvIGAlVrFKjhHCcQE0G0BTiF4H+Gb78WWE/KPbaKu3CCEA',
        'V6kvsVnlSW2GzImwNf/NC5xIbQ7Q/uUTsWgGMR3poSUwBIoC4xoGvvkT+H2xmE2WN0bOAMKEPqU/obD6oWxW20vJoGL7yLlwQjDx',
        '4Nm8zq523ISWVtu2kYVt+2bGVk7G/V+xlGdMQcqcAD2AjkcDSl/VJgSy8rhllIaFFSzJH9orxXMZvtoV0CtRswxKkNyyEttjS2HV',
        'jkKsJSxV8qmYOwtlEZTUQ1Wl4l0T8CuN0mvxF9ZrqxasKTy8oc+jLEQiz+JDE5eZ0lQmicH9Cjr1kpeovfqD4NzHxf4lq4ADQiwI',
        'yD9/xMBG5gIO9w+HPUJlvXowDkEIgM7Azs+prFcgM0sLNwQAkTqrim3B/UUhDGOIhVzFKP8qawd9iK9KJkMVQyIXhCDgNz1EgIEg',
        'UHe07kLwBKrZNXxZzLq9ZQ0eE+7B5gGjXENlWckUH3Yx+t2jFff1BvorOWGEWwv+NnZWhf7optUO/VTsURz3qExtYS7OzXyPwQvf',
        't3FzhckJ/UDt/tXdDBCpdrOM9k96tlMAyrQKm5p6sKT6sfftDDZ+OwWKRt5PTXTdSuUCfcqg5N6IPQ/ATEPQBkwbNn3o0kkshPpx',
        '++3jdpjxbWpa+YpD0wXIO6h0M7uGZjfGEj046cLI+PsMXqg70R5gKA/fFAMZCJlhLYtIhH415OvE3jlhUoQYKUS6cNtn6Fqt2SDR',
        '2y+3x1JV6oadgO0DPatNUA6LQNJwibNKnN06NX1V015f8fb77QZ32tuq8Wglqkn4dVRC6G0wZ4UDOZcI5hLt2k6SOLKOmVcR0F0W',
        'qxBNpW2KhphS9yEM61ZLOScQ99QuAUkeb9sAjbPfZDaUm4POUFQK/ffoEAWgQ2i9AkZBTzYOEcFEoZ5+k7C4MbFwtICWy3a635pd',
        'xOZl8b2IUqJeoGl7xxMbgzHzBKMuZgBbQ2ZG1II6EE8waB/+98LpcL4o1dy1oQcPJTJwnUvkAwaAavCxqtOxqrtsHohBjbm5mjgC',
        'ms4BJaaCS60nfrFRO0vzZWpipGkOdmqnk+vrj8HwCyMvoxaMWoBsiC8xDgc2L7TBmMaeYiEUgNjtgF+Zs43lNi3CTr+Drn+HNm6E',
        '2zfBNSOZvXWiK5R8O5COIHDRP0tVMuecp1q/pwbIksucu0vEkvOyqApbwQLmjPCYIizSLgJ6nxl/+O4P/+3Dv/nwbz/86w//6od/',
        '9uFf/vGbv/rjN//uj9/8iz9+889Bc6UpmoIuL2koRavLRmH/RkzTiBb7fI74K+9ObsKd8/IMLvMuqWwxg1ZL7OtyeUBxEmG2WMh8',
        'MBSnYl3EItIn0JjkZ73MaKpJIQWy6DbTTEjPsjTaS+0ZHtrbc1BjYJtIAOPp/E2yh6Xn7wEIOz6nV1F1fQXFKAsqAo8+7n36UCvD',
        '5dQRRTRzkEg+nxvPpbsexJ+7DzfZHDbdzNnucJuoSK85iZT1lSVYnUfAI0Agqy0MRKs+N0Tc+jNs8znFBtYUk4WdDhQh5VRxQNJ5',
        'GwqLJPFlOr++sXKig90NV67i7s8XXRiiGBewz/mVNDbJ0KCHcU48Uo6isQvVjhvWQL24EiiZUf6QSonSu7SUBZuTNENCIXY9x18C',
        'IEDd49NHv14I/VUMJrcEG4NYIPON5cK4cDZ7mPySdiA1kYKRBYRKoGM8HlWkMsHD8AsjPScpQJsokkbZyElQHgUGyCtRjV0j0kDx',
        'v0ZYF/eEjnAPDI580EP1E7XgjCOCQa6J/QaWHvCSvwMFWFXg2vudD/aFSROXOspC9LWUFEWhdSnewR47gU9COOs9cH59PjvcES+E',
        'sDiHJySRva/0jBvrCAJo4rPPhpMbEdRfX50A71zLQWfDdi+4YYwWgV0jZu/hI2DGYuxUlmgw7i0X905nwF1Oss38Hb4W96t7yxl1',
        'B2Un+G4U3RSg6CCpw/IjRLhs8NrQ3/LNgpYUZ1/YP47Pu76HhVJEqLNCRyg7xA7EPv2UTa1bmyOxlpvzqKb5o7a/3+5sNNTAjk5J',
        'DEpWkjwgjLjSZg8VGz97eFMlgaSN509f2K/fnD5vsucPfmu/fvtEPJydnj5DN22/ycagFDSev3x19qV99pJct8aLl29foIvk7NXL',
        'M3p48Pjv45/fnL56SK9vfys/09fHp2/wz4u3z0Xd3xg3OKMvQci1LpMwA9ELnAKWyHsQuaC3o6kgrMMwugCBGYOc3HAnwSQnbXLo',
        'hBIwGjkX323Vi9Vmzwpqgcp9IWc20EMbRn+z5rrTDWwdUFWYRZdrh+RJkPCv9xxlSyTTS2SO1C+F94Uahan0nzsoU0FNSO6l7DyK',
        'L8HaW3EwpkAbgD0I3XFocDmYmQFWEf6CbiN+CaWg1cN+kgFMYJmhawEp+B1sSxyWX6fI4I7wReK4TsJMsnveU7IJgaYqi7Sfi3z7',
        'kA12mz1GZhEZiJpfoymbril9IwKdKdnFKf8l436Y4az55qrd+PLBi8dIDk+e/vY0jyB/wk4dGmom+2yKTptqowKqj9MUE0UugLhI',
        'bfvw/Ye//uEbXHn99zv6/V9/+Evx9sNfiPcPf6fKP/wX8eXD9/T339Nv8fzf6fdfiRLsUX7Nn0Tpd7r0u2KpHKv4LL6o0YvP8ouG',
        'p/Am54Zv/+HD39Lf7zX0WPJt5Z1qSAi//fCf9fdv4Zua57c//IWE9tsPfwctvqnpQWJTLodc+Vm+9K0Pf/Phe1iWjQOL4DmbFtDl',
        'ZkN5Oy6GTUB7jmCPQAVbUAUJLDmdv5GAiImrt+KzAOa7ynT/ujLd7wqTzZH4fQmJ35e/Fr/p6Qn6VcQGgG/DKKdwx/P2Kb2erIBJ',
        '8AF1GIwhkJ61EbP64Z8o5OLfH/6pfIdF1k9/S1/+Uj2p+TSWKtRQki5SiQVYbNpu5xP0bu7mo6Kt8loJtLmQKE5utKCMS5WIA3MD',
        'M0BA84PqIukQnb4k25hJuZmYaEb6vszJzNbApOhFCHkWIkcry8SLUG8u5Hq1HyF8aDsUHRHK91DwGN/FpYBwGFZZ88GGOKWmABi0',
        'FuB/slCkageb5KGPDNQ2qofuLqirN5d6b5oXg9UZVTQ3GhxG3aJfFwBQNpIERL3WDY2jBuhtxgfAp/moHaaoqZCgtTFL1PTWQpHw',
        '1ghgYP1IyM5Riwa9XwUPREKaWGNzQXAvrVrgztF06xF5YwcBmHJQpl43W4u14P0z2JmpsIuPsqL8IPfxIxBH2SJYnM+WS7RKu1RH',
        '7YLoZBKU0GQRThsqt7cxmEcCfBNonKISEcKoqb9kKMq+mthW8s4q3oA1IdJCw5ymkJ7n0k+lmcYURH0lvAXVlFJcGfRYgm1p1SgC',
        'M3rF4aCdg02kDaXzI8GQT8+Bx4CHEr7bOB7XrCNMzqJJf2eH3R2Yh2b7s7DOT6T5WnZTPPD/ATPUg4S+AgfMOjCekLKaEt/nZQ+l',
        'cIzShsmeo2IsHaJAu7Ydwr5i26BiboKmVlILGMAP7TSX9ej30NXKtWyRbT3PM2/bm2QvjGhz67xHj/McGPoz1utYZqGJDfWxErqe',
        'FWTig9bjJYCUmZ0DB8R9BupcCsuP202u2GuAgZuIRwBF6IP4HXbwO1iROFoBZ4pQgfCaAdo435EeQWyJ1WFbA4aHesqKUMnzbQ2C',
        'CrGTdyPkwBdIQxgsgOX40ziMRFzgiq1wVxfjM0q2BwXXyzZXzEQNE/fHlCcXgAPNsiS7yMOArQ6on2P83dtTFFY5q4BVFssS49oI',
        'ToLxCVOjp0Lga1q0zgHzRnlTEIamtp8Qly2NEhiy02StrlW7DQLQC1E/msHfpfAjFkjqSICJQIpqv+VuvsqeB40OezusTeDMsXap',
        'GDGpmAn+z/EsUZyCOcF9E98Okg4WiPuFM3NF8AJknYsT/bNwZy46sBNRH3Kx7osID63W0lrmFF/hBUnxRFMF3AI9UiQDixczWnhR',
        'BVCsX9ob4Yw1mGFZRS9YwnVbkjDQmTUr+VgRkVTvx4hMbFBZfxXZosGaxcSARKCTiKDC5NRPuSriby5mfV+XVPGf7XcUVktwbTRC',
        'kaXIiywwiXZoWXagiUzWqT6Ng5sbsOyGuTy75GC2S5Y2xXZd8Grjngf7K3rZwWDWVqzwPOOhGadwFCXXDsrcXdqUFdttEaOUyt7G',
        'FLUQ1dsK7Er2QymSa0GUmtv2Kon3O9OyDqgUc9vbKJKEs/lnd7rgwao886UhaZoilCYGx211zkjORDGPQQenDOQT2C6Nzzrvr8NZ',
        'p/fbm88NQkiYyyE870Xu+1J/IpC7WGpH/D4ElSz37Tq4KcocHj/2Uhtx2hTLI2E5krbzCs9GNPMgLjl3Y5DtKTbHbK8mklk+Fh7/',
        'Ecd0UhXzhtFxAqZK+89T/QsrCh0Ug8Wm6Lz98OzULCyku0H2LY3XfgiYeIbHvUyMCAutwybaFCGHJkb8bTrkUQgdyDHbpc4OO39N',
        'LiKPm4vqByI2Qo+JdAW7gJHwbQwbGPrL3c3SUnPHhK98zmKt3E0JColU9KVL9BamVQAYayj0th/u+BvxbOYR7blc5kpkXNOhiMKg',
        'FQ3mNHx3Nru1A8w+P4bVtqphWs3GR0+rpOv4EtYgXoHoSRX+0ZKQzjbvat6ry2i6ZflLlC5WsBQfUCjWp0kqrvs7LPJta3lbVODH',
        'da1P02D/YRpvMPXIsD5GCDkJlmjjicQIWgf6wxMgddNa5n2qxcMtJEwz2Mmenb55c/rqtd2fgLgpvtrPXr47fUWFp4++tP/h03cv',
        '4EWGM53UC0NbnZzUxX64CrPCeyXqyb6Kbg98/pyUTZRRQ9jqAZTecAu7rf6Acfqfk6J3pFwQl7fV1luSf/XnnSyZ/ydTYDKwckV/',
        'SA+EEpsyZ1BmO1mcmCZ0bmZC/SB7T4t0SgXIkpLxA71o4+eUcmgSbY2/jNBLDQ0xiVAk21DMEiCeyQOLYudlrc9Z6Kcyx4sUdzyN',
        'DBahyOfF9TZlThcdU8b9njQwS2z4xwwu2pnQaJyLBChCCT1XTTBic6xfLhbnYSXa68SCOHRWZlxqCYhpiqecxmQBbMc6MbU3xPjI',
        'cHSLHLhTymt17COpURiApfQo67BVDqfSH5FEqhlNh+3EdHQDEY3KYjv0Tal6FJRG3EGVwmgdpKnlONckjqQI+NaL9HFSz/LRJJHV',
        'KqiqU0RWffhXuqRQsGFbmSNmGvssaE2MQwVQo6AwbJsmTEPfklOG6WhVqGUuWg57+hOAXwjoD0DPuZt6PtD4F5QhlyfIVaYmYVsU',
        'ZnjQ5W0pdPlcZewSuUhOFTPzSmYEbkdoe63AiIgDKUNEergptgx5Hn/r7ESKrPb4pWjx4FF7kBthVjYO7ozC7pJ9SrYfQna7bUGq',
        'NFSryIMie4SoOQt0mKFVtbPLZH6bRlvjo6K8GVjbu5z6ze9LmFGqkr7bgE7o49eyUb9B4DBdBWxANOdMtCOA45QzLaVs39Gt8HWt',
        'xsfTcKvzOSDdA7Yj/5uiJ6Gr1JLSXRdcECO2lfqSGrvJuMg1MqT31zguAmRL2tUwN6SeEXK48y2tnuFdckt0StSWM1t5HqGvKU7t',
        'svWElgNpsV+AgoROCUw5UbPQnIk1qAouyaE0qXELoT9BBQiKuHV8aURiPodfTiGVdepTQ/MN0Cie3yG+qOahPmrTN0qUKI2DHSnP',
        'WE2uBHlvTX9h6LwBQxu91KOYeCJd88CnclClJlRBurt+htRYBEXqUsK2xo+HSofc7SrzC4xrqHaj0vmFEa4Md2fjgKoPUpli/yoO',
        '6eYBlNOv985GBemJuVVugLyKhZK+LyPKU+BU+ZKHqzUeZIANTuYLOBnrdv7eL4GwoSkos5je52nhCzJkRSbDtXFqzFinPRiCgN26',
        '8KvLR4Ccd7WlX1ApJsTo0pv60Aul7hin774o6GVRiXt21fRATR10VYgh8HghsLgxqg68XZuyBUtRnNoQBkKTiNBOwiVFbsxdjatJ',
        'irxkYWTGssytEXISMp74qLjPOuBpiqsJ7C4ulrf5ag9SuoUh4OgES5HP/qj9m5ePHjxsMkEv9tad93WMOhFZx4doVNZFTpZ3TbjM',
        'OnQWbcvb+EvqmsWQoHBVVkPpMm1a5/poZzIGU2Rzi/2eYiulvBQLgxJXc2kapTNmtsReBjO2DnOuZCKiHGaun5oKxrmGdQ2sMy+N',
        '1RSZhumceqmGRjH0f8wBkuw3fB4YKqYipq6CKiIyc61jDzewf5PbjGIsQA/XKpBwo4Ir6EC6k1RUUk2IRT1d8VGQBHytFyqaYuQx',
        'ErrHhqK+pjS06VIbS7GUPJYnZFqhFlWSNEtmj8wHFQSqbHZNniLTEfew0k5VVHhFZZw0yZWulcflKPM4nydJ+DxnVuU65v4YiYNr',
        'Adrinobj3vKGPX/IzNPZu9kXAPJgOIP/uh3o8ro8xk0eSzOvZcyA6PVGx9XvEzWRHWIZeSKG3FfUFSiL7qzA8lm3hpNyT9htHlzB',
        'DTNGFr9Elt5His6k1LkQBuWtW1BJXyM0GY7IfYQ/y/Jy1tmbaDODXPVTlcWaT4q1YJ5iBYkb8hgK3QeVzkWpeDHJIMctuxABybNX',
        '0SfxWXdyIyn1ug6U2eejG6hahWA2bHcCWEdWeysVSBVNHzjxe2Lii3u1IMPX2VBnC5cTPrGPGs4Volt8LnLvAbmm0gNG+TvVWXRm',
        'YhLV5FHsVR2bqcMmSNB5t5C29EafkjiX8xdpixgXYdkl3XGzxfiXNjgwKUnFgVHP0egzC3e6ya7CKA19TtXkRWqYCSlzr3PzDl05',
        'iAlSAwlW0GZtJFXLWhjk/JU54Ybo2GhSeKNWm8C8f8TYNucuCkDO5ZcAjAlMGkMhaGJRU7CqEGVYIA9bGMxAYm0UzUIHuNut+pXu',
        'ZPtha9LOCUFkU4IlYKYl57XxFQiXwDdU6kaa22vyPIll5TkSTnT18Vhkelx5SQ80Ekrb2Gc0xvmdFZI/uTXBwRKRL/LbS8+00sPc',
        'VDmiRTgrTnzT+AXud/T05wber9e18He11h++1dX+8B9vq/efdL3/+T9UPZE8R2lPIFvTxWzZlMom4s0tx9dKBqmrVsat2HCuDpa7',
        'Zax6+oO4FTGqURa7RZxee2tYQ2sGYxE0oIio4LmLQfPUUhci1JKTSmbxFL3lSG8cGM3widjKWzfZqF8mmzsY0z97yLRwMWGgL6as',
        'cxHR/KGCnZ+E0CeN7qTvfvQ40K2HiSjpbF6b6naHw0QlLaVGNZhVbns4YsqWT/NgPXVqFiDHmxrk4dnQyn1dvHpwtpAOUTi9k1/2',
        'UHSG/wxnUH7syRk60lM4L2Msm4UyOi0jjsvqYzQHZ2fKfWm/jY1HtGFBxLESnGru0pF5kWoa4lSSWTgnerGY4Y2PGKJfXymp86h9',
        '9urlw9PXun51HoqGaUT1YguCxns5mkRQlUa7JHZ5fvaFXm2RhiHaZBrtJJg0FKWDMPo0S1mXUydtSIKJ9yKYtypeP/UQEHtX1/Ad',
        'NlxfXR5tVT2jHxhf1PXzBfbjObAEHzuEBAZm8eATdZVTlar9+/prTg+hAXS4MH8cVdSRPZzo4h2U01UNqu931ORdfZN3tU1MUfFE',
        'f0A7WRMuUzQtp3NI7PeO6Kpp8/D4JSqrKGIL8etjx5XIWXWMoOlklC0zfTHMqtRP96qSvS6PV5MIIp0sdxFhNqdIOjpHKsd+Z9Wz',
        'sO7VAuotLfZZccTKPi3q6DRR2adl1R/jp3GJk2jTvzrMBpb3vgCS8E8ks+PKjt8C4NS+zv2b38uDiifIFMqIK1Xc4IW8ombV2XuQ',
        'ukcHc2R+qtU8ls2IPS5mrW6N8wmb36/LORSJgb5vQo3yuGI7vCL53kWpKPP6FrPz5WFEqxjdyMThLLoKWOSRheJE9kFgg9B9f14Y',
        'C/PtAKRqKkaqvHAH6BFrJL/Sc4tdd2/uFoeh+9AwnY0uhamFm9IEaflaFE2P8CjWPivIYY+ypHBrp+mcMFCWza44OkwEgjti+VNU',
        'TrTESyOQjtWGnZ/RFu2bUlBJI5yGHFhiWHlxD33Y6S9BVxb12KfYGH4TjBIC3ppSn5h1CH8sanQ0XUJs1tX7CmhV6ifbr0vRFkmL',
        'B/tdHhY+4Bz61rgLr+BuVxckFqY4DadYaFGyDjTfFG6koupa/19AsxmxD4uWH9eiC+6NIstV7n7AGJSiZpkFm87Fn/8Deri8nZqS',
        'uOku7uqd2/apkJPSEIH6lMhxEGjJLxVZGPmRdfvswStQdk+fPX39nDQbI8DwmlHiNGR/eQs1xTgfnb1lzgbd8lfUtX26MGBcan9M',
        'Qy6AJxTeYphOKNe6m3pvApIY6s35rWmHHVXST6th5zyL2sPcixKh45UkqbiUhO5I8PG+oDmUinvPLSBeQbhqLhIIWpV8puncPH6d',
        'sriLAy8WL2bS3G4eqZvI5/oJ/UCCcdHbOxg1GR7w8HZ7cSrIpPu8euib6lkVB3AxvEfDqvAgBW4MOaWiR6zJaswjnGf5FE5d4Ih8',
        'LYlHyqwY6vYQEVnLaVaon4OGA6tmbhgdXPYF7XRkiaxUOQc0qaVT9QTaiukI/5UPK1CXFlB7mgb7F0GYu0ZktADuHPEn0wYoYoPV',
        'Mw6HWVJ32hET51L86w8gJd8/ecJM8Q8lgAYuTh7SN0tnkPuxR6c16Vi1FigiFyTMMDhCOUgHCKKc9BgYyfQxjmJcupjNAJJpXX9k',
        'oj6UB5RhHb9KjEygUtCutC3UDrNu45F2XnX71NcxvnofBLf0JldG6iY1XUYiHoCEkWYOcF0b3RegnB/UlD7sIhlqbUHuJxE5hVGS',
        'id1xLoYXaX1i9xaX+IntWwKXqyTwUV4dJMSWgUJrYtwhi7KUknZLtKAjogXINDVEfszlfjcZc9hX1cqd3jBA24xd40xnn3f7zRup',
        '2eD1GXXYkNdm0CdxkSmr8dyPSq77WilQFgOUCkC3FZjRfNjt1ZBwBNRNl0kKuBp1pF1IfvqpXOLeFviup+J6hjp6KaEgfHRAR8cZ',
        '9SrkG7/giL/jbNGBW9+p6rDxU0VOGOid8nPWPXJiMsOLwrY7ClThUqPH1oAlOJe3f4IqkOGNO8CCqljEV8tN6nFH0MIQ7TPY70wJ',
        'jE6ARvt9rvU0UewkK1AcMMxCG7+MtuEMccs8jn3hJd5HZNBi1XYI4tvM1aymJle8hwxrylDA7FbRoOUkNTmc5fF7IHMFtDKbo1sD',
        'Ofk1lLOPy+6CEulWqPeoUGYnJ6zX+H9BKndHP0Is/2+I5J9LHP/fEcV1UdQt6CeKQJwd5ZyLf2qn/SBZkSZzhm8qvOLs0DdiO/Kb',
        'aXhbdPl76zhEi21h4LU/hr5eARlduS/pXxogxCjBetCZ+OeC6NbXr/dhwv3CEtRUBpPCwOMg5FWb55ft1lQt/UtIGKPwxK1laRbj',
        'ERYYxTjalmIE0IbILcRQgBpSJiQdban/ASXVmm4fzNv3VTJJTVsp4WrHPd6K/N95RaHpNsFkaub2kmiLS71r01JjD6lZdBE6bfwL',
        'ckQdcqdzhE5bWGRg3KfQu+oKFes20IFQrXH9C3mieEmeaieuyWvX3K5G+bx5J4J4CukjOh1MdeW0ZXqI066k9lS6ysmvaMwWA2sF',
        '6OrOapUsUlVXnEB1MDVa3V1oNA309DjtgkUpSLLRCPEkiLgdl4CybWQ725YwCR5s/C+/+pj5jG8AAA==',
    ],
    'mp_model.py': [
        'H4sIAAAAAAAC/61YaZPbthn+rl+BKh8KOhRX0q63rmY0Y8dH3InsOvW20+nODkORoMSKArkEuCslk//e5wXAQ4edTVvNSMLx3hde',
        '4Js/XNSqulhm8kLIB1bu9bqQl4PhcPiKlXmUSZaIuEhENSpkvme6iqRKi2orKoa9T/uboorXM/a3D58/YtVnVaGjas/KQmU6K6Ty',
        '2Y8/jKTZ+vyYfb/4u98gsc9vPr1ikdZCEqTPdCYSJrZLkSSZXKlgMPhY6DWGTOwKncU+UwXT60jjR7BlptWohFzLvRZM1sCrFFPR',
        'HlBbYfGiZVFbaF1shMx+JpBIJmYpiXTkM1noBqzK4g24/gT1HqMq4UEQYF+IJISQcn5T1cL7ieloI4gIhCrzLM40KyO9tnJVQteV',
        'VEw8CNhgLaLkj6rTkG0j8NgxDgZGgn9eviEqRaW9gCw+SKtiy8IwrUFGhCHLtrQJZpAyMuYcDNwaaK0tPOkR55FSEKtBUEkWa7/b',
        'atE0Wf5gEkgJeCbl8WqQ1jImplFOAO861mUYF9ttYfBeDwaDlx0f88teFzLNVrMBw+ehiKPlDMGizTTphotu+L4bKnFvJmzOXgef',
        '3/4YLt5+NBvrLEmEbPbGA0tOpDAYok2Hmcx0GHIl8tSzrC25PA0sKrD6MzjhdaAes1Veh3bJ4AaJZ0m/NLpQKBVJywuuCxXiiMc5',
        'QptGvtXQJ8HnrcR9Cawmf/nX28+3hHDX7thwYSDFDY25o5TM1e0wGd75bEGjBY3e0+g9jYgPvl5nAF2E5O9j1R15Gwx2E86yHnL5',
        'yuHmD0VS58LhWXv2TAlpfCZKNZ+I0XVfqxq5x72gBfYOTf4IrUH8U1RFMCFAbVgVUiieeEfAoA9w/A76hjZJaGXYnarluDxjfBek',
        'eRFp7mFiuVTqvtLdelAWj3zqBVsRST6a+GwjRJlkW5vS7NtWCM8L9L4UYaT4jmxFglQFFuIoXgvIcu+zNeyxjJSYT8b4BGMYSDxk',
        'sZh/hG5OzAyVdM4mwZhdME7A7BnktLLBIHIl+NhSmrbo9s9rdbnAvrMTxfsBshHkPJ5BSFsEFDayvU8i2T1nPrsdF4qnnu9mKpM8',
        'bfSOyjLfh6Q93/kMgBTu0im4gxF3U3DZ3ZoqOZ7NpghNN5vQzMDdEEig1lEpbkdujeo4lkHxdnZzF+gCjkrI7J7hcLx4KrRCsdlw',
        'vpvA3zEbkSDPGMQzCwruNAsxyJGPRxOyjSnCfDTtMuBVU5afkgNxuvodsf/ed4UmIT3TFS3QX8IuLuz8EP5+82CzZZFJEVXcwPrs',
        'krSww2UWqfm7KFfiiFVxDvG3kO6lE3BDNbEpBU5kmO145YtZ2cVF/6C0LDtzfeezG5+96UKh3UEUb1A/m8IMOyDvgodMPHKLdOkf',
        'WxTJLKotopojdcYGArF45R3QBMFe+Dqd+b3Xi2P/BGIj+aYP0VLM0k65Tiv6YAWs+H2btC/Zpi06plXCySRMyUHoUU7TsR2Y6tQa',
        '95QgfoNtpDbgmWZ53i+dN8YsJjPmdnlZFHlbDJA0rhyg06j5BPpYcYajTKZD1DdVpHob7SDTIec9KUKMkXoPLvWgzkMHJeDVc+pT',
        '3Tsh9S5QcZRDgaTQYVkhvWIdtp0Qd45HWVKorbWKcluLzxb5gu97xoQtUcorYSLJhckbj/ypdZvc3+UFSsT/PbHlxMWhnPYSx6Rb',
        'L23s/BCTYgcoXdEhxkcn5mR6LpmnrgrYFuUrWf14eYLe4JwpCP9bSke+c36rHHcGQgKfJdEJu6NCgCodtSsrn9UNLVjBkZqClBfE',
        '61puKNddLT+OkF1zfD9ecgRdltd8RY1AfRQR33+6eWI8zFz7+jviAli20h+1Nduldcnb5lJjvOK6vHNhsqS4VRbJirrIlOY2nClk',
        'yF0spKuX7QSIyOK4naLr1kl8tiCNcwDR620IyLQVhwfVEeVKrCCPqMJlnaYwyRDEhs7fMJGiTanPhucJKkQYuij5DVRTqW1UGAd0',
        '22SNJVmjZ70Ze8LnG2brEyN6rEjNfQxFJUtqXHdQsP4tYnvh6mPBKwRvDBzlIV+a0A+K4FFkqzW6LKWTOXrC6UGlbzK476fz1B4v',
        'n07J3lEU3Qvjo0uKMRLf9gIYR1gGSwNaxoLjLt7WCW/2VZm2JwL1D4RzZNtY/y8of6EoZQnKko6qldDKtNknpckt2HrXLFWCTh04',
        'cT6kzn/YE2jXlBukKAf5TimiiWC2lCgTb9HX3t59PeIOFN01tXFJjbOtDyblaPSlovj1PqMVjFJByIQO6vO47pJ8gu30afB33qkp',
        'yDH9jbxYZVqZszy3R0qjDozmnOedzy3zkoOrR1nr7kGnJUyPLXP2y9AyGM4cp1/70eq8jfAyLzTk9EOlQOQWFJQa3hkR4wrjEBUE',
        'NW3PLcW2FTMdJXVhbt3c4HGYeG1YtQ3FiHqmbAVjCKRRInZzwutCqR09pT80QtKyEdJ48CmlaQGn3KKzeW+am5u7wVO8bHjZDcOt',
        'ieBv2e3u5MkBwF2ylXRHD+OillodvyHYI+zI64GstyLvHYK60KiagKu3vGx2TaqUbaqUzUuA4t7JKW5eKAyVufn1u7CZY0TPdDLs',
        'liy/EQE111W4ulQhDpLQPPXx3kF+hO20Gw6HN1WEkiRX7N3ir58UnUL2ndC80FH9gfGWuGvS0Juxa/aRNU935ikvFkjra/YPlrTr',
        'JvJtlNHWAjdgu9m9AqoYsaUCeu/rWeAapf1ATPC+duXeNAzNZbC3vnD/OLebXWeNvArBlCdOV8dji/v9VIzoyhQ8xz+QurPl+YTO',
        'GnpysCSWkY7X1pjKEKOkOaT3y1Bnco8Evhr/+dpnw+sPGE8uJ+M/oV8bTp4fTK/GNJ1eTydXV5i+MLvPp1fTFy8M8Lib/+reyf4D',
        'Y0nO9pwWAAA=',
    ],
    'mp_train.py': [
        'H4sIAAAAAAAC/5VaeW/bSLL/X5+ih4NFyISiJR95Gc1ygKzHMYKdZPIS7xp4fgbRElsS17yGpHys4O++v6punpK8mQCRyWZ1dd0X',
        '+eMPR5uyOJpH6ZFK70X+VK2z9GRkWdZVIaNUZKkSSRaqWGRLUa2VWBVRKGxarrI7lUb/VoWQRSIeGbTEvSseomotsoL/ZptKXBZy',
        'LgtRqcfKcQlLKspFVigRVdjFaNcqDscEW6qq9EajTSlXalbTI8YfRZIHFZHk5U9iPC6yrBK/vr96j2s6fp4rXNHxYnr2CZeE6+s/',
        'Pv/68au4GY9XTMEtXTLZpfjMN0lUFCD0168f/3kBUF6bq3SxHpeVyktxMrkdjT5Fj9UG1Npfvl58vbj0khBULqoIpJ84M6HuVfEk',
        'mLYoXeHRHxtgAHOlCAv5kIplkSXM5LUC1iIVeZbFWkh5kc3lPIqj6knkwbXwRRKl9sQ7c8UpRHodGGqPxJXjjhpERqCMB7qInzS2',
        'mk/HIA8uG4THBuNlD6NQcala+i5kS58nvmVCNiRDSLUWaT+YG5UKf2UF8ygrIeck8FNRRYkqfxYPa1mRXEgIsVpWtfVANjIW5VoW',
        'LB9aSpQsId1QrGSOBSBM1UNzLtmMeMg2cSgWcVYqb/RVlZtEjUu5hH1IsVirxV2eRWlFCB+KqKpAltbJeIxHFZScbipVClumodAa',
        'x3FVJn4tontVi04/cH4GziU0Wa4BU2xSYYcwbO9fZZY6dEJ5F+W5Cj1yEa2QIFhuyD6CQERJnhUQRppmlSQDKUejeq1Y5bIoVX1P',
        'COvrRFbr+jor66scq3E0r2/L9aaK4ubuqYEjkTfHpJsE/iFLkebN86xYrEcj7PAIpRelpSoqe+LiLL0SRkUqE2XX93Je0l8bnEUx',
        '+HIcpyE1DxZZksD0cca5ED+KNPtDzsTF6eS4A6PuoWZAXByE0EEFIJ8OgsDSCODq7wOI0WgRyxI7tV/ORgL/QrWEJqC4KgjsUsVL',
        'V1CMcCk4ueLKFdo1XPJP+lGhozfSP4L38ADugt/+cgGf9iFOr4D5ZImHg+QmrgKs24ymD06+U2LD9rm/Pn+CCQYUqnyOl74vLF6z',
        'Griw0gdtYMzvRLQcbmRnNc+nb0X334+CwWYCAUf84+rD+J0268njhw9irqoH8tUwW8Bz0gqeoKXrIU5BDQHT3PKxhLPfCwR/6+L6',
        '0mqlRP9ykGgM0/tCNkJCdhBNLB1XLFwSrUdiaW3vnz0kFquHAGzlnnqMyqq0HUEemXslnMV28CfgAP6LOBavhU36eCOOnT4BfUHf',
        '3N9qmSUqSWRu5y6k+JQrP4TmycR8q7BazlJSzP0MISm1paMZhYUQry1OL6pUAuJaBeYP/cD8WqTeSlW2dW25YuJwLCXGcE+oUq2p',
        'iTdpMawGkbhBcdlHYWszZcHgGaNzdvFpaokb68KaiakYE5H4WblEBRLng0v7cbF63rMLMshr9vOW/Zp11hL0MGE67vvyafH9m/xl',
        'k9hmM3x+o7DbOXTgEXYcPnRAp8qzxbo0e4tsk4bmHNL5a3EFbIl8tKeuSLGCXHbitJarIQcY79QT4YthewbVkFJk41LbkywK+WTf',
        'mBPvbhn1XYOaUN06oybyzGW1WJuw8zfYoLqPFqpjuQ/raLHm6KIjirdYZ4CwyRAbhOCB7N8HgtxvCWqJfNS0qSSvnmz7b24bt96I',
        'qVObPiAQIN6e9h06QllG5CtkCFUgO9tM08C7ZE2j9q6GtJuH29seYAZAnGI3DOFGrVRRUl4x7jVu6RuDPqeH4PEmIteVN9ksA/ld',
        'TtqTdmLgbIDjkeLo8dkZoTo+e9s8pUjKec+jFB1wVrQfnS5irSOKPUVVUrC0rcUmlNZAIozJy6M0QIjJiifEqSqz9WaYXpYG8zhb',
        '3KHy86+KjWqPIJ/dh6rZ3YIWCnkMtc7NzBWz8fTW1ZfTGYrPEZlXHi3uAr3LfpBpZWgEF3QnfkAukZsq68Rqg5Ie16BaHsSjF0GY',
        '9zKK5TxWtrOzSwti1FvJN5YhJokWRRYYi+d6v2fuOAnVz/eI1+A+7R60td5+QtR6e4rwhUK+vjyd0OXJMS7fnTWX0zOz/MyxVBMz',
        'fesYQkt5rwKqAG3KWDofxKh6cuQGKu+xEj2iBn5E7W7IqpKdBFdWBe93YJqWBwCTT7Q46Qw7jBaVzdh9/uWEpgJedvhAH/8Hq0SB',
        'r8mA//gghf1ojgJmpVJy0azQW9ydBNicT1WIrymBBOgu4C10APPla+5c4sypWfQKlccStsR81WpF/1Lbgsy5StEVq/e+WHHd8IXu',
        'CtsxIJ4MQ/ikfmZbuidDNitQ5EeosDv+sAcYzjyEdYWOiqV/7r3/+unbwb2k6MObOb6fe98+/t/FN+cgDrQr302rzskAl9z1+VYJ',
        'kSu0oxtlHdxk6iFInoLyMs5kRZ7CxaM/Ocyb+qPeg5Da7gBDF/8b/Hbx+eBO7XFWu0XHhIPwqONzlPd/ki0qefcSeDo5O7hJ91Zd',
        '0l4gq9OyHZDe9PBJYCEfy+qQ2F0aNOS+tUmjR26cfha0A72lktRG2207iSaaBhWFklgLD9PbGRfslUtzJvfppttFo4uSZrXO0TRT',
        'dq7WaC0TmT5RsIgSnqpolOZc8sfcY3+k01Eu6eKDuu5BwJIeFp36qZfcob2DpxdU+RtX4fo7yO46Zk+1JyFDJd80vJbTr9QprkuP',
        'GQ6YujaW5wVVA5aMIa/wSRCKGcTRUNKGeL41c5cdyk0TTtTUN7r4/QyENZ1mnSiyzTWoJrvx8mofzTVjuyCdFpDba2+RoVDYRcqc',
        '9FAMOS8Ue0/IM43OUILqD8s1NDcVI3kqcd/J69LrlgU6psMiNjIOyOfwvO02oQHsvvq7B/MOA93FFYAwHS+q10QDLpbUeHzyzrN0',
        'Ga08mBq3WISMk+V9tpBzHwg8vtL9Fwufmrgfmh5VK+H47H+4f/aJlj/0CbqPpzMuv1zZOM8ZVjgL1O4VldU6N8ISZRLoRZNLrkwx',
        'KT0znILwzz2eqJlxVUMwZ0+9FmT3qigi9HhkHI4J9cijRI0eDfRFwmMA6dWDAGZC/6nlOqeRA/HC1Y05O4DUzPlOXd2ce7/9fv7+',
        'NyOWyek7ffacu7y2PGqo7ohDLtCFE5xuXvjIoyNY8hxdjRZsjxgN/1q0z/mpnk82aK4IB+3Qe+OC2YgLJh5a8QyHD3oCwbsQmUjq',
        '6CvPgJoRmqNDtZBPVN0GfIUNNyhKb3RhTvEqDUwLp3VK06MwYM2qipqAjmPZGgX3/WGUwC1/Qa1uhhnmAMeTea7Q4ZkaBUGwKeA5',
        'IHrvQ5lc2zdbiw8pUfUZGq0HFa3WlcaD5Yk3fXZFB64+Yx/o5BlsxYUfwxzmqpKlD2n8hIbc++nM2V92NeV9XvpTNX7niuWmRAFx',
        'sN51GnWBpckw1r4QtQ5E2sVdIxry/iEihBqZB2hJeP7o142KZh0+gyTkf5AQfhvBtA45lHRK1MXdjcVPrNsWlMrYfYBY74IZbukJ',
        'XVptR/dCmWs24GkH3lTZvdqWoJrqt3usDsVLisVIzyENp5mSLf0+H/Gf8tnE7uVD2AjSlEK6htdzGFkvsj4Oa1cbMm/UrkMVQIN4',
        'zgXIVDemS7bqTx7W8jLIVaEDDLmna6LkjUUdpUrmKgyjljl4FVUALG44sG/CGU8L6hijw5rfxrcFx3yfnL/KmrZD+4Wvj3P1NB9l',
        'ge5HSm5IsNyNfz6FlV1n6MQ5P5m7Ok75/Nv4FEWbTe4/MLWJjsnc6aDp0fMdFgMNG3hZr8Gl+hLyIbldAqqsQm5kUAKgmVAdaRuz',
        'Zy2YBikIkC9KOEUQkHBYuf4cR7e6hnwMDOrtf158/fbx988kGWgdLn5O+k/yesB17oWbJA+oULJJQU2NQDemfhp1zXLLanseb7XK',
        'nrevxpevdLY1Uz+2pVevnmdia+zhVc8eXt0C/1S9nXnT5fMnmj6Mm2dGs4h9VzXQhIC0EhH9OgIELdoXTBp5FFtS8nMDjPyxjYuZ',
        'd6yeG8WJrb29q0dx98287c7VIzdWazPLc55rDYttq1nyPZ2fMqpLMsT8On7pbI91llyMOGbJutMOYolyVQc+1Owe/ZjSoZqwDVOJ',
        'lFIMp4BZTfqAiOduW0DqaMfH1TjYkOqEqv+aPN+LwVo9/aU34mRkBnwIFRxt/tpFOOuOnPpbKa7oQOmLk9nBRPMjV5TLqCirmrZ0',
        'EW9CZWyY4/xwxP6dw602wvJYqHwCdUVGL3Xt/rSuFmwr7B0RN/AoOtKMJuYwIhrjE4s0n4SWufqgoZ6W00PzVkODIncDEskXNxOP',
        'ihKbFuj1GDy0tPkijxqsY8bgtJNgLR+z7Dj9CeiKrJQSmK5AVzDkrn7o3+rGigvrlmknJlqesrIMENsohXfG8IQ2ILSFTFeoNin4',
        'DUT86IonrgofdVFpU6wcDgDZguh9jVYG9e0LmLwpowP9RsOoNI8jtBszy7mZ3LZvO/ALh0tpoPdSLbKrfuIM9CEb2kSqAwlgxaJA',
        'w+yMhsBgY3EHAYcDE2lE9MYXnPVshg4RDMF1RxertEmPaepR11V6izjKoREUFmlWJIHdaRVMQemKqTfpFyKkbbu/hL45Yzw2lQxI',
        'ewiRajCY1Qbpi2mnyDA2jVWKg12fpSjAdheV9FoY5UfN6ECYpg+kmKwBtWhNDWJp4+yLjF7Jouyr7JPeZJoJ/Au6LYoME2qG6jCx',
        'N7DoYXPfH+EDdhMMab4w6Z9ccahr+DYOpMY/0ScBfdBCLerSozOyJN58nQpqebji1DGZn3yHKxKtTwOorWKV8rsaHUkoxdd4QBN/',
        'kSBzqqfNWo8lij/D9wiUMeitv7Ipb3BKLm3QzOPa/0+tXehlvCnXA+NtpT6dTL5f7Cx6Ls3sOvSwlhGhdF9XBybiDXdvJzvb6+JA',
        '7CtVjXtuaxHPvJMlFrekPsrxJ5zj70iUR6VZf811Jj2dHvNjcfXht9+/0POLq/diC3r1chKlFpVakMbAQ4apqs806soqSjeqCz7Q',
        'U5OwfwEi6kyC+huM15ABiZZa4yrLA1lxHuzupz3m2cDF2oH+btOzf7ZPpdiOpvUsZranoqX17x2Y9QjrDI92SdsdUfV98YXypqeR',
        '7xDY7KB9EayYb8KVquqZ5oH+6Ofu+IpEHrr8IQzPKBcqjoVc0adhVdaYgrVHJHVsO23miwds6k+8i/tvpcqeMFhHwdHB0Gcat2FJ',
        '/2Kz1cauitqlSnchQ/fb38iYTqsnjRdmDbut2LDJWeUbvyMaehFkCgf+qmfivChiXYDxC74XqMiVvKM3oMFq3j0L4c28Fw1kzEMH',
        'hbqA2f/pO07lwV3TN7E8uo1T1375oRi2ULO9sRCSOqJG6MVwaO0dTOs2pNwxI2NAnZd+1Avd06yz3+mzsO/pDSsvoDEL2JNsw/J9',
        'v2naeV3YdFBtO6lroTpsdG100NvXoUcPezuCrKZ7YkuhiM0Ljz/FJPXZJohC0jQW1ZPT2sg6I19s9DZ5SKMY3fcaoSkcHDbZXK+5',
        '/GGCPmIA0JfvVEPWn961zfaOUQIE2ZgLkKYU25EMaGzkQR+jdaWx4+W7h7bC7LwM+e+zu87Ucwdmk8ZRemc7owNZ6M9nHzKkJVmR',
        '3dqI21e/2+Xe7Y4m3L0td4fbQerdyXDLTmZb9mqHF1/K9NHuBe1JqvZ/okxszz1koiCityEclOt4XE9QHESEeT4XF2JLVkpfBbzC',
        'fXDxihzqerB6zauXg9VLXu2PSy7eXJtvVwd4rwNexhYws62jB1V6PKOhIqv5NNih6ccoou8UKTAHAX8EGAT05j0IzMcb+jX86D/B',
        '4JWYmC0AAA==',
    ],
    'mp_eval.py': [
        'H4sIAAAAAAAC/9VYW2/bRhZ+56+YZR+WbGTG9hZBoa2C3U3S7kPSLQoXriEQxIgcSUxGMwxnaEsJ0t/e78wMKVJysgH2aQ3YlmfO',
        '/fKdM/7mL0870z5d1eqpUPesOditVn+L4jj+t5DVhe4sM6Vua7WZs1VtDWtEy1YHK5hWTNyL9sC2A6GwM8ZVxd500tb/el2/+YXt',
        'alXvuLxoeN0yXpZdy8tDFkWrZsUWLDHdjuk1s1vBlNhwW98LJvXmQtbvhKy3WlczViumuDUzIvQardhbZvU7oVL2lCVSsWu2d0J+',
        'u/nx4ntnn+nlEnGaRTf4+IPQ5jkOOdna8JYTWaXLbicUXKsNW4uKccNKrZyOFbxSpNMFQcAYoxmZXuod+MHtrKg/iNa4gOx5aeXB',
        '6TV8J7wlWfRy0AEmkl5CNZwibYhXCTW1spoprS401EneNAg5e6hVpR8GV3a6EvKvg3l/d4frujUhGkTHAxM8hHlKM8FbWcMDfxwl',
        'g21tJ8Gt2xBU3u4QJiQ+Wrd6x4pi3dmuFUXB6l2jWwtLlbZIkVYmisLZjttt/7nBZ1mvhjvV7ZoDRVM1/ZHVbbk9MjcF4rhD3ED0',
        'YnRaccvp7GUURf9wPJnSxablVZJGlVhTCgqtCqTOJC4oMyZUOaNcok4qcV+XArkS72dsxW25XVw9w7E9NGLhxa3WUnN79SydRwxf',
        '8JqyGsL8sNWIjKvmRnbG1z3lydcVdUBfNCyhABITFTjSudLaGtvyhsoWrnDF5eGDSDMKLKnCpeC72VBQkFRA74It81n4dnR07vUt',
        '+iNSBY6kruCkTUmDQJAF1XHyoW4SxCDDNwJSOLcTikfqw5IGV482ZCgyoSrPpU16vHam9dc/cmnE8dLb219W6alQ1CVdwMYzgeFq',
        'edN2ImffMimUozsT3hNWn6fyjRWsIBIbPE/izq4vvo/TwPDZeKsm44a3LT8kPY2vEFygGZ99h8CNaXp+R4M0y+m9l+t1KikLNKRX',
        'AmzQxpno1aQjPa4KoYlNvr4hAVSMDTTWpSUk8LzLOmfJJQpUAEbQjcErr3WPqjjg21jeWjOtKF9S6Jw1CnqbjGqhXjtBe3M8oq9W',
        'oPnVcHQDcTu+d27sU1eJe6q//SjLv4PGd9e6kzLxtFR+N6lrTyqyaRdKrTZH/rsv8V9cfYmV7KnRGXtE4JG+8JEZd4CzdwmWufco',
        'P+oG+BUOunA8ob8b6A+P0x9GsZg5d37PrE48HMGHu9FfA+VDbbdBFO+sLrmxgaZw7vrPmWlkbZN4HqfLy7wPhftJseUrKaqB1OWf',
        '5CZx2VU8PnWcZvXCD5OEDJ0h21VXErIvYowgER/NI7BbEMcyltqYOM/ua/Ewzo0v4iTNyqbDTx+K89Twx1PjjX0sPaGJlpw9YVdz',
        '9xPfffifLMi0cUYG5r3JSompNzLicH7kFQ/Hvm/h6qhRB9DlZHfL1UYkl2h6dsGu3HQZWUzbDHadRGEDgpm4JIPTcZutwMfZD+x6',
        '6ucKut4dy4GQCrJCv/P5Ku+xK0RiNfE0IODDcn5xlU8cDjdAGQcXALAZe1hezXNqpvQsEIGcT0wOaWbPF36OTi0PWBKdfu4z94e3',
        'm3rlMrt0VzRKaWx7aMTWWepO2QCesE/Um601iyBhRiGFDRu7XZApfo5h5aO1AxCwSa6DRjcOkBqPuoBv2iyTYVCk6cwf+kFEd+lg',
        'Kxh8CQet42sPg4xQOMGGQIbRxjnSjwlFwIiCUIMqWlcLlGfhtrLA1FPhDCTuBo46Lx3jIjgRsZMvdzwW59eCU4nOWYR3sWwR0yrx',
        'Pu1xcz0C7D4B+ew4RhfHOH1u4TLYdQDhooDTTatXk62LLkdrl9+4nn33xY3LY9BosaknPea6kMSmQd54YlVuuC3DSAEWwLXBwbMd',
        'yIlZ1vMadO4kH2HF5+baZH15bLARwf802Y4C/vtoezvrfeuhk7hPp9nbR6YZkmZ0ez7JjrQEZuf0hBP/J9MMFVm7dScMtJQmFR1h',
        'VoXRNNDLZvAUNIXRa+uy7+gdLg6kHklkk+FVCvxMKEt3GBd81yRApcVlmnXKvO+E+CASAtRs9AdAIbkj1Lw8CqSKf0LaPcCACgGU',
        'tentC1ADMjSh6zkCq4KjYMbN1uKBcfLEgQsFPT/M4vry8jKEh+jgQHiUZb/gd0JnXpmTmOGdgpE1WvBb99z4SI+heM4+fpqxeEdv',
        '+ZXE48ydfBoa1r1wUCx4xCZOMhApFvdcxmm2kYCI+NvsrdFKTjKGodIgp2I3SexRSRGfpLel9y8SQYF6mWEoVoWTmjRpupwPrucT',
        'prDUEu/8DE/p9VyrTkwuNjRzvwRyy3YZb7Su4tx53zq4gvy8T8a0xVZfI2/Fv1YcErMcpyJf9lH0nVTEM2wby6ucWtlNK18Q1OMk',
        'Fv3a//NlEYZdk+0EV8lyz56zg0c+rGFkB+1lG+Bumqfp1Iyz0NFEJjCGM3bsykmm8nH+iWd+7h3VXO8WefHVb/zjnnZPquNXtz/F',
        'R/koyN5Cp8ERYub6iA+KsxpqTZKSge/GtXmPsYHwjlwgpjVxFR/vP8X5cX0IESWNqRPkdAt0F/sZW3UU/Ad/thFIGol4Faf0fyeq',
        'VqJx/2aYENxOCebRadCKV7eFeN+h78iU5Hgc02Qc/ryNc9qcrifvTPrHxRaJkjUks1dAawEIIhtuhf/sNzJRMadCHqJBNW0HwumM',
        'X/56x3797WeWoDgOLCRLAv1lGpPHL7LX/3nxz9c+FPEbwU2HNSwewx5ERn8C5q0OIIwUAAA=',
    ],
    'mp_analyze.py': [
        'H4sIAAAAAAAC/71cbXPbRpL+rl8xC9eWAJuESUmWLWa5V7LMOKmzFZecrGqLx0NAckgiAgEYACkxtv77Pd0zAwxISvJuXa0rkYB5',
        '7Znpl6d7Gnr2l5erIn85jpKXMlmLbFMu0uT4wHGc8yyLN+LT1eBq8N5fTkUhJ2WUJuKVKFNRLqSYRUlULORU5Kuk+EHc5lEpRS6L',
        'VVwWL68Gn3/78OtndGxVZWESxpsiKvw/CowTJlM9zHyFFv7BwaoI57JnaBDtn8UyC7jTn9LPNqLdztO0FO/Ofz0XQ7xgWnH12+Xn',
        '4N3PVyMqSVel0BNzGS3jYJanSxEEs1WJaYJARMsszUtMn6RlSCsqDg5MWT7PwryQ5p3oNM/LsFyY57QwTxlK42hsXotNPViyWoLm',
        'sBBJdnCACp/a+lFSyLx0Oy0MokqmUZ6ES+ma93Bc0G8XJEcxCPY8r6IhCybpckmbV4gLIZ6JJP0S9sTgpHN0cHAl+uKo0+kcvBtc',
        '/HfwefAr3p1BEObLWzkOSlmUzsHBwVTORJyG04C2z6UfXu9A4B9tXl98veeXWZqLTESJKDCvnLp6nf4noow7+fM4HbvO85dyHcZ8',
        'oo6nR9KjDTMsKJdJ6dP6RhicWvk0eeFmfi5BRCnvShcLpC65xAkl1FOTmaB7uASRelzdIB86qsIZDZ0kTQK5HMvpNErmzkj3DApZ',
        'BuNs7I6jsmiJ8YZ+RtO75kBUOUTpyC9WS7fd9cRLbtoo0yNmYZRjH/IQ7DxuiXWYR7KMZNF3nYHTEs6147UgInLaP+m80tOA+97J',
        'uAyx8qUs82jihp5om+exJ26jciFCPbYYg7mLMg8zka5lLmhfV8ygYppOVktsZCFcJY/iVUv8dARyfzrxfGJyXlUyx0wJdhailS59',
        'kB1C7gKUu0SZ2uVM5gGop6Ne98RwxEe9pqOu1qRYIEujpHy6GZXfYOkh1eXh0MG2OyMfugAnZzFENBM3w84IHFs2RhHU35TmY92/',
        '7kf/JmlSRslKVoXFGIRVjYc3o6pmjAMaj9U+hHkebtwCNGEDAzptZ4RTqmvGjZp6jM3e/huIkDNqiWm5yWR/Bj4uPXt5sUzcMU74',
        'L331iAPG2kxpvy86T6wKbEfLSubQEqWcy7wgPaEHaAn3qnrx6on1eQ5pc0d+mGUymbo1/9N2bBTvg/Xq8nFdbo1FR94caRyyICjR',
        '0I9t7PBOqRplXWAFw/UebqEdMrSuR+rAUEQHvy7qjdGieZkmal9IKNRpLGWYuEPzUA/VEuFdVPQ7njVpUZfyKNB9GISPzN0dile9',
        'HjUH0AuK05ZYRIoCzDmBFEIru0RWSwyPfAji2Wv/1aihwqbRpHSnJPr9PF1hFzF/S5zgDCfR2av+UBUqcuLUU1V24SLiwpGtaNaF',
        '0UXQDjRDkM7cqZYw7ORURAXv285mOsnL0LHpc8ayLGXucLehQ0RBiiCcfxcdIeNCCuc2hR3catAdib9VDZIUy5zNJBT8RBrDMovK',
        'ACtIczeBwt0Y08ImuJhE2cZPszJaRn9KY4Ynq3wtYelKbqh6aekrlPxRmRK2ll28qYoPuCtP7yY4d/wXZwv8kt7OVmCAOJ27oXgu',
        'EvH8uWhzU/ECbblpmW/qPllLBKClItGdtZhCPQiW1xJZpz/syuOW6Pgn9OMYZzamk4RlQEX7jAo7/AM1KOketcQRleAHaaNleDeT',
        '6z4Z7o63TS4z0rnSNW425A5McVXUpaJB9XpEr8zQRZ90BQy1loC7icxKMeBfkUIQKOvtnVHmeZr3YYtcNPGGPRA3Msyn8ZibswQQ',
        'FAiAYFpkto115RoFJ5q4ARVEnl1G3bgXPUGdOICDheP5yxuM6ir8UPR/zVcS5wmJBtff8KsWOMKA/S1AQwSp6mlOqvzC//DLxfkH',
        '0sZhsnGdd1f/JNzoML7BInN/LkvXmefhVJItB5BhVZCzScKAPpliWRgl956AVT3IjKdRUvFRhgVg5lSJ2zka8oby0P33YI1VFJf9',
        'C78ow2XmemoD+1/vYc7x06ssKiGmlkXAjjk9HzpUAak04IonAtjrAx/hF1mqAoJGr/Sb3kHGOMypRD1R2WW/hlktQPsbCYLQQj1R',
        'C9iLYNDXm8QvhHXo4douJQTUsG97/lG793an92aoQWOswXUgv6zC2NRO5eSGbBcRpvAFNTU4twXU6lWdv4OMW+hjmSfBPMz6rhoS',
        '5I9g2PTLAHCADra5PHZZmvugjp207pOTLoHFonEMvdf/etMTa5zSBKolnGyckUFRbHz0DHV7Ry1Q8wB4ZRndkTNjdky/fsfCZZZO',
        'FoXpp94CMqVZmvJmA35GSbAELAHY0cZLt1ZV8AJTKDeQ1CHrfwpF1kU/+IJhHMRpUQ1el0CclIpeYgAClEkPnM1cvofFDSiwmFQB',
        'TWJiknXtkXyt+LqWVZqgklXs043c9ONwOZ6GouhBDTSchUCzfeFZFoRIh/Ao0YHA9H8MccC7doQppsYXPh6DaNroZUZ8BpDeA1XF',
        'ogS0J3A0ziS7I9K8cbvFUdPzKlgz0XrriQnIAT+goeY/UlVkgRQL6DnsMgVYqaAaZHE0LCpdYfoE6wKYUPa1i0MT8SmEzO9ji8UV',
        '5bvtJ1b7Sd2+npfZmxYVQ4O7TIXXBMPoz8XA8r0dLjY1Q0dDH4cWYcEg08DjTc/SIiqjtRRT7X1RmEExLlz8pVjA9sXpLXwsSDEq',
        'w6Q6iXEKjwwIttg6h5o+Z2vbnBrLUuea+DLNMBKVDdvd2kEhqmmGh0az11jRQKOMDpp7hfGf6g+fo4J6dDYG1RkfDA2bm704WpNt',
        '+3T++bNTVcj4X5pQzbF33B/Pf/5gj1vIvc2iBEc1iVcFzlA1bzbVzVhLrBJtaofOT0TLT0dOxeCk2NgELoC5NJF4XpPVjWW/WopQ',
        'eyTCkjklDvM5OQzUVbnozBXgnIJ3kRdIjaFOoHQrUT/uEf4tqsAWGS0l3sck3g5VOj1Bxt5hg6ZwmkPudS37xKA4mwv//OrjZwuL',
        'luxZucZY14rK0/Ju3vZwrlU9asikUcAYvXkSBJu+y9ZuC/Fm1wt50N9lpXvDAI33Q3sOzTFzGaPFBrYGTdui25wwKaKp5BGIQAY+',
        'MonCRI1FDJGHyVwq7+Zv8Cvox9ONLXlVBzhsnFjlHVeYi3cXUIrhirabG+XXUUddMr3Rrp6MjW2Vcd0qUDDxwmd61Bu240mrbv0z',
        'XO462O0CyhbwmSFqOC5oMo/W3/G7xouzHEHa6L9t+X+s1/Umq3KgdH4zDH5YCLVp3k4sBEzlib/3xUmTFdgWSGyOioMR+7kWe2l7',
        'nfe+j/8oLqIho9PovA9Met4uV95puTL0AAZ727JBK6pqRztD6PXe7V2uzUOsAUbDmfMV/HLf+4ptuGd1VfvNz/+MMvf5nXHbKrV2',
        'zO0Wx5W2OemJ94yPlIo5+T4E8QB80K8tUXtWemEaBDTXNGXPrkILzXDoe6flNUVY3qK9O953IuSTdSii5IYPVzdl8cSCMNqhmdbq',
        'vREbgWd8HUzSojRhmFsSNs/Y64V1VCnpoTCO3fVjBhSVekRnpCSp062jRouTLUexcYIn+wzTSU25srpEF2ihyYinFifMU0da+tiA',
        '8rHAj21Q+heL0r0E6REattXbY1z30VuRaExubfROe+RVLpdhDkaIknQZhbGguwamwjan6CfSmZDhhG2pYttTi23rIBG8YxI6FSHi',
        'UWnEp0zkOKL499Aydt9rCgm7RfMmj1OEHIUN7KZIHqL/qKbb1msxVCEUyXd4cbtKghgQ1sVVrLpsuIfPxZIuPaKc4tTeLj5WNA1p',
        '+urQtp1/M1x/zwTKBvEE/Xqm7zQ8WQBzH5BVmOo4VHVkLgyeNR7k5RUFsMj5Dgmh9+HjyZCY1vMzZtVtvXeq9N5pxXDdCl9xhAKL',
        'wxGXq+kG+GIpFRc1aycyjgszKhcH4yiOeeALvo4K6GLKBKDqFvpWy3RNV3lQjVo81L3ZamuIaViGAXP3Tne+2kT/uondFxy6RbwF',
        'h7tNZrTiR+ktseJWT8Cc9LbYd0ejZ0K1Yt7VdJEHOd1BmeCHqSHT7+xhYoXW0Gxodx5hYVzG3UY7ME6tohnJwj5SAGJroApOVYPt',
        'RVO5zPJ0uprAIDWBT+fVHs3cVTymaAIWXS0zdSbnKoyJU2lcXesz0RfXVisVu2yEM+t7cLA4X5GrO09QCOXmnmt+z3IS/b2drHvS',
        'Ye+0w6FYKxJxfnBw8Ey0/2P/zLIPLn75wG6NCjP0hPPsTfhmdvaGIqiEJ6jkeHwqQy6pAhRUPH7z5vRorBsGyzTPFlz+Ogw7k4nd',
        'PIAiqaqnZ+NXp2Pn3lzyZnFpuLe+pM/itKQbeRVwMq/+qpCucz6fG3nabu9nG3qisDiGrQ1S3cL0mcGNCZZgiLlU2GsGf4wMz6gy',
        'UChhWOu8k3+E/1iJz2FCgRjnMi1TfhHn2vWoS2VOMm4Vn+dkSn9LokkKvP3xM8RNKB34AbKQ4BxyulUyXYjyGQDMONRe5851hqKu',
        'Jt6fRcmUClzQ26o6Q3sF+upYR7+aI2C1xv3Bc125fcPwyGUnVkrDWN5tXPr55BO7toDJaOujQRRvFDpG6cF2M3+VQVVK96sT3snC',
        'L7Iowa8yzcAqTDa20K7Jo/mitOoUG/vTLEJh97iDoiJcSxT743F6RwxXcpf7hrgRc5i7rkr8zS0cKkGu4kt1nRQSJDkf6rsFxSEl',
        'ofBzHVSt9XKl6EsLbZOmNv5J3vRLyvG2Irci1rbmHjVG8yl+6lqR0bueuIPD29SznpXIkRMFriZ4ywgyQq88MgqRKJhTG6NoTlex',
        '5DDg8IrVmKSpcElTEj5x3/DN2REQjkt4lyj0xAvGvjSzRy9d/8gCPXE4BmIln4N+TtJYCV9L/783vkHDNvlRjWIYOQGXxBFWAJn1',
        'hp1Rk+VpJtMyf2ifFJPXDZ1nr8anJ687TjMKytiTaaLl7TfBTdpmtOdFT7DP6Pw/UAbF7Vcw2Hl2dnZmO+9V7gN59XwmihyrSXjn',
        'w+taUICjOoQ079MczcVGcM4V42K9MlktZU4iy4e8izoeRxaYlFjHHebG8tNxR/jh3HwD0y2LfvekQWN4t44h+m6XblsViWq1LRHf',
        '9js+WC8u+k7PafSiDIlNGU1uClqfXvt2g7tiEsYwKHE63+19x51c57d3P13pCzVLTb8Ug2QeR8UCDJ/OBQ/0g/imo9Q6pKJ0i7c7',
        'dhmVmHfm/GoQriDdIYZfoWnuR7SydNJ3YjkrnYYM+lq3udMtiJsl9gpIRidxCluJtqqY0y8eEP5aYzE6TMved8v9We2J88U/oB/L',
        'cfL9Om4m8zKC1G6g4qAlhrWg3CsfmFOS0nKo9NKoVg3QfjyZM+itppQgV04zTuKi18Xmlp7f0/MkBIM7VvA+VNpmWBM2TKA6a1LU',
        'NQU2ia/VbtN8am73iBiaWL0m9KpWTRdezcVuTfiCZqzXMQz/rRkf3I27HYlnurymLP8BWb5RVz22LFO4iuaALNpwz4J4QHUn8k1I',
        'jmtTqpUace9wdu4f5BN4sAId/+g1xloP/2iknMUFe49Ux9LV/+oMyEQPQr7JVRl4eL+W1ft7elchMudeJTVVmmpXZpXE32HqbVuw',
        'dVzkcOjE0f4JFMsi7DsKW7QYXTF7v9lVKVopKH1AmVB0jzAV7m/vOFyyJeuxnLPuB9KRmGgLh20pg2saiIJSlLmmQnGbf1UlVPz0',
        'tEZQVxHKe1KS+NOxY8Vl7Ug5WeRKTTRuGZ5SEK8pVHDin9pKQmcGQW8WWTiRLuXk0lu34x7LU06mqQoAGuQZio7stBpjREgjD7du',
        'HaxzLtiqtNvKUnQNz/HKeiDqTL4Sl//b7vivz16BewFgjs+cbdsDEhP3dUeetkRXntUWSL6RR9OOYxJ5Ov6pskeNWcSgKMNxrPKr',
        'McrHdvftFosoa3jhX/zy6Ty4Gvw4uBpcXgzq3NgRL3G7luKrIEzGAeW+BasEfMxtHTosQ+PJyYlT0UNDiJNu56Nws5WmCS4vzFUu',
        'eChvF+Psi88x79QX+EP3y9C5JAf+iw70qyP4YgXdv+hUFgrEUoRr57opexgoqEh6BhZw0na1NoAfjpWZ1T2GrmZRWTG6CsGbED4z',
        'ehXGN3cT97vXYTQE37yec7YRXh8mmBPdIijoc474gdspQY1LmFUcsnL8OqBdA3DZsyiw67+HVpI0aVe5EYKZSEKVFuLSeVCZUeIs',
        'qzKKATDGuZZjFf11tWYWiQTYexzIVNiIRfEJzfWAcrSU7+tH1BztBNb3uJL7DwdUCPepcJAKa8zIDCX942ayutPmwP+duV9V8Xzw',
        '4F3P/5rcz+5NDmYVWVLd9zigFLcBWnomPpoL8E/phflKQ7hf5/cep8FhT523lLAmvp4PDzl17XB074vBWuYb+r5hTEcvY4oyErOi',
        'Izh2lcSygPRiKIo/FiGwFiuL26iQvrhaobonfq8+LPnd58mqLIo537qY9LpaXD4wDnL+jol0pY/ZzkUZJRuxTKcyFpTWKGbyVnx8',
        'S1cNbFrvSlX8MZz0aKuxaYryQjEq7X4Wo4TYnkSV9CJlSCy4YEwhOcOgNqUcdW0YwW4NiBfdHbqfcex6C7e75CNYzoFGB7WPoArM',
        'cTjfrLD2N+Waf2OfQbi033zIPWGSED1UzsMMPymLIaJPd/4qvvE44Lud/539vvOiu99L/VA7qN8EOBDTfM2Hh0TU4ajnH81MCfs0',
        'dUm304F2Q3kuYxS/8Lso/yvVHG5kccgAnypNBPdwpFj9MEkPMYAlthUFjvP9vortpNRH88sqr7cWMrAN08BCGqiJQiYl5TwXPwho',
        'uGWbs31DCEQuoVxiHF6Gp6jgJNt5GCXog76xXC5Dc5JbNgAbSDbzm1ink3CM30Z5fhMa0OJJQVk8sF+52LyUVFrNPhDu1ctPnl10',
        'XRVxeLxd5lEmZmEUU9zqEUZ4gjGsEEblTexnkZm6cc4b7kqLL/Y1jc7oEZ4i66p5iDcGyodeZ8NDy287HA0PbefH4jQ0NB7d461q',
        'X+/hds7+yzAoYJBXB16sgcegk8ZTfGEqXtYVFZ80O10/1On6oU4kL7ShdMCBOWDeLVtaFLPTqW9xe5hLlfgpmdErbZQm8aYFuEY6',
        'MsoV3/EilVK/XYQEbqr2bY5IKx5excSA5JMURpe1OO0sFHF0I9vgozY9sBq01WotkqQqOcW50n10g/xNXApXYxWfWFtL6jfO5Rvo',
        '39f693v8fv6ca15cw1p8a2AUSBfcTurzkW5p336IPn6CYElsN1gGxToPlkTlwJbW7xGaLeGxNaoGwFUKtxGeRsLqzboHh3vNkVlG',
        'yXjuNHzopZYt64p5dPCEdr48pLu4rjztkdr9qEvVHtZVHV0FKIJaTsk4HHn3zaLr3aL3ughQwOrKDXnz98gQ5KduatK8myNb2dpW',
        'xVLlkxzivA7JzfNYShoVt6ZCr9LObbalY8eOcD6unVFoseaMLfmReJeCMTixdAKpK9vhLYtRZZ4XMs7+i3ZicQT1pRIoMOnz5xW0',
        'IiQEx4PqKR+RoZX5XpC/0cokf2uFR8ZPlM7y4lpwaswPdZKrChwqd4w9L5PP6O/YGrACJzV+285JRskZgYOLn8kIKWJJhKxEZbvJ',
        'o5bDYvhCZ7RjiY7JedlnK5bKTqz3ZJe2qNROO36Qx4uKMxSMI+CBzTzk5AauqyroeyZtTKqy+oj0MCqmOd4dpqqohnmQj463/McK',
        'H24nFO6Fi8esA9kpalVQwooSE2dcepjigYhEpTgVuLg0kNG1lKBnAKSBinpshpKU5Hppc8STkGELKrAv/8hid0/xy/AQ1OpT+LJX',
        'YX1hZUHm73hmmhmMeWxjzC97MKZuq5Ir62ns09+KZtxAVs0ytPf/UOKE8fJ3fXxrkf+T/BjBo7q579UHSCDknOmf39M50jv7/GZN',
        'LxTsMa8uvantJJLVU0scW357D5RgSZxmw5b8h4cgjFLDA2iNDWXId191PpISgWUfFGW0DEvp+Y+hbs49bHhCJ9YloK01T5TW1EiW',
        'fTNSk7pAacsTpb3NgXhGY+7RKqpprVkeSLCy9v45qYme1rJssfgD5+GhSiQEr6ksv+q9kn7suVF/j7fXSgHKmtrVfPWDitACX5ES',
        'pzqdScj8eYJDrfOpLahGGuD39vvflYcL0CWbYO3iQrz9Z/vyov35XG0oBZQvzv8RfvDY3QUIC3NAsZCy1dv01wNkPqHEAvoaIqIo',
        'dsOpPd0+ylPrKElxne7VVKd0hJQEKNwaSnnWwA3fgdLUOCtvcfqA31Cl0jFe4vYPJ841z5edBjL3xBLYd5BP/XHqVGhS3owgpTMu',
        '5gQ1liR6EG6ayDYntHHAklvUSW4MvCGnRiNPFiHcQe8xEZkWtWNq5YBV2zq1AuMMq/vCmQDRw0CDQlb5+Yb+4IWg3s5umGTr80Pi',
        'DHj8BUFmKPNpeptQppmzdXYsle8wIhQKTXtvBRroizBCxm/JJlR/f2DXAmwd8Lql/m7EtIDJxxAPqsum2l+zIs6g2TcMziydrytA',
        'wyMuzcz5h773mMRhUUSzCGgopbxPOB9y13cHN4DCw7oxiW7NHPf+ro/+DlIjYQTvRJHJmKKI5kuuG5mVPeFASdPu+H9AG1Nw+AZ8',
        'uLYX8vGt0/jQb6o+IXMmeVpQuD8wX2xZaszb5xq9J+NVs4Z9Z1yIVYHzBxiUW2rTzKq+fnhKTZJ50oqO7aTTyIKBEdPr/EAJGvSq',
        'o4/0CZ4ZNSThMX9TxT/P58xEn+gt1wkyYeaHU3LlVZ3rqL/x4lAK35cVZZhbH/vua6wcRJOvVH91tqdtuir3NyUq+W+VFJI6FIY2',
        '/aVz6KsvmkOfvzbGVpo/1cI7YKqVM0cfaPsUM3y4mYqz8leRBxDkIKB7xSBgeQ4C2sAg0HFPtZsH/wfWKSNEpUcAAA==',
    ],
    'mp_export.py': [
        'H4sIAAAAAAAC/61b+3PbRpL+nX/FBKk6Aw4I6+HofMrCFa2s2N61ZZWlrJXwWAgIDElEIABjQImKy//7fd0zePFhy6nTbiRy0NPT',
        '09P99WPg7797slTlk0mSPZHZrSjuq3meHQ4syzpbFXlZiWouRSnDVCzyWKaPlEhlVclShPidVUmeufWIXExkHCfZTIkwi3liig8Y',
        'wKzqLheVXFV4pkSeSfGvy3fnYpqDUSauD18IFUmM2iqay0V4LM6uL969vxpenr46e3viLWLHGwyWKpzJ41pGMXwtFkUgWUyvuBfD',
        'YZnnlXhxcnVCn5eZEu9/Pb/E53xZYQ9qmVbqySJU8yqvVMD7qaf/qfJscClTGdGWRLlMJSQtpbh4f/b+7CUEEMo8OxL2NFnJWEwk',
        'xJeiKsMkwyadY7PlciZVJaYYVHOQ/TG5r6T6Q+uPd6z1NUzlLQYaPbLSojwjLS3DdNCq093BuN5Kl3eSFdjs+lHkNzITRSIjqX5i',
        'ZtW8lBLTwE9V5TKqliU4zmUYY7Ww0k/omAocbBrey9IjmxhMy3whgmC6pAlBIJIFG0mYZXkV0i7UYFCPlbMiLJWsv7OKzedFWM3r',
        'z7mqPxUYTZNJ/VXdt8yy5QInDNPJinqoystoPhiAyqOJXpIpWVb2nguOeiROyixcSLv+Hk4U/bUhf5JCesdxGoGKIMoXCzoFJU6F',
        '+F5k+Ucyw6d7Bx2aOKxConixk0KfBEje7iTBaRDB1b/XKAYfXp+74px/vz25fiF88eP+gSsO9rCno6eD8zcYSWVmn3pvzq6uzt5f',
        'BofPnIH+HLwm+k+fB2wErohgCUJCbbIMK9mfcjwQ+GnmjaIxpiZrg5jx7sPZezzUTweDWE5rI7TJu2Ao5cIwm5Bl+uIcJsPfSYqC',
        'RDBn6l2Q4mmW483SfGJPrU+Y/Xn4+EmMOex/luFFPzGYFTi1Er7RDCZTEXt0op7MYnWXgKM1fNmdRj/kQkm2lM3gQuLQfLY/L81h',
        '4bYdiyfConGzsAeEiwPyPBs2UU/MMIuIRhYECRfKGo+sLM+CxruscVc21kGiWAkC+8/Ecx4b7Y37Ehpl2YDO2OUV9JqlhFdl/LhW',
        'N8TVJkW6C2DQmCJvk0iaTUfTGTi99U7zbJrM7MePu7s0U2ivET/ftlu99IK5vLy4ssFRDyk6A3Yy5tflxhJ5RWVB+rAI0jxi3/et',
        'qFhi7E4mszkQNs/Se/+XMFXSLMKMAgWkkGAVVfanm2Nx600xDFFIh7deogL+DvUGRZ5k9ECChbhlo7px8QF2pWIvqeRC2c7nmjkD',
        'sd1Zz+gTT3LbaM0ljRnlEi4HGouVTdowOgXO/YKVgM7lvfj16pfhMyYV+VT8QWR/HAvbhLwkpoMe7rsQKZYrIkkQ4qI57CUCgUv2',
        'oMEbR15zqeZA2IbG8QhYaeE0gTlESbxyNb0vRmPX/Nd4VUTOPe97d0f42m4JJ3ByMotwVra1rKbDZ1bHtglLGmf3ZrICtYudtBSs',
        'bVqnDLOZtLM1N4OwXlgU8ET7jdN3QOygfhQl/We0r/rZjfB9yDoU+73jygovLMvw3sYKOK/mK7HtfidWjjlKFS4KIDonGDZlAUZY',
        'Cvx+V3uEHrQn2zoLgD93coJJigzZ+hDcJTdJMJdpjGk0chYsY6Kq4qJDhbH5/Z0e6OikIMTqgR2JQd4ib8PUwgdAHi3/mb0wtbrQ',
        'UXhylSjIvqZkCFJry6a5sIRyZFXWmDdD0V680N7MPO3CGZszLpLohra+YsoVUZIysNYKePQwFTjj7rkwR8wny6LPDh3fgfbNHct0',
        'cNroi9MRaxlbREkk49HxwZicCNPoIw70Zw06WR7MSuCOo5GQHS5osiWbMYjd2RWk6jVgfOOKV9g/HntvmMp7xeOXDaj9Jctc2USH',
        'oPtG/xdX94X09XNGoaOnNV/fsGcup9kam29lME+Ag8vFFi4kavxtjLYJ89D5EzqQejoCl7I5/+BffXLga5LavVn8Z/jwyUNOaxqH',
        'XyoZ66Sl656YmkeKMX6HV9MPDWKu9b+Z5f2JMGHTpBZryvCOdgWidQgcdACM0LYF28140NCSK+31kDFskXHPZZ/Akg72VmtgzZPh',
        'ETfiuc/pXf8J6xM+fNOnj1VzLDB5lZfAQ0q8wrtReByKH2iNsbOu5xElH+N1DKE8hpOIhNL7TMqYHCnzr8qlNN9hR7HUI33EBmEj',
        'CCJ3dGOPGEG6/j6yiJ01HjsegGOSSsTsrT/fC+1vV/h/Py6km7uNOxvdartNwInWJ+NYHzp5clcb8mNhR9Ho2OUEbiy+A3pErE9X',
        'HI93bGhjf6iDBCEeKQZpoMkT8ihalshlI1S4lCDkdxlXxFwa9uXJ52verKuCb3DphhNZLRT7vGu7eolRfgN8zWgB2/HUx6WUf0kb',
        '0R+mnOIh5/z7Xn/apfihlkyi2FoubCudFx/dInE//jl8ns6TPy2X7eUxtIoybE7/9WUCcIJJPveuxM8gas3lZwz2KOdkEWxb2jSR',
        'fA/3xxRU6jl9A1psGBC8+gtGIP4LClpzuRUpbLFdM33hNHx7nPUFYRwHjALpCDyw0lx/2JyT7ZrRgVDCEp3qPODA+0vckHL3myHC',
        '1xFBKp2m3usJ1R2XSEZOMy9Kga/2Isn8fcfxkLpjw1xnG9WizCHqOlQ9MTvoTmu8Zdt8kzecuLxY97FLvN2aX/8JCW2yujK/C7K8',
        'XNh1lVkSkHlkeZRy30hZxMlCdUCrTSHv5rKUdokSDGom4dfHkJuTue81KWTdBrFPDo/MehdYrxGChvUi+R1h88XowojiEEcNuoB5',
        'QgCOByDrBAEj2p5xKmQx4DFkKvgLpEvzmY0/UZoUPAr55JDqf5yOU6/U3aUum/YRc8DMW8gwsynfpN4KM4O8zea4sxVwf8c+6adI',
        'J56ah4Xk5IvrPkLTkd3Rxwh5Fnz5+PCIf1HUoRFHN7PaQPhGj8zbkVdOzdODS1b2jbz3YT6TOBQrVFFDiiQu8sB9/n1QJ6/RPFeS',
        'GovUeFKmDlKoUZwmU1BaBk4TouNuLs0C6ZlfaQzoZeoEmwtSnucTZ6jKx6YbNfhlvgQV1n3qdNxOL+TBoe3U6YnBBRgtwIny4VoL',
        'oAn4a6T/AKV4YKyZyru6O0dVZpjR3DBN8zuYSCHDSjWqQIYdy7KXwezQoJEJbGw7Glk824L68Zl0YiESIjLarQFwp8mIv5nZ/H/o',
        '+EEa7WvVuEhtSKOR4fxFo366zX43jXxsnKqIwkP7mlog6sYnFDTbv4bBUpWqdJ16XQM5xgyK660RfmnSBvevIcUkz1PuhRDfpp/E',
        'ZVaHKz00tMzsN3C6HhHLMQDBfNKgsKcpfnXpyP9T6SXTJAvTmaduY/s3V0yXaRoAOErEFNVt2Vxoap2JNCIemsO5MAv6WP5nsB4d',
        'H469K+PxYSqNIibKNpSOtwhXQCmodL97TvYFgIunOJ4+qacoGXLOeLGerYizePwY9eYTfONPGhOdLRPM+US5SjIJgb/tWLKeghj6',
        'QRuiPve3B53fSfXdCJPpCJO5a50N+3do6XfvarvIW8veoswnMuA0sa54qTJx61Z9r+b9StGzpaQYHfx4NEaCVJcWO+qJLuj8Q+bq',
        'OfWvqa2FgqCsBg+oMrQEadFpKc4ClU8rsgfO8jCAzBhZHjR3jEyv7gq2TamMi4RhWngzhDhZchKAtWjGfpuIdHK2fm7RjY0HmmVQ',
        '99uC3fVf3U1rR9SyLEptJZMEil5mla3ZmM6nD1GBC0mGebNq7memRUmI3GtHzRGYCVr4NFs8C+vUV5dVIyJroJg1pEc0IOvd7x+3',
        'mfHavnFq3Hpc8R+X6xM6r+YE9QVRs/zLntPDjLMOGEdZtXXr3d3Cyci1bD7CTj+9oDjp4VRfwqdpWpM/ejpgMcKYRh9lZ0xTl2Bb',
        'aPYc/AqdnuQvtSvCsJIFAALyNqsQ0wdFVsJNkd8ibpKOYFDlvZil98X8kdI1G7B0uVjIuCVCalOTtDcPOPA69L3cdPoeOLBRGZrD',
        'LvgRky/DA59egw9w/Rokeuig3R9Pa3BoDXq1gQtMlivChqTFmS1dhh0urYFg5Tzcr+01x17tduu+I28rPfRVJ7he/dvTOtIjdqMc',
        'SKaHrF6TkxMU/cDXf1x9MokK02CSaN/ePKY65ujrZFs3JPUlGTyt0R9fkNRo3dBs7Ry7W67OeB4dP0VW68X73+iCW9jIau+FOX26',
        'i0kdi/IHusE7PXmjUwfrrQwVXfJaZru0Ku9W37j7Vn2bPNR7eLKPYh7Grehi59T7z9n7y9fvzpFyLJO0wgCAA2UgxGR5fP7trmdk',
        'RYl4CKuYhwg0NIk/8PUr75La480Nu8VZz7bxpjuu98KAsL6UAW5/xJviMtvnqxI/Iq0gU/bbK01X6LuTFGPnv749e3/yRt+BsL9p',
        'PfgJJeNHOif8yo3qeEMcVIEwIUZwc71/G6ZL81IBwYvS0FGECZL3XN9GiQIpC9+iN3fwR09FWACiKaAS/Zd7S1pn4sixNgQyPxZK',
        'yyElNSEMF/i1CAuFHXLR0H9jA+XSrRHdlmE0p7qXUA+RPxf7LfAdHtW6dyzT8qC+Qf+m2GLUrBMR7LZzW9XcglIMbrGz6e8Ts/Ye',
        'NG5grSE9ARlFImX6CtCIWzeXH35v0LAzqZWuFcBirXjuVPQK8Ka3roWzxrVTYde+vqsmP6bLY79/k+zqmFsP6y80PCkmMEreMN3p',
        '6KtqukLSl7eu+PTZYVu1QBmcfQjkx2WYWs6uA1/zEi4WfXMtwlviL68oK8ji/E4FpDiffq3ttFFhu0s9xecOJVA/iOGlIQzRpz6/',
        '+6XFyOwDfWi+/lPnDElWdaLfVze1fr6Uzmt0hlUcdeNoW2XqT645WGp+s2j63L+6YjOr7gxpT/JHbadoNO8UzvM2T2uLzc1koC09',
        'WY7x1wUhFFkXoiMDQudxZ0ldC1LTkftJO5IRBu1D+NEtBSUqcGuPoppmzSLat4xak2hfYPJ1dMmXJSUOxhdasFI+Pm8/dF7YZzFW',
        'RRqiiIt9eeuagu7wyDeVHdYfsTadNcE45GuhzOVSc6ENv1tVpOVT7+L9u3+eXdavEfS7Ccqts/VtFdhmAdafnOaERVlgOlf9Njr3',
        'TOZahDcn5y9en78Mrs6ury5HN+Pj7eloB5jFXQmBlXk7oXmJAGHZ5PNFSJkP6rNQp6MszYYtsYh1asoNBoAypPKRV+B/utsQOBvz',
        'sKcffCL8jgl7z3uqx2Zqk1hiFQ6+a5vViZjP2mRRlc8GsElJDUe9/YAE942C1xIz9TAIbKAiqD3Zp6Nu34NZi1x1PtAGr8WO4LX4',
        'O8ELvqDzVCbTqXlpEsj1tennQ11mezB/TxecXgwBornt7KgAB/1bRc7uEXE8jjooH4IkttvkqNPW6yY57a2i6aSY28APo2RMWkmo',
        'V0XN716/ytxhmR7zvmGeEHMIM96FOteuGPUZ7pgGo/OUrJB7h8sUQbGDSRwqRxa/EdkC1BogaRWAp891zgOR59rgTh92NK9vzwT+',
        'bsh3/ha0rWvMeKxepfMA/B7ovo6zuffdlelqLeXSnaNesYMh856FF8OCtVZKqYspU+GV9GrY1NLFyrH4hCefhU1/yDAruryrApX8',
        'JaHMfXl07B1MP4u3/3R+qrNV89LsJ9IHKfdRL5V71Gr9EY7wkfPZ7cHdVO94g0lHD5ssftJRo0PegFGXuJ+XYJ5V7xlqk3eNJsz7',
        'kWQt02RWg4SpZEFU96m70+K6GzAVnYyO3SzhSes3Vpq6eXk1zML0HkpFgDk511KldH4n516ATwZrYm9xEyelrQVU5kUDruKC/Kbb',
        'ETRnAVBKR1Y0t7r99k5+jYBCHmc6Z8mMGrIcnVMc93JSUECw913x1KWndOy+vf/f+O4dOe2lUUh9KnPn8VdS2OBBjWV3I8EdWTuy',
        'POvLxHxgqII6rbyVlyzUPL+zmxfXFtTYQt1FJfdsEXawHcRwwKBKKmD4VKeQ4tN89Ig/PRp/ZvvhEfpAA+2VBg833x6N2eJRCGLn',
        'OALSyP9srLSqkuhG2fpmA7Vu/S6/6sz6cWPW/cNnmQsOvfGNpHFktekifSP0hfb4qAd6QTU6HHsqInXDnCippRtD/kCXhsp/Ruic',
        '5qVvfT959uzoYGK1571ZtdeVaveAeAnzBru0AVFYJ6nXSWgdZ1OJtWTNcVmNDto9te/hozQ/PRH7w4NtB0Jv5qplUR/7Ve9fXJC9',
        '4cxD1csCS8YRepUkXFbzvHl5X4wIW0aPuB0D+xhjuZW/5+3BN+ahDyGnldVZNbyV9N6wjjgMpYGBC6/IZjXuwMmiNFcQLpnVna4F',
        'vWxr9BjSMdfv+3sn5QzqzqoL+lYaRAi59xuE5plt6X+tYdH2Pi4TbKCDCtuIkYxZFDg4OOnrtl20+vXNB5HqQNShtqJlHHL7TDc1',
        '6Su9kxzehkka6ndVdEONW3mDul0f8tvqShJ3Vb9AoZuBoaezOfylZh+962j+QQK/uFY/tniL1M/2KCTuJtP/kMXqpIeb/5ZF86nD',
        '7GCA/QQBJSFBwDl+END5BYF1bG4m6DAH/wf7d8FrDTQAAA==',
    ],
    'nb_helpers.py': [
        'H4sIAAAAAAAC/71bW48byXV+568o90Jgt8RpkjOSsMsJhWhnRhdA0gykkW1kliCK7OKwd9iX7ctcdjKAYSMPBgLkIS+B4QDZBHZs',
        'rIMEiHexCWD7IX8hsz9g/4D/gr9zqvpGciRtko0AsburTp06dercq+a973XzNOlO/LCrwlMRX2TzKNxqWZb1RC1ilaRiFiUimyvx',
        'XKbzLMpScRDtiDDK1CSKToQtQ0/4aF1EU7kQXnIhkjwUZ342F88Pxs/2dx4+G/Ydt9U6kFmmklBk8kSFYpZEgXDd7scqlhuTxPeO',
        '1UCoU4Xx07manmBKmQk/iKMEyLMomc4JcSr8UEiMVulcxEk0VWkq7NiPRSBppAyPlYjl9EQeq1TkoacSgC/8UyVSgPpR6Gy35kqe',
        'XoizKDnRKGUq0mnixxljT/OJQQwMZ/MoVSLKszjPRJolSgYElEXMkqlaLLYN1WmmYuEDk5wp0MtcyM78qXKJmS1e73g8y7M8UeOx',
        'WZmQITgpM9CVtlqm7eM0Cov3KC3eYpnNF/6k/FzIDDsTFN/pPM/8RflVrqFsuShfMz9QrRYaXMLp+mGqkszudTCZbvH8JJSBsotv',
        'OUnpaYN8fwHiHccpcAXxeBoFQRQSE3eEeA+S8YkciL27vc1Wi3dfDMWOy2+tg6cH+Dqysggi4H8K6Xow7LmbfasjrMw/4WZqeZ8a',
        '4guZJNHZg2H/Hn3O8+NjPzyeyakaz/MJgW31qCNRx+qcXtKpH1/QSyCzeBFl4JY1au3s7+6NHz19tveKpy4Jdg1sPPZkJqsvEFF9',
        'BJGnFrW+RPq1gepU1jplKBcXn6pa9zmxyHyHk/FcKxQ1jFqtlqdm2DV7GngdMT3zhi+iUHUElNC8sRoMD5Mc75/kvsqGj+QiVc6g',
        'JfAvxmqqXXYPoliFhEv4M0gh9jST4VRp7As/zRyhMBgMmECNiaCNKX7RPdKz47+eHP87EGYPMj+sT/D0YI/bVZLU218d7u6/Puww',
        'TWv/Zeo8M6uY5LMUuw5zwOBkWBZ+qEjpYldPOSgRYR1QDb3yQQN9nEABbRpJFHtDy9L4kil4Ertn0s9s3QIc2piQkUqmFRrsI5jx',
        'Mg9JF/YgZok9s0guCHAmIeWesC+T6ZUzEJc3cBXWwDC1Ldrux5Ef2oGMbTQzXx3nqqBLQedDzG82PTqxZwt53BFBip85FkNLMNvK',
        'a7Otb37+V8KiWQlSz2J98/d/LSxH3KFx+LWtCgCcJGYRLg08sz4KgW7jgbikRpDSoAWme8FEOIamMfZzgoVB3jtsIGj7P+j11ksb',
        'bJt9RAZEnatpnsnJQpUSBQwkUjJmQ6ctp9n/migUU5hng7jY1S+amEIyzBvEryB5IdPMJpwdEQ63SlpJMFLS9XOWsHMSLwYjLoHR',
        'wAJbb+MZL/yMoW2HOHle9Izq1DDA0UZ/dDQIRwSm8estsQwtLBmLhW1IABQbvErg9L5aDSc5wJ4JuC4jJtWc/AnTsIbFAVtGjMHD',
        'TMrK/Al+b0NHR06lJ+8ZtwmntBMt5KSdiugMe5/7Cw9zk9vChAYLBH6uEmXWAyMwThSZL7shmJaODzBZ4X9c3TIGMvKutgNq/rLR',
        'b15sI4EG0TTOU4tdDt5gkqGKeqzw/PSEHLwSjz8ERIIuz9buzaW+cZ7CudtWF1vJYF3RVx90RL8Q8WlHsLxAVCAGRrJLFlvGc+kH',
        'fIQbqEySDyAXFngfhVYFSwIUsAC1mZftTrvyXfRhnBZejbvC25KnQkuYB/EFnuyh2s6gMQcbyeRiYDgTwC54bsHOwHFWgNX5VMWZ',
        '2OOHr12vqg1vBz4infC43RxaeH9aR7Njphvdae5J10/H8hQWkCTOXkMqmYIa+LHKxp46RZxDjIZ/y3zoU8/ZNvS0Hx+8bpPuUkQB',
        '2m65vZl47H/YFrcQt7kIKOViHKggSi66m7dvb/UgBO00GN/ybnkGJpAfRwmhCPwwSqifZgZSTYZhFZPTXDHp6KBGBykc3M0pLDkW',
        'R0Ga3lQaVRdOLT01zwIO1b2HhtL2B0LmFEbUi6ZAaKckgUkUAYmXIPIkI2h8eqiUNz6eDPtuYa0QG+5x/OgpeHBPhdMLkjeiM5Zp',
        'CjokqfBK0OuKh8+eicf7+7tk1iT7eBHNELBK2mNEu/BkeO4sZO4pl0JQtioFbeQqdUTpHlBshw5wttFEYIVhZoM6usG4oduVMZFv',
        'w7dpA79s6sB+7YsRIZ/4AIaD3Tl4TZ4gvBAcZzmFk+J9q7Cv1eeGOG9DBSiIvVmOATHUnQlcfGjf7X1wv8M/WniHRqa87CJWBnIy',
        'W0Qy69/HYOvm+KapXHBn23Uq0otwOk8iMhhEhHZ4Lv3Q55H8c8lOakw7nFDyYt/rOaM3oXgrKUbeb0F2JrP+fSRGWZAvBCve4aNn',
        '+wfdlBVrvRJz4E9KeK93e/M2cQhK2bVrZG9kTrev+puO03BcdRGgSGwokFFYtO13mI4BzAeCFuxk4WZJcGtKBGgTkIkHYkdncolp',
        'oF1B6+O7Hcq7Qk6uOKOiDEz0C8GhiAEigliuynq2xZpM5xIksMh/L7kCT+uy1BFF3E929XndoDB7wSdMocVFvMl0mtgQ7q29gmQ6',
        'OwaS5+5OFM78Y/sUujIZ3utvwmIM+5vvd8SzId6fDLHeVH0y3LxHQhjwkMcHhzaGOzCdNmhxVnCflwaaRJ1kAQtm3DbQESrMooV+',
        '7fhZZeCjIEYoDCdEC+WFlyvnxQUrg6M4K4fj3Q/ch54MfmAjDpAJRCuDsSY/31cbW1jSogdoMo2rVEA4/Eor4BcGKzD0w4UGPZ/M',
        'M/AR4mREmVUZ7zrQs9uDtnPUQ2y6TsfJ3tO2eTRgaKzBDVNykBmlZBJn9nlHnDtHbfpuj9aCU5c7kdOTM5l4pMHgi0vVguIdsUQ0',
        'Pk4kdzJH8IPlM2k2DV/dJAmowD4/GvQ7YnD/7si4FpllIUfYIIne2yOsGBoAaBiSPLA3+mtw1W0nQsEpplR2WvhX7E46XvgnaMLG',
        'ySxaDHn3VvAYw8OsueVuzij74Cd2cavHBZKUEgQWKkH2iSpCIYcwiJzgGPKAXFafzdMCUlvjQKcpf07ls9f7BzIFpfNaNU2cEVob',
        'Dyx2tjTaYp0fkDmmjYLxNFvWEa92Dx52mjqB2c4hVVM/qxbxdgu3xnQj64wvuJgUwKsjsK2c9v+RVdM1DTJmh52les2qcYum7Onb',
        '1z+5/uz6S/z/j+tfXP9GXP/y+vPr3+Lzd9d/uP7q6x9f//r6c/x+Ka5/8fWPqfdfvv4Rev/QEQaOhvKwf+SP34nrf+OXf7/+DJBf',
        'XX/hImu+LbYQzF3TyC8MboL5XHDTl4zyPzF/MeUX/Pzy+lfi+p/Q+OX177nht+L6n3kID/g9Y/sctP03xuHzXxk1A379U7HZ27wv',
        'rn/tInxcWnv7ByqlAuWABv6MB/wSkwP7V1//iGYEGqzvV0D2GS2aKPyp+ObvfiL+6+fij7/52zXqH0Rg5qH7PEriuX3oPnn4Ynf8',
        '6vWjR09/uPdqVX8m0gN4b60plAmnIUftSazaFOKbYiy9owkOC1PUO8ZhpNtG640YiQTRxtnguExqbEzUEXd7kKaFDCaeHAgfTLFZ',
        'MG6De1DEIHLWoiy8qAriGatIBDGldKGoI3KNpOh2gxOP3skb0OruUBEljcI2hUnRiZvKU2XH62dSTPoeVwgMyTHTtS2IiXeGQrmc',
        'N7IaMu13OEK2d9yDl/sf7r1yT+Uip8TfudGOMYINwsAVoRx2pE1WwbvRgvIG6h38X5gmqoWWOeZA3CXupIJLj7qOVdGlzuU0W1wI',
        '+2ESYIAMO8IIcUc8TuREIm3y/GM/g61NZKKoQJ5gCBA735W1esf0IFQZFeAHZTaA1RHeRtrwP0kI8mRBGUyiPsH+AohkannD1iyM',
        'VI9AXbgcz24icfHJ9dX2PMvidNDt1lJ8eISujP0uVRBSlaXdnf2DhxsPn3axbWdq0q5qXYhhqrT4ySMRnaDTO2r7XnvkvAOJbyMq',
        'kWcu9nqeT3KI4zQKyS+Rw+q+Dn3KkuVit0gykaR3X++OC7nZwMvh7kE32XT773df7j3cfb7nBl6T+Cql97Mn+YTofxeyy9RIl0q2',
        'yzdOOxQpMVhpt6PNXu9kPAET29VUBWwx2dszjlKwnug9Eo+wSVAHprlTTq7F3xJdYWm7xIrAgaJF8xRlQkBYsFA1vTDFa6o8DcVq',
        'XYp8MCfYVDTA01XnMDxpkQ9Q2apet1pjB7jzQVErQBRkEfqBuKSOAdK4K/H4QyYAlnMCusWlgS076Rs6pY/u/JSKBmRzazpaViYq',
        'vaIiVLPQXq8GlAMcstx+YiOgh3ylpqDLqxxH+sDCaaBZqTVUqJj9ZwlczDiDTDdlKSt7zrEp0Ym1hNbNw4UfntjN5nXWZmbt0pSC',
        '8FGUD16WNJSl8RtLa+fTwRtm4EOZagoqw9emweArPqTKtXnTQJy1yuPCoiPmjs6EnFJdx2qWST8KizIPF/upxozpC1n6KHyxf1gW',
        'ggaitM50VKDr1KV9FhMFcVDiOCKliMLm4USBuNVqvSc2vuN/TOJx4nuiODyeRmnGJTQjqGNkuLZ5N4UyQ6jnT8mLs4sZw5fTyZ5T',
        'Oh3NFwNjel8dGUSjok5HUxfYOfhIxxHsI1rV0MJu0SHVUtMxe1Mt3GXh7nCu9AFyhJ0RJcI0oHp6momZn6TZNqeoXMAjvALRGkIV',
        'E6UhnriA4As7jdA/yT2YRDGFRntJFKemkwp6c/K0scbAaJ2ypEdHquvYxp0cOyCg12UmyaVsCyToY1JNBL3zTBYzUgM1uFIYxo6B',
        'qPdplh8cWdRqjczhCbOQ7GODlUt4mp0lIm62DJ5YTWu1R1qEPqQnmEotD/mQmWMkHcumlL8eVIpdrp6obKozT1EoNIsOYIZSi8Ew',
        'LffeKLpGP0Qu5XvDHToKG0OaCNypGRJ/ZoaxiDPJWJluGpsF6hyUNuPbEGZ2b4k4c6q2lrZiREew+DZPAnkSoxcTOOO5Ll9TNNPR',
        'nWXKPjazcCI/3Cwq2OSAhuLyqtygE8Ul7BReX3n2pZ2ybFgjjNSbCymp7STNcdWhUUOdcgj4OqhvGIVjFUyUR+HBmMtHqX1y1B/B',
        'HZ8c9UaOU7GLUwBWjiEhKtu9FdeDdbHX4bWS+59Zlxh9tXFJw6+s5fNn2yuhOT2xnNKhL+3WmvM6igVWquyEr36XgAqfGxsUKlh6',
        'BNPIjcS2jl4cvph1epn8DcYXI9YE6wBgqjd4uwwgvxdLq1nMLT2fLpyRPZjGubVkVS0qvEBqkTQcAdYIhcVnooWEsEgv4T4a1fQC',
        'NB9hg0b1aDu11zA5UdLTnr8pr0BgpBV27mMFpTBCyqPH8GEdkYfwKeNYJeN5lCfDnkt3W2Bl5oQz8MPhVnn8QnUn0hs6hZqznekI',
        'wNdcMOSjDUVqD/6sf//qUh//pe3Bg35Pf3XLD5qMPt4vDv6bMl6JywQTleRSFGyv1ZGKa0QZgejJwfKumPAXL1I3bN3v9WgrJuYC',
        'AJfOrFBSLIs9q68f0Pd7VdhnFo+seV5F1+BLw/SAS7y+4bxGV8mhVGcxmkf4MFwacYx7f/DgAwpMn1/aE+7SVJverRrR2BUNenI5',
        'Bx/dzdlV42CO9oLJLXYD3N4E680SzBAx58x2ZgmyS8Xybi+JhY6VSXDzTOk+8rSXTagr3dOdszY0+5jmt2Q/N2ZFQtiInlUiXr94',
        'evhqfLD3cvxk//VLkhaODjfLe3cUG1EzT+4sxYd7KRI0mSHUDJRM8wRBfzZPovx4ThfWzukYPgypkTeEjuXz1CQNWwKikFOAgfAl',
        'nUYJxYUx3QnIQ1fs8oF4aEZSLYLpwmDEkFDLRLpL11vqmmRUVDtkDrdWvQpFTmMkEsUpKX2bY9JAno+1tLHuNvyPUdwU4dBY8mFD',
        'dTRFd2OKoVRdNCpRtWkpYwQez7RQs8yo/UqI0VRa7eaaB6RmBWTSoZ2+Z4IW4z4opSS7RjMt+w6qtdjVupnT9pokiTpr6G9Ct+SL',
        'Ct/1JgJoxmqOwYocm8yWUooMqeY7EGeqMmBIOn5DOsjUGNNiRjYBqHIByVR1XhbbzRJZ2/AHZc+G2Or1msugzf12M9FVL0jDd+PH',
        'Kxtfc+aFtV/y6JQcr/h0QBgPYLx55RHe3X831nqHrkRubOggUrvytAhT8Wn897oh38L9rxse+HTpzlrD2rWi5YyqmgXryxtQk0Bs',
        'yKyMeFg8NIJCVpZH883B8i4m37vUdy3rIqhLTHfX3UWcWc8f/pDt9yuYQwkM0K2cDWrhGq8GOtEl5HFE9/T4rvCp8rb1VWEq09Tq',
        'AjYMhL6lWKJ2KJkvRHapGMJ2jBhwbgxFefWNzdiRziX80FPnduoMRqPG6AmIPmmudankse6+JJFdLq+8NQnNhxelq5PbVT1C1yLg',
        'd04bFePWW21Cwd+Pwpkf+ikYOxCXCxXaNMK5ItfFn7w+fJNwGO+/zZKYZpLSELK3eR0HnZXowh+xzrlibWEumpt9zbCz8hb/PxUS',
        'qu59WlwCZnlJ2aPq9ro3LcsO7JQL71j0v8M1HzZTrSWTT4Bk4A5e7r3ce+wGnmXwNdr0MIpOL8MBEvB0Ljfv3edb4gZBqDO9kA+v',
        'aiMpjahuZl8ZVpMacrQ5S6JPVYgUFtunj4jMMoflcs21L8B8f+/lq6f7L6DvPP1w3hG3b2tuFNVOe4l2Dbk2l4sW3nJycsPYlRTF',
        'TAYMsOoaaiS+h7B6nc2wvvnZP/zxq78RJsbzFIloySDzpwwesr1wqhiGyz5GAEju2WIwbTPMqJJtMYmyOXExSjwdp51QMdPe3fv+',
        '04eH4NArIHaWDEexullzee4l0VjztHSbuV6PZQZ5eRCnNmbvCDItYTbs14upjVOb2sofvdz/i70Xws7DxirpOiixTm8+i4i1TG6R',
        'BS68Vp36lb1ZTyl4s0Jpgya6b6qmFQlVHhdC4kiI5y4QBw2JKWyUEJenR4P+Jowh9GH59rcqrn9XKk06Uo+AOXyiC1hpdY+YbNia',
        'Wn39onCJJF13q48D1M5NBXgdCqfZu9b0S2VmMms8IAeqJyOtr6tDTPeiCPlKwWQpwNQr6BKr3xRDqgUjvakKU5mwCmFjp9npkhfi',
        'WvyAoqVOcQgULi2QbUdtXbWI2yn/hCM64wim2LTGMQpfUnt6wHek6ZQI+dhFcVD+NJDHitbKrY1SZcx/E0FIVwL7xkbGN1WjqvUu',
        'w9O6nBVYQ4TNNNlkYQluyNFZUTF86+GIYbDth/xnHcVy87C8FedYJFVTp/Une/7lOnc2AAA=',
    ],
    'PREREG.md': [
        'H4sIAAAAAAAC/3VZ73PbNhL9rr8Ck04nkioq8s8mctsZx3bczMVpJnHiu7m5s0ARkhBTBEtQlpW//t7uAiSd+L54TBJYLHbfvn2A',
        'flJX2q9qV3v1wZ1NVVmZpDJL6+tK19YVqr+o3DdTqHSn6pVRhatN6tydSs3CVUbpYqfWLjO5wgRb+EGv98VUnmbujdT+ZP842Zsk',
        '+y/H6roz+7lXbypjvhk1N3mufA1Tnu37ld4/OlZugSfr1cLmtEYmLzAcS+EfXauluzdVIZN0ofOdx/D+bF3e8tM3My53s5GiF+Ze',
        '5/zUoyf2s/3IzrePtbtrHzJd646Zh9JVNT0P4u5p8YWtfC27t8USezHlWJ2tdLHElvSiNlV0WNmidmp2fvHl7en127/efxqvs5na',
        '2nrFhiqjvSt6tFt6xOLmRFaQDFSbnCxiWV9bRK0y5I/Jxr3eTz+pvbG6rHRmfK93gcjsVLFZp1jbFsG43+RI8lxXlYUZVxi1pPFj',
        'NRxeYeVNZbLhcIoIr8sNrMpExLTaFOTBOjqlOR+ciEKvMXBlKrZy4Wud5tavxFC5CQ8EHYfJleeZG8/GveVMml5FkVJbsqJsrbba',
        'K9NaCobtGtEgq+YBkS5djsdsBLS6r2bO/7pK6c6aC7vEjmQxt6l5NYv982pk9MwVNBWDyKxW81zbNcFsbTQnkpNWuBhH9q+GZ37c',
        'O1W5m+tcZdUuofDEUHsk9+O/1MfP72e81cIgEyo3+j6gu4N/ty3IMcRa8rc/VucUXM5yrzdUF0DtRmrQm1oSn25sXgfE9T3cp6/P',
        'CN/PyPUZpecWwYOTX4Gl2eBEGQZDA8/MzTdrU9SyPVQbFZ5WewfJ1lUZVvp7Y4q5EVRScecOs0zrS40MSLQL9ZIn9dT302AXyGSM',
        '9T+fc9QwpKhpiB+pKyDRvn5nrz5031J8kM/U8BLhxefzPz8ORhTZrHJlSWAfqi8aEK53tGP9xM5+D6ZMZiVAkTuwWT2netxLDhOA',
        'fw0A23ujXusdggAAeG8XluqVbBJwiuCEvFG+zAlD/QuscVqtr88/qF/U60+fr0fqBq9usE9wUvg0Qlwu8fbs9It+NxD+825TzU3I',
        'fq5T0CZVmJkjjFgvlr5eghx5L/w1lLn6R0HT5q5YuE2RTcmvuCTGwT/ZAiHTl+BVRGVEgy50GAQOId6iom0HeMdlqLoBWOsdUlMi',
        'XrXpDAV8VWZ1DuCdIPThX/UAONQrh4iWq52aV877hArYdN0fSUW4UGg6Bfi77q/BRcIzrR8JhyiHu8I7wCI5mpqdg61nnNTWaew7',
        '90QhweozwkpcoYNgXuQNEnxj0mRf3RlTEh/m0uAijKRgfzAf7JHpiwegSWUbgGKOOAEXBWKrQUBwmDA84JqtzBqtCsnlwGATeYCB',
        'P+n0j7krd5Qg+aQID0wod6YMBHEACEWsNyQBqIF00h2Wpy6VloabVWjpM9XPzAJTaN+MkE6LQ1mRL8QC9hu8P74aqb0j/DmcXKk+',
        'EGkGnLKXR/yBXi42eT4gGBSJAeNlGflCKFkbBMVzUJhudLUmw0QAZFwtTYinx9BIl1SWcUPwyRSo+dk1KmZti/7+RL3HsuMj82ow',
        'G7VzU13PV2KUGlSY1nwG11ZkMBHkztEKECrV38KjZFOqI/Uz1eXceeQf09XeRP08GDWVx0a8Qbr6h5MjVO1fRXguCQIoPeorQRml',
        'mwwbA8fygHuwkiYORNaoUNahqY7Q2yz5HOj6CEiYo4AREdQi6ypgioJAiqbFH0Xzyj5Qh+LFI8dOG0wz2RJl6tSCmHZqVt7ehPhN',
        'xkfIpLq5lRCpF+qa4giJkOpK9RnuhIfkckYd3g/EGixcNhb22cTldyYim3BL3wLs5GiwiyhPBYWUfqCJAwtAjcR8n9YTWNEzUUCY',
        '2SIBhRO6FuQH9Z7cG4W9Q8NxHRyOVdArXAHDYVqmwyGHyDxqmaIGo/gbTNl0wptBZ15iHLg/d8skt3cmtyvnWPSkRPGcis/Xb5KX',
        'iovrRM1+M87/MaPsAgbSBaW/owfpCkCgOGDv6e2FlGJ6exP/QZB/V5tia+xyRdqKNEZYhPvwvXQ0NAZq9SfY1MroLCeMisUbMtDn',
        'f9Fy2PYACdnHxvsxIRTVBhpxJQ5cvpMSQfswFRbJzNySRB+MOYDnZn7Hug9eQDXAJmQTaOXiFvncmvSWAAouKcw21Frm1pr7jfPU',
        '22snLbdRX6AlU6AaouxEjTT76PQEMHkQZgz2VhagQlBl8x02zRKFuKLUIMqgE2k1auBQhdLugoxoiWaF/VN0XY3PlORunfSFc/dH',
        'k8kk2CX45NCGG700HJZrwglopoqF7KeBbngsahpRKTVpHhJOgNwPSocHhtSq/itwj4L4q+loVaqzt5L/ZvRgxGqnjd0LdVEsKaCy',
        'rmLaGdEBY52kpAB0tQOXQT0wjyP70DySWCgIOozVUb7kBqES+aL6mj0mrMGSipYoLzQ6N8USlYqAUx1w9Nnwwj5IDdNrtjdi/rLF',
        'ImcJLmUBzULW/UCK9Wis/tyVRBWEKPIxYq/pYcM/98IGa/1ADcvPK1tScQ7GgOJHxkojBp8KDweGnaFRjHxU8ZLUg1imNPqxegZb',
        'lcs286DG4crdM56n23HKLniflpj656gdaehz35NTxYkIYyr6Si2Ryy6kcXyx4G7a1760FN5OoreE43aZlclLTxt8g/WNjk3NS5BT',
        'hIx5WVH7ppqaqnOTk7pHTUgpgQ7QopP4FJv+YBRVOCMOOGPBoxnoVHMNBBl/LWf2WunTl9IA6PW6zIn+xEOQanyXxSgBDyZfnKgL',
        'ISA5rDT8o1l2iItopx1pgqCBu01NgpxECEJhJdhwOaczKjQiGHoywjhgyhsZAUXotnjb48YwHEJSZnaxAC2gioZDOtsh8h9OP32S',
        '8Y+XbBdksFMOvSggYeIYbunNlAVNEou2JsdBHsruhJXenL59JytZ1uv87akFxr2/YsvETFugXub5xgPo5PV1VB/MIV1EkSdB4ql7',
        'H9pr3zyUXHZTaVDtssz48Rg/ECQeiNLMnaseQY50moClJumEhQ/bCLAsRJFjW0Te/fcDOvao9/9NdF6uNPrQBfyQaR6L0kkSuQbZ',
        'Ur5PhFk8ZvxBOZyo35TM+w3ES4+/AzL4fwDu7H3fa/jKp2iaxlidcXsKqOuUpJodjl+ZI/JqMv4ViP9FTcaHB69mU3B4uGX4FdqV',
        'pN5r6J5NRYKW7bOmLZ0F4CEOhkPkw1tPFCbppLFYn3MqUCex+JgRFFVPyDb+vD9pMA0bLzrAfaSWBMJwa9o1hZZ3z+oR5BavPOh7',
        'Fa6BMnNvRdw8cawSrSTiBKji0yQfseICwgLSaW8ohguzFflRIOlyBBDVPwpB4TMRqX2CgODosJFr0D9CYeFNhFVH80XGugyMddkl',
        'LH5ILmmxJ2kprNNRdK0w5uL+P2RBqaCy7cmOyBXBUMwbK165sQxCXSonpXtMlhYIyGQ82ZMdH/GkKKuI7H/oUKzGIg9zzwdxS15W',
        'SDPKhw6AgdiFCMjycVAw1SOLDXlTHbD5p2RRV60wP8gtT1O6cOCETl8J4R8RsIVb47Au3BK1AY4ITdOMPEW3WQhUJAdp4sfCTv88',
        'OFdyB0p3T3k4zsitJNSBIRhu430CZtcxwDiFVdUuKe38zmQDEu1XFH8/fbR043zkOkkSCIbKKcmxRTqtUn1iXbqyky+onHrrumKd',
        'j90ljnhy7JXE24rk76A56f24bNsneGVIq7C2GBTxNw22bAF6Vc0pWIZIdLBHKrJ3sdM0Hge9H4/8++iwW1tkbss+Hu3td5zman2K',
        'FR/Pl2E3t1t7Z29RjxlKHqQcZN9TJ4MgUWuP/UFM0w2Xo8ZBd/kpidHBqIOoXO9oDwRlHEhG6vTfdqS+/oeOiDjCtJuDMSbAdgOB',
        'ezHIEWq5O9M2JaTKKoof8PPEDObKp6d9Jfna5L1Znuidj21bOvnN6XaIkACvAEHzIISIjXEGRPBrKkuE4viw44JGj6lHTVUcHKuI',
        'gP6mLEMkwN18HeSp9HIYFnomENK1dNXMiXBortLyHSTQR7dNOhdFc1cyfQX1jQWIyOkAxqcQMryqjOH4kx6nLTIcOW1NN+TzDmW2',
        'rjZ8q61me+AkzlIFFkKcKrpieqHyAvuaxbvN6rE3a12OmtDQ9Y0wDUCATZKfacQE31DxjU9TA9N4MiYB0uelcyAwkQl0LW3aGmiS',
        '2ObZD9ofGh6RwOjHcmsHfle2J+ogOVcfzk4jj53E2x5k8wF/Ylg7d82hePW8DpeAIQ3wrv+hIpdTr/am+5x+wtk7OUB9qHhndE6n',
        'Iy7FapnvytVU4cBY4jCNOMQbhVi7nWw+fwxgo6vcRhNeqPfXsbqhutq6TU7ZuTNIKfQupTkc78IRm28DkU5orqRGM1ELbXPCgS34',
        'uOJE0LTnIWEJEjiZofZQ6NrRAZtXQmS2lSuWEEa+diXF7BRGqD2E/kSXgHy1SHco3ocfYPBG5+1lTs43ntyZabHjK7LwPFKy6F2+',
        'MxI5M0KT546xsOE2RUCQGs0XZM3NEtTFLz9oXYq6J1Gs4sb4R6rwUx00l4WAIpHEts8AVc27Zm5ue3/31NcciOinBuyf5AifgvgG',
        'Xjop30PGDcvvfRlRUGsd7bKUNggZFn6Pa36ypB9A0S/+B/TQpXeIHQAA',
    ],
    'udhr/udhr_hye.xml': [
        'H4sIAAAAAAAC/81d23IbR3q+tp9iSje6Ga9E6GBpi/JWbvIEyQNoV0zMikS5aG4qe0fJOqy3sruulG99ARIgIQ5JgNRgeARQlWfg',
        '8AHyJOn/1OdpADw4KRdNcNDTh7///v5za/F3//HqZfbvS6vfL79eeXZn4Tf372RLK394/WJ55V+f3fnnf/rHr57c+d03X365+McX',
        '365my9+/fvzg6VcPnt399k9LdzP16m9fPlcN1Z93s39b+hM/X3l29x9WXy2tLD9fuZu9WF59dvfl2updeHvh0dPWQ/x2BV9f+f7Z',
        'nW/X1r777b17S/+y9HLpNytLa/dgqDvffJlli2vLay+XvqnfXRzUP12U9Z/r/8ouztTHg/pv8Ef9of65/nAxVE/OsnpdPX53cawe',
        'lPj5Azb5SX1R/2f948XBxUT9fIb3F+9RxzDEyus1/KA+fvd89TkMdsb9/1T/Tb0xztT/KvUiDYO9VDAl9Uk9qv8Cw1yc1O9pRov3',
        'sBu7x/W6XY/q9iX87vNPldWTulsfqh/4c1KP6+PLt/WnekhfbqhfO3VRV5H+3mFfHXghqwvslt6uLt/VVd1VT3qR1wbY8lRPZKz+',
        'v5+p3gruYF0eteutehP7KTKc3lZd4ivw6NSd6AyTUZ0M1OOy7qkHHfxcQZPLdfjivD5hmmyqtvBmGU5+4enDJ5kadCzkg047SDwg',
        'Y6noRSMt3P8KqHbdZSk28qd6oVqo/TpWvanuaSm4TvX6m4zWD112iMDqB5/fJH3w/V6e4aKKDN/ZwFmdOWyleh6p3xW2uPxBNf9B',
        'aLCdAZlVA5jCUL3Dg59hnx3VQH3sykZc/pgp/gb2LM0AVX2EM4QHQjNcUGboAZOC0Yf1iGZ7rF7CLgPi1bu41jcwj64a6dimdC/D',
        'BUKT9QxH7cCaMrVd2NYeXqgJI6iZ1SeZ2QWf99WogAsu7xU28XrZ5UcaEdsCqbvQs+pe9awI2sZmbTX+ofopsv8u4E/sfx/JVeXh',
        'uCWREJmhRPYFOm7gOPwdsYJ6Qb2+qVa+p9m4jaseAeOf4g694SUjYXm+Q5gtPBgJvc3ODfSk46TDF3Lo5RxeAMLB4776qexzU2Zm',
        'qRYXq49tRYw2/uYh+9huE9dQ1J/xi0Jep0Or6XL5Y3jwL0b8ahcY4C/4AtOgq7AYpwwsNIQOI69/vji/GAmeZ/XPIEnqvyNUv4XP',
        'kVcODcOVfHiLup+pXa346Kr1d4WeYwF3DQHV5XteH0ysaDr1ir1xywaqd2eXAEQad6gex6iURrtexhw40W28c6tQgfiiqg+E49Ur',
        'am6ncgQUW+zhOvaYu7rMaNTySL1+ao7LkPCLeUGdGHlCR2qACywMgVOiTIGQ2ilEkuQyCCz2ZIe2Z5MBCmmR+J2gu+LyDUOt6njM',
        'rOofIRoI1r+HcxtAvzsav4gKHRfox/h2z1t9wAcNwiLPeJ8KwrcdBuPTmBpBLImtANgKQO++BnQlPRq46eZl+nUkst5e2v+tTCgG',
        'e9Hn9eNRO7j8iAeLZ9lHQGxfvqHDyHs1bKauCNdjeOYdcIO/ROq+rFnjQYLrra2OwhwzeMOsmoc8UL+2opv4V+wKl3FA6BnvvMKe',
        'opMa8/n4BCQEBsrwmKBooFOJeod8B5TtNDIMykMPKuwemdyVYBKuDBGSpw/aMkjKIcuMQsu/wp794j1R6Re/W116/ur3L0W/Z2uC',
        'DAal2JNJACq7Ngc09X7GqWzZGEXMVIgCQ3BZ0KmKKSm2PkIfBxo2+umzKwxP8uEaQqYXYJMvXUhPoRW2LQw08LFHUh0Qzf4O8QoG',
        'Qv75n/VOhAcjVExjiodOpFLAHwPhEgfVtK5ErKO+7QkXAazoY49Hxeyjs449tY5DpZn04OM5ojZi0warKEgUoL7g8iYixD5hfEwB',
        'zWHfGAKKTLhCoY7a50L1VASbMmJjcJ91NoKht4EKQkbbEPe8S5NXUyMW1JBYqmeAowFxo4oF8tgWNtmHAXZRw8XOxyjtC23dkC6H',
        'HE/qusD8WIPrMeI2gqR9bFCVJFIe4EZusv2B53xTZOlUJkLdATtQ8gyARA2aW6O6rNbLvQkVIsfP6FSqJ2j4vEGpeoJULmmcU5by',
        'aGDZ7ELyDkzPEpZ0zoDlHpnPaCSxEXD5Htmwom0qaOed14DTY1rULNrRlWiWIS+2me92PUGMZAG22GCA6fMhNGYE9rehjYbgCABL',
        'seKvqIuEmGGq0zSEJr2q16zKtWmfu/WZq8ON7dV3RSHoEqf+HytpOdhHhxraeGg+grDIs0AqzA6oH409btbepiNMm98Vuntiglh0',
        'yLtpW54+N43oTKFSZpk15+hngANECkWhF9+nEeiEhKMUhhV3ca6zMtRMDoOIeTS2aDONKdFroc12LR1IL/LOOGmss59uA+tz2IiR',
        'XU/p6mKfV5YY0VqeawYi2WcgekKVLSxPWWIBjfPtISIL6pZ89C/XQe/3JctUR5WZbJNhicQGkaiPpyxM9tn32kTJ47kh1Ie/orP6',
        'UD0ekgoaUNT1Jdq2pppxzGi7YSf5xXFkEEucynlu2xq8pQc3GsEG8swxQiXUE+Oz+BC0hmz8687WTbXISh67rCfsFC3hjyPkBV6p',
        'OCrJykTzktlmPeqdJY2msNxwIgMF7lTDT76IElWEzVdPPF//SMUwgY/+Lil0Ih9JrhsFsmAtkfZLG0HsxCyQB5xvLNZA4NKwZVYJ',
        'TcQ/Kd4wlo8DZBpct4VDjYhAICrLJUk2ydDVCH+eGLezdlYJnmkI+VR3Z/Uup7w7QCTsypq37FmbOCHuOU15jPr4VQlb5Jq5xrqF',
        'P5+vri3/4eVStvLHV79fWn12Z+GOa/UCiTsswBYiFu8cbk9mNMvk8DQRVqPSig/yYEL2oaszqz8YcNHnMGOPedtyiLjgrs063JMS',
        'uXBoRFoXTjuiqetU2KA1uAdh7PtHFu8xseOUbyUo34pR/kBiC8dmlUUcY4wkkLmxrar3rtHJMxUjeo7j/xhl6jYQcuIKxPoXo5DQ',
        '+OAAQoV1qNEUbBn09Gt5jHotuI3P8RMYsFvg2c8JD4+pRUMMAVbMsY2EiZsHGMWvRhXSfQRtidSY0NWYDyd01yHGlu5sLxwHrgyE',
        '81iacDzfhgjIGKVS3BXIyjjNC+JYaJTSRjTw/InD1WTEF/4+6WhOPI4TJX9uQ5g84Zej4J9ab25NTZGWMBc624PwAp5owh4yCHZp',
        '/fAGbjJN4Jwmg9yPYUaejbSKWdhVuC8jFARkSbQxntdngYCyiESAw4PknDYKYimRWJ81jLKmXRFz4MeDBH48iPkq3zcrOT6inLIR',
        'pR7lDUaeOBjZeMUPXdS4S0VWYkznBQ9x5ljpw8RKH8ZW+mc4DLimBn4f4RnpRw8mSQwWXYY3xfPkygXiYVjtmLBZesbAVdRPOtJA',
        '2bd9bqTn7JMvzYFqCFZwhGtmkj1KkOzRPCTrshm+jv47BIQJ+bQdPO2g34idxGQC+s5wjbETMsHI7EW8PUkolfpFOjk7vhc/SYbH',
        'CTI8niZjjXN1m2KY65qD8oCdreNjVMqxRD+1NeSpeWq5Q8pOmWNRXycW9XVSZRO3t6WHwWJdzyOwIrj1Lj/mKUnvCxhHPLqdIlB6',
        '47puzLGgTQQmMnf+lnaX7DKFPaCa2A5aNDRQwuCn/WlrZDtkDw/rxKi922mLD3RMxy+skzkahwPRcIjiK5zjyLap5zoWTxIc9GQu',
        '1dMR6egAUY/wpHTwuJLWYFlFngFbJL2SDTbbWOeDUARHk9HrKAf2RkMr2CnGaku0e+5gNDsIeXxOtl9jhamCgWd2+84D5E8TW/V0',
        'GpAXomah/uNCeZeVTlzI0NVgdoWncjGhJvxeVGCSSwwTooIEnSnLW7ifsj/vT+fFNAOJKc9Ou4hKbNra2+p76/DoFsTWUcBQQAU8',
        'YnkwyNKxzie9eyruo1zOS18Oip9HEIZR6aOoyE0SKM+0iWeikD07eItedbHLvdQzikAeE8vDHEck97UKbql+4nMQy33gH/uUXyLN',
        'F0m/hOuYUI1er75YWl168XL5+zV5Bo/h7+W1pVfWMzfdtsLt05Fal+Cy985msi0xPYUjpSEQi3WhIxOapiNp4sHALecaWrA5Ba5P',
        'JPRoVCi2rz2gqhD12ugdHAlUxfYb0Vt27h0eCgcJTSrpDgMGYpzriWMFmOebEMYWZb18CC/+SCh/hMHlCHZ+oZjI7PAX9nZ/MQse',
        '8oI1PoTa2sBnEdGCIKeEtf82pskaNXUDt+UTc32Qqnls4pATa9t1xpGJ/TukzokaGzovKe7AiFnaMUUszvfaR1AEfAkuth89+rly',
        'EFcBJvJHJ064j6Oti/pe1DvsPah4KTZN6MScWwqVoGLioBpSMrkdwjWxC0PBPQ85psFSymm30LoNeSxbWvoGeCGn5sDWOBCbTd5R',
        '14h1K3Tq+fhz7mzCCi6lKvuRSXLuixPFbOnYyyrZIJeOiMrYmo4h6xzjH1b3lCkW0zN9Dp7VdmjSthtJKq65+PRmF14p18yC55uJ',
        'iC4byNIK+IQl0xE70Q0tKC+jCkNdY/JiM5OQn0d9H3X7xnTYpOxzAyJtX8mu7JIB26WZxc7mrFQILSPtv8TUXM43/GRgBbnXakTR',
        '+XOdISoENO3ooxgeHUQp0zbl24otbm7YSXnAFh5en6FSqgp6Qd38fibjgH2fp7pOoyRqtnVYUlyq6LdH/eaj9zW+cfkeY6Esaq7E',
        'GdHIZy9EXZN40THS1xO9x/XJV7EIwxTRGVQ/GP3Ay1hom5wKW4GLUDTSuW9GeInIfe1cKmdMTakYydvGI2l5K/uq6UaYpXvjPJ5y',
        'WS48ujnQtDcWvY+zeqlnZcaEyA9koTG0tmw9ANCmcZ667EA7KNIte1CGdMxO+Wgk1bPNbmQ7U67XhcdX2E6lf1L8BfM4/Nw3jEFL',
        '+ilC8xAdQBEHZhCGjOvSEm3Upz8WCSq92JcHoBwHLznFoxIJo12BOjvcOI9Y2/7g5rtPS+KbNiJAW+5+E+SrYrqdTm8hpLY72uUQ',
        'hfNig7Y9p1T356RLOazDs51JoQYTERU5bQ9ZM5XCQvE9MM55CVq2ymMS8w60B8RapAO5V1rksVMK0EtmI20ECn3gWNzA3OQzKsqC',
        'nZpdJ9aiPTED9pt1QwU56sy5EcBIhTUWvr5BpXmktv+HRFKIncQVQ8tfuLrTJIx3tZEuigR+wWGjtygwj2wdAN3YxL2jWYTpLYqZ',
        'FDluZZ9TwYeFJ9cKXFN2XoFJAlCrofNUGUArYwLPYupgtLfdqFlWbA2cO6H8QoIf9oA9LVWa01JCMd1ciBEdxAivVPIL6bEQ4P04',
        'ZaTcYm0rbaWJnQVA0SbNpzqWub9NVF0mmI/xC2oyyMRbuIskW7oYY3djcIFwxmXsowJvsp8lKbapAkcXAs5j3KciMgtP54ieNW5S',
        'DIVQGnMYY4C0nCRQ/kqcO4VpSO3hdAe7+AGrfcj76M7X3FwRoT5v2Sarrj9gN/obbTByCooYixqU3RzJ6Zy8iw39morAdWAHKHV+',
        'Ye5EP7yMn9C/MV+ko5WKgLXuX0VRnhkvdcme3s4geFYG5gTvkWQ6da/oA5lTpA3YUSohNU7BNFWU5K40PnzKVShd++6mhVkrFaZq',
        'LdzG5jXrcyVXjXCprRbwXjafyYvj8hhQaCgiRW5RJq1Oh2IFWfyFEvJ2K6dNLk9DrP9K/DADMcz6GMk0PbyDus9rH/oqnwkgaF5O',
        'W1xXMgD0ZSVkrjJ9CY+jOWruXkX1FE62iBfc9mwhIIMVoRSJVM/7s+ly5qJzF4ebBYhlqYZJoonDrq/KHsatlRmb9J5oKn9si2Tg',
        'xtSLIhpDt9QPUD5g5OZEBxMT25FKdu9s6NFN9VpbZ7pe5VTMDUjJrPLW/Np1HgREG0xGjcR5Y2GfiRWnAww6j0G7hT31elqppFaR',
        'rfxU3qmg3tRxyUaC0BM03LtGr2xaH4vLI4s02qXUkKBCvnLH+XTGyrwvg4P86WmFhH6xUD5v4KYvSal2rrte6DmShPobubYFzUtc',
        'yiNKsp5HI0qFz1oPblUj4oJ7neFKtQLes8jBb7omwc60qaywx1iSHo6sjp36Ab+0FXja7PWWA1KWR3j2vMobF8cxbysnSkxPGS0C',
        'RI+QncvE7NrrkZTi4KUB0LxJWN7ewm3+iedYxc/EPNO2807drJv41SNTIJLuKuI7IhgshU38e0t6yHi+3oTiHMztfgQ2E95TNsUQ',
        'VIqkqBj7AmYoPl/PPrvdLUWKHCHBrUt2nCsYnF0mpwO5Reiw90UNEFCOGlTmcg87hm+VCPq2TvdXij63UtHn1sNreezkEpaRroS1',
        'Igp84YYfvyd7xQUHzffGZVSAEUlxGkbbxgCOeNBD/XYM2fzEZXK8D0TAu/WXVy9vaaUin61Htyrv+uhkGpmLOwi/qBYrDkE9Dk9+',
        'jIJF4elTBqmkygWvoYyVYpprRLqWJiMOnBgXSMRInYucxt3kINAkt8IodIT0KDtYc2g5cE61Vtu37jYMNMHG6sCIVcmH2mQ7DPjG',
        'oM0pON8s5PmmSDWBMUOB/01bLpNhmPCUEixeLbyn+87lLG1dwj20gdu+9JHTEF2J4Kmbxr0KaTWqA3MFTKlzMmwlx8tO4OJgE+Pe',
        'MwkShBeCEO+uKQXeMVns4tdSKyl9pyzWLjvR13PBvm6iCzIIoCnrZUfvRn+KUpLZhdV0e+UeX362Ti6afVNu0G6MmFpZAdrQd4O4',
        'Vpr/qVghoaVFqbzmjowo78dzfG8tctRKpRS0Ht8uToZXMUS2cOK0MmoSedrHbDZU7LgO/CiUXoRPsSP5Fq/mfQul0NY5jPqMSOdH',
        'kQU5BH/3h+nHJtjQlymb6MBaqb8ult8WNn7qM0P8tVkXTkFsfETxK7EGs52Ffh5PlFBo6YTyio3q2XBbWHRYOEU64zxMxopd4eiP',
        '6hVK2zcUt+NxjoYi7hm9hpPpG2Znb7WxyaH6jyfocMZBJAkicv8W34Xwq9465F4uBL7KdZ0Gx9OS+12r2PJZf3TqCXSGW+6l/DSn',
        '+OxZV4Tr+804LGuuFOIJ5SKOOsywdDtHxMqNqDiefGaqb1Myos6edxY3LY3PXMDYtq8vyZxkRTh0zrhXYssx3i92GBWP1s0evhxD',
        '2k+sivwQVY2rrYh6V8ytMVWje+3GBU8qNaX19a0KnkLAifI5+IA0uBmtJH/3HK7rqy3dCJEU+HMmLF1ozomxFMmXJh6sN9zSFh3A',
        '2LFyAArHBWon//5avqp5qoX93ZC7PoeeDKy0QqyfG7dBEZYBcRzEKXKFB11d3cEyboDgVWig2ve2I8fL+oOLUDYJ86y91QpFkDEf',
        'ONtu/Ayl0n5a10v7SbnjEzePkIYv90E22rtF6MACKwqvhqj80BXjdlccVs0X8xyxotFxzZ4r3skzj7shlcDSenobaDZDBW6QFwPt',
        'hk3lStS53NnE1uqAE1S0v2e7MQqEBE1oQmypXkk2nl3veiXNUpzHTOa0TuANYzguoYObO6AfSw+kO66stFmn8KH5shtz1YeBrhkK',
        '4zntKcgZu4FbH9umBA1g9nNwl5uxqox2GVweZxVLeageca9E/nWeRjDRhXhe8Dru5dL5ak7w4JTD9BWqO+x93nESCSQ/wPvHO/Qj',
        'PTkqTZXWfrJDSSbjzVXezLWhPsNrR5G51cqDYPeSGlP2Imqy6mRIOeL//ypdHqQyvh403HmQEiWGShvEA37ylP9PyISVzjphKpI2',
        'QaG70roio/TCe0H0OI/c6022lVUcxQV3Bya98EDabuFED22dsOQ70mmazSaNdXGHVX9FAXBL64rayYhWRORp0hpANcnTUwxnzJM6',
        'sc3uBhG+iP/O2Tdf/i/bThlAgm0AAA==',
    ],
    'udhr/udhr_eng.xml': [
        'H4sIAAAAAAAC/81b3ZLbRna+91OgdKMb7kgzkmVrV9KWK/EmWxW7XIm3ct0EmkRrGmimAQzFXOUR8ox5knzfOd0gwCHGojPcSpVL',
        'HgLoxunz+50ffPjzl8YXDzZ2LrQfX9zevH5R2LYMlWu3H1/87de//OH7F3/+9M03H4aqjoXrwrs37//w5uNL225fFlj6R2/wIH6+',
        'LO7tIV1vP778sd1619Uvi8rFjy99H19y8e237+/efnz5L6ZvZXXbfXxR9/3uj69e2Y319qa1/Su+6cWnb4qi+NC73ttPf2sdCTS+',
        '+EdbehNND1qLsCn+eWhMW/yr29Z99+GVPizrdtGaZp1+HTf6JV2ePiuPY89P/15b3O6KaMuwbV1+R1/bwrW81/Y4DO8cCtNW+Z79',
        'jwGE8YJrjXe2NXhBEYUmPmO8LxrbrHGAvKQWsjemcf4Arsi1TRjaajzYJlpbhWZVfB663pVW9t9ZU5IWeX4foq9WH14J6WfPUbku',
        '2q2JlSwuQ9vbZtfjRTERkGiszQPotd3ge8tDFGsT8V8YusKUuL+vXVnrU2Hoo9niKVKAHbsS5wVNoBgb3ru2WsnLeNtUD+QYOaDE',
        'cuu0l7x+baFiXdHV5JBtP4dDPjYXdTtr8Sh3W1uwdSN/5gc2kf9YE+Xq3uBFNY68trYtdjFASVwDMo3ytsZBbdfj584dlUfP0DT4',
        'tbNh5+2T3HQ9JWW7Dodyxq8KJ2fmxTb0RR/wcm63s96TQSHztQxD7OyKtJjCG5ABXofYr/hMtDicJ0Fma1yLm/0hmrZNGraDHnc0',
        'TDxcm34ut64Ogyd3eOLelpTe+iDHioMXoXizv+RQpAh7NdhNtqnsg/Vh1yQ5biKkXUFlo/XCRTK835Pnrf5+8mXcURk9GgIMm1T/',
        'nHYTjiX9/ofaxN5GvMtsNi42qnQuwmxcX/OpDQ2GtIHwKWNWeYupsUIDsWpmfztYJBkvljsx5aPpYvO0OP8lFFYWhDWuVTFnhnWh',
        'JAvxc0uhJc3teYauxw8Yomzq3UYOCT+2xb1s6U9x7idxH8W/9aa3iUvgYpUMsemsf8B10GLK2kFmwoEy/M9//XfAIVXh9+TaY56v',
        'VCxyhmQXw+htcY4d9EpchujjurPxwSSLnymjGOdEIOlYT2uEyfaHhXgjuQSXkKTU2dneaT/qa5Ii2Ex+QIWbHaQrZJFS8aaDJ/nw',
        'x/85sXesVb6dpernsBduRItdzj/yK7b+J9uCp774AVaDWHI499wvyQV1+tKvil/FlCFZY8R9qlDFCkUSOFq2I7KGv9tRmkE1WZww',
        'PAYWRsQYMPbBVTlQ6cUQt0aooOba/rBCALc7CkCInpJKVw+KeoartmjEz6vf7vqIs4nfsSQUq+UN1VDq0omFTNVpWcBiN4fRjrh7',
        'A2UZ8GOVzjnGWxhXviJH72yJ55KjOOqxULTZ4N0uueQc3+davSrWAVZiIIHtqbua2+DE7EQE51aAuOj6AKfZqYInuj4P0XWVK0nA',
        'zUR78OcUtXyA/3Ml/Hg78M2AZi9O0MwP6YHbs3DmB+/ngdaAM+sQW2G18kT8HSQ69ZQqk5sCun6QNdClsIfHEA9Cu018m8R//kzh',
        'CJABktirt2t5K9Cm+BZTMP46iSXrKJfrEKo5D9Kpz7PgbokFd2dZ8CP1nEQwyLXygLhsKq6EyTPq11lRUA0xp4awEiYABRFc9a4t',
        'syEbRGyFP91A1AIUaahPZfAI/7hqv6wKIuUB8GnFAOq2st8ueGhiCTHAKJRTATYo90Zlx60UXaBNW8dlkZ6dNrt2EtjyYphpP3Q3',
        '55zSX4bIRxo6N2CW2RHUmAElGlPBuWs4XJvu6G1HQlczBVbqZpaYaDhCrKHtxd2MFnGgEBQKmhyGgYNgQzDxfW1VYXrSA57aHTQQ',
        '4oOFR6Bh0t4itMH8NvjfNkDIrYSNmMyMwlBueNe4fvS2HR+1UPX+cIHOvVnSuTdP61ydII8oGQ/MwA8lcGsKboTJ4rJoeyBQWXEB',
        'bW+XaHt7lrafg5jkKOzaKirvvEkRoaArdP1Q2T+NV0dK+RsyoIaMW0ARa7d2KXNIhkWUFmLTXXCSb5dO8u1XnaQb1p8VATMAwnwZ',
        'BqhxoSjjYAnXW3WGuFjZLQ4hgY4QQuFtLHYIGF3NXxfQ/W6J7ncXasc0Kkl83hOJKCgYTYTARJYB2F9A5HdLRH63GDnE8Ut4mL9V',
        'w51EheRPs0ek1cGllBE2146xX/dICcok8eIBivFFE9/8aEFOjM7sD417cMHPEd4Utwixk/Vwd1BVETjRAl31fMsLmPr9ElO/v1Dy',
        'UMopOEGmc8h5nOSTPckdnSvw1nrAH51iQSboiQcJgUxReApwUPeWtlG75rgzeAJDV9ZFXr5Mp94vHf/95QZrIjxINPQ1ETAPLr6S',
        'Uyfa7Bfn7QWk3b5eREqvvx4nSIbJygQVkg6ahDL71FxoN6y9K+FCTSTn1/ST02ilCLXZkbiJ2I7JaUoiR+Wl7k4AScD225QXp2oT',
        'NTjpqi/KmvnjqN0Q7SUcWsaSj8BkiIiptvJAC/kaL/M3bWly7TE/lcgEHenHBC9A2QkaHxmChBPbDY1wvw0l2Qjw4MQfPODqdnCe',
        'sbMsQZUofBCnJOAySQScJuTvc61JX1SbaoR9AGFiEMDlrS2RY1DzpCwGEVRWqDtBUMLSM0d+kg1nw206QhLnnCP0WKXgpXyfaJrY',
        'rnFSBkpnqlwlRafRiO0pc1cJCk0h5ByksTRELpEhvWsscVdL0LUHt5iFuh7WeVP8TFAgJzBU9geHXeVVOATxGXLvTuoQRvWaJ5bM',
        'k/uY3Q4ikYLo9FWaKE1PPn/pV/D+w6tTvfwtlV/MHW7v/i8eS7i6YXWYx6CeS5kBWbEpAc+1yLsq6tAIIIHiMg0O9BGl4HABKabv',
        'TXmPPBF3ZIM6tEgcNCGzu0FB7E2xHEVSGedxmB1dhIS7GbkMIPriS3zHIiC+ffOcvuPRCSfF4Sakeojyp3PVyP7kXtfyaklCWJfQ',
        'vP05jPoJfG+Jj8VJa8pDX1/6QRwVZRr2bSqPE+4BogpC4p204DqKv5gk3L69qrw6a+/zcbW8D9loXqbnZWHEdAc/pHo+YS6yoVMc',
        '9nsl9esYUZHVHsRhSkL5EO7tWPItTSfFTJhOfjfwkm0H11pPPOI6ik/o07zzmLEzFrPUE/W2YDG2WcQvZJMcIh2kxnB4BQDQ5QL4',
        'deS/mFrdfntV+Zsx+rj+cI14mp0wu2jAXSxEVhlJTV4tPhYOwqW+1Ugg0Em7taePX0cIi3ni7btnEMJPs0YFWzVErlJqmqZnk3JI',
        'NVhxQ1KlmjIrxEl1SloNM541QOiHbNXSsxRQzCA3KxmeJnRxrHGnTZxQVw2CnvMFTdh6wBBpX3bBP5s3+Cm/YlQfy1AonqCnYkGJ',
        'xh7JWCEVPhJmpRaYNoORm4hTRxwfYNvP46zsSUMYQhliKl5Pc7ptDMOOxe1+UrvXbGNe5ZygAWQn0we5/VcHxMt1fbHccPvdVR0O',
        '4utYGAXYl+pzV+wtTaGTHKGTMuqxHybRqPt7+6ZM43W4v1iXuL20MDEBXPQhuLo6LfpnV/Enrb3oUkU9CHJ5g7mvzWuk7CCt/dW0',
        'BL8qrBOQoBKUxEVyg0FaFBO55dZtSvxYwiPs7u1KnUzrNlZS43PvFACQWlaspyN6O/rCfYhd7XYnnaFL8PFiZeT2/e8XQGoKaNvm',
        'Sx4L+E2218FXeW03xoJZEpB8OdEaGxOldQ92UrzAw6zlHutpwNpCZ6RGSFhBvu5MUgfOm3i2v2VeAGDInRjYb3V4Fms3d6//XtmF',
        'zNnA89NfSItXw9LReTyjwyAyPR0c0VZIKg1e+NqL/cXdYino7vaqDO/NPeCxapiWZaSTk4dNJmnRijNkiGRem0hJ8ygwXClrwOuW',
        'STJsghGSZdTuipkeM0od+SpZO5JIqw5Ieic6oXVhUvc16GDvvB+bcdJlPkabR906M8DSc1vpyNjkL2SrIw5Sd6I5EQKTCxUOQ5VP',
        'eVBhveKIPAs2LgWuOOmyC2e6YbOJGc7Ny19EIrZE9ls8cCRAK89YBb/tZcCByAu3CK84yGUrNv6vo/nL3eSn28lplkuH+mYDFI9z',
        'YG3bjk2+MzBtMqWyGpV7LNPZDZvRj8cdTsd7UijUyqgZa1FSkeOcR56DSQWTMMTSnlRHVuNEI4J8aFy5yuRLt3/wikcTkudISbez',
        'bSfVvbF6OhkkGIH0yRiZQCDpaD3Kt35LYoulp7vrlp4ACu5XOUrQ5TgdgbLNzodDo83pIMOaCmXMAzgsnAEvK6fGgwXcKEfcMz2u',
        'oT3u+JwubPVEl+6M1qYenNGquP4i5c/qVfe18rV7/P5zfIwWADBrO9ROE0fVO47i6HwodSulUWyrfQEhWhnkRMfhOLWW9FSBZzfs',
        '4E3Jc1vJXOfYFVjRO2nJqrFGRXic80vSu25VkeAra8znwFApDfhBEV2ed5tXfskF8RaAv1fynYtFxbu3Fze+k7C9dZzymlZNddpI',
        '5D8f5aDiSFmVw7VpODoFLkBeV5mDol0q8SUOZrFWdnftWtl04s+7Bxmkq2h6/XGqsUao6HUmmlktAoCMdqnIj0aQNCC3Ho7s3IQA',
        'BS996DXrAfe6PLFHEC9FTdZveOHYGhuDmMCb7jjiPSn1pviWB2nz6PfUoSGcuPK+xaYEdIgbzosJ7l0V9rWQxmyFuGEcZ/KmvE8c',
        'QdbGhyQbdLEcmk5GPjmDDJ5WCXIBcQX/LPWicT5Nw1/tfKW/TupbnGl0U8YBsTulTSccZClyreNok0zhOUlG6ZOpz1CkUN6vZtPw',
        'MnRjGvv7XM7lNr1Yo7x7d1XlH4dFb4ofx7nRETYy5Eqn0lsOz2cNU4dtUjFyWiAD77eAjMWPx0fs422ZcQ2+C5F1S1vWrWi/VunD',
        'RtNr4q/HK2VEbqtTwIwzD8Z5bXGKFm4p4TPLJIz6Q0obHBekUcTTcbsGrux5wv9jKjSTSnNSeVD6BKCdjshrYTjFIFHKHuq8xV9t',
        '8j7Tyd6vGgu/Kf7aJ6LydPBsBpxwyoPBuUChnx5oYUZmbWdDz9Gk4chc5xk6LZMmX6Uv2uj8o+ZHnLYBLFv8FCG73MZIuTfPvEtp',
        '4DlE84vh90RpmN+wcEXqj52JgKxWCOBUqUDN40Q1u+ujQLfwjG0SjIujs7mOh1is7N5dsbKbEn0iZiHI7RgTc/8upyXyWcXkq55B',
        '8d3YexSpx6SSdNs1/TWHH1nO7N2GOW8lxb6xs8w2xBqGvnF9d12Ud75734TcAGhwZJlrGWFd+mJrbE4S2B+PwjlTPGh0qJNs42dk',
        'fAXUaARReUAmdRy0aHAdzVmsSt99f9no9jQnnWfEQtLxI7PfNeCNKN5K0Bm8fOfE1NxeNKG+WP29e//cJlIN4sCS+hxL5CMLtIL+',
        'uKH1dEJOlu+Chqjn0Pq/poj9xcbSabf9ZNZt/FhIJ1/Pjd1ogy4PbR6TAf1sJs4+ytJZRiiKp98YUyTtxEsKJ3AVlsMu6OnnICdf',
        'qZzVIRbkjy0ICdl2HMCU7DWyphXFlWhMpymLRxq7FTqqrkg6wQnC+k3ySwZHakJJvSxzhemZaokLX96wFo0Xt0GnItZHmVX/D+ca',
        '3ix2Ct68XhjokrTnrNmnMrw4lF20vX476pC7HHKZgQ421cm0Bct+U/6E8DBBs+2WKYxr8wAfgcYhDaNjQUrodbYvfabap5FQAKth',
        '9k1JYufTfoyD4m5pdPmDfE796Zv/BRNpkCToPQAA',
    ],
}
for _n, _b in _FILES.items():
    _p = (DATA / _n) if _n.startswith("udhr/") else (SRC / _n)
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(gzip.decompress(base64.b64decode("".join(_b))))
sys.path.insert(0, str(SRC))
import nb_helpers as H
print("wrote", len(_FILES), "files; sha256 prefixes:")
for _n in sorted(_FILES):
    _p = (DATA / _n) if _n.startswith("udhr/") else (SRC / _n)
    print(" ", hashlib.sha256(_p.read_bytes()).hexdigest()[:12], _n)

In [ ]:
# 5. Install what is missing (tokenizers, tiktoken, pyarrow, ...; torch is Colab's own and is not touched), then report versions
H.install()
H.env_report()

In [ ]:
# 6. Doctor: GPU, the model code, the tokenizers, the network, disk and Drive. Continue only if the last line says ALL GOOD.
import mp_data as D
_rows, _total = D.plan(PROFILE if not LOCAL else "local", EXTRA)
OK = H.doctor(SRC, DATA, DRIVE_OUT, need_gb=max(2.0, 5 * _total / 1e9))

In [ ]:
# 7. What would be downloaded for this profile: names, sources, licences, sizes (DATA-CARD.md has the details). Nothing is fetched here.
_rows, _total = D.print_plan(PROFILE if not LOCAL else "local", EXTRA)
print("\nNVIDIA: no Armenian text data exists in the Nemotron datasets (checked 2026-10-28); Nemotron is measured through its tokenizer only.")
print("To go on, tick I_APPROVE_DOWNLOADS in cell 2, run cell 2 again, then cell 8." if not I_APPROVE_DOWNLOADS else "Downloads approved in cell 2.")

In [ ]:
# 8. Fetch (resume-safe), then build: evaluation sets first, then cleaned, decontaminated, deduplicated training pools (Eastern / Western / Grabar)
if LOCAL:
    print("local dry run: no downloads; UD text from", os.environ.get("MP_UD_DIR"))
else:
    assert I_APPROVE_DOWNLOADS, "tick I_APPROVE_DOWNLOADS in cell 2 after reading the list in cell 7"
    H.sh([sys.executable, str(SRC / "mp_data.py"), "fetch", "--profile", PROFILE, "--root", str(DATA), "--extra", ",".join(EXTRA)])
if not (DATA / "data_stats.json").exists():
    H.sh([sys.executable, str(SRC / "mp_data.py"), "build", "--profile", PROFILE if not LOCAL else "local", "--root", str(DATA),
          "--workers", str(WORKERS), "--extra", ",".join(EXTRA)])
_ds = json.loads((DATA / "data_stats.json").read_text())
print("classifier (single UD test sentences):", _ds["classifier"]["accuracy"])
print("pools MB:", {k: round(v["bytes"] / 1e6, 1) for k, v in _ds["pools"].items()})
print("dialect x spelling (MB):", {k: round(v / 1e6, 1) for k, v in _ds.get("crosstab_bytes", {}).items()})
H.checkpoint(DATA, DRIVE_OUT, ["data_stats.json", "fetch_log.json", "annotation"])   # annotation/: the first batch for participation/annotate.html (its KEY file stays with the coordinator)

In [ ]:
# 9. The token bill: eight big-lab tokenizers on the UDHR (Armenian / English) and on four treebanks. CPU only.
#    Gemma and Llama are gated: with ALLOW_MIRRORS a public copy is used only if it is byte-identical (sha256) to the official file.
H.sh([sys.executable, str(SRC / "mp_tok.py"), "bill", "--root", str(DATA), "--out", str(RESULTS)] + (["--allow-mirrors"] if ALLOW_MIRRORS else []))
H.checkpoint(DATA, DRIVE_OUT, ["results"])

In [ ]:
# 10. Our tokenizers: learn the suffix inventory from the UD lemmas, train bpe / mashtots / two ablations on an equal Eastern-Western sample,
#     measure them, then encode the training pools for the trainer (resume-safe).
if not (DATA / "tokenizers" / "train_report.json").exists():
    H.sh([sys.executable, str(SRC / "mp_tok.py"), "train", "--root", str(DATA), "--sample-mb", "300"])
H.sh([sys.executable, str(SRC / "mp_tok.py"), "fertility", "--root", str(DATA), "--out", str(RESULTS)])
H.sh([sys.executable, str(SRC / "mp_tok.py"), "encode", "--root", str(DATA), "--workers", str(WORKERS)])
H.checkpoint(DATA, DRIVE_OUT, ["tokenizers", "results", "tokens/encode_report.json"])

In [ ]:
# 11. FREEZE the pre-registration and the code before any model trains (sha256 of PREREG.md and every code file)
FROZEN = H.freeze(SRC, DATA, PROFILE, dict(build=BUILD, build_hash=BUILD_HASH, run=RUN_NAME))
H.checkpoint(DATA, DRIVE_OUT, ["PREREG.md", "PREREG.sha256"])

In [ ]:
# 12. What will training cost? Measures this GPU's speed for every (tokenizer, size) of the profile (a few minutes), then projects the hours.
SPECS = H.grid(PROFILE, ARMS_OVERRIDE, SIZES_OVERRIDE)
BENCH = H.bench(SRC, DATA, SPECS, compile_=COMPILE)
PLAN_ROWS, PLAN_HOURS = H.project(SPECS, BENCH, UNITS_PER_HOUR)
H.checkpoint(DATA, DRIVE_OUT, ["bench"])

In [ ]:
# 13. Train the grid, smallest first (resume-safe; each run checkpoints to Drive every 15 minutes and is skipped once finished).
assert (DATA / "PREREG.sha256").exists(), "run the Freeze cell first"
DONE, LEFT = H.train_grid(SRC, DATA, SPECS, RUNS, DRIVE_RUNS, max_hours=MAX_HOURS, compile_=COMPILE)

In [ ]:
# 14. Analyze exactly as pre-registered; the verdicts, the tables and the figures
H.sh([sys.executable, str(SRC / "mp_analyze.py"), "--root", str(DATA), "--runs", str(RUNS), "--out", str(RESULTS)])
H.show(sorted((RESULTS / "figs").glob("*.png")))
H.checkpoint(DATA, DRIVE_OUT, ["results"])

In [ ]:
# 15. Export the real model for the X3D scene: letter attention (36 x 36 per layer and head), letter embeddings, the landing's two texts
H.sh([sys.executable, str(SRC / "mp_export.py"), "--root", str(DATA), "--runs", str(RUNS), "--out", str(RESULTS / "mashtots_model_export.json"),
      "--device", "cpu" if LOCAL else "cuda"])
H.show([RESULTS / "figs" / "export_preview.png"])
H.checkpoint(DATA, DRIVE_OUT, ["results"])

In [ ]:
# 16. Package: run record and a last copy of everything small to Drive
import platform, importlib.metadata as _md
_info = dict(build=BUILD, build_hash=BUILD_HASH, profile=PROFILE, run=RUN_NAME, finished=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
             python=platform.python_version(), packages={m: _md.version(m) for m in ("torch", "tokenizers", "tiktoken", "pyarrow", "huggingface_hub")},
             runs_done=sorted(p.parent.name for p in RUNS.glob("*/done.json")), projected_hours=round(PLAN_HOURS, 2), units_per_hour=UNITS_PER_HOUR)
(DATA / "run_info.json").write_text(json.dumps(_info, indent=1))
H.checkpoint(DATA, DRIVE_OUT, ["run_info.json", "results", "data_stats.json", "PREREG.md", "PREREG.sha256", "tokenizers"])
print("DONE. Results are in", DRIVE_OUT)
print("Tell Claude the results are in Drive. If you are finished: Runtime > Disconnect and delete runtime (an idle G4 still spends units).")